# StockEye v2.19 — modo ULTRA: objetivo <4 GB RAM / <3 GB VRAM

**Versión corregida.** Cambios principales respecto a la versión anterior:

| Problema anterior | Corrección en esta versión |
|---|---|
| Recortes de góndola diminutos (1 producto) y "planos" | Filtro de área mínima + contención + máx. 2 góndolas por foto, y descarte de recortes chicos o con proporción extrema |
| El detector de repisa encontraba el **riel de precios**, no la repisa | Se elimina la etapa de repisa como recorte: las repisas se **calculan por geometría** a partir de los productos |
| Etiquetas de GDINO poco ajustadas | **Grounded-SAM**: GDINO propone la caja y SAM 2 la ajusta al borde real del objeto (máscara → caja) |
| Fotos en ángulo | **Aplanado**: homografía calculada desde la máscara SAM de la góndola (vista frontal) |
| Reflejos en vidrio de refrigeradores | **Filtro homomórfico en el dominio de Fourier** (corrige la iluminación) |
| Productos chicos en la foto completa | GDINO **por mosaicos** (*tiling*) sobre la góndola aplanada + YOLO a 1024 px |
| Fuga de datos (ráfagas repartidas entre train y val) | Split **por grupo de ángulo**, no por foto |
| EXIF: cajas desalineadas en fotos de celular | Normalización EXIF al extraer |
| Precisión baja en producto | Barrido de umbral de confianza para alcanzar **P ≥ 80%** |
| Modelo único a 640 px (producto AP50 0.19) | Se descarta; queda la **cascada** góndola → producto |

**Optimización v2.17:** se mantienen exactamente los parámetros de eficacia del detector de producto (`YOLO11s`, `imgsz=1024`, `conf>=0.65`) y del detector de vacíos. Se optimiza memoria mediante lote SAM 32, carga perezosa/liberación de modelos, exclusión de outputs de Drive, reutilización de mapas visuales y cierre sistemático de figuras.


El índice de celdas está al final de esta celda.

**Índice de celdas** (se cuentan los markdown):

| Celda | Tipo | Contenido |
|---|---|---|
| 0 | MD | StockEye v2 — góndola → aplanado → producto → vacíos (Colab, fotos propias, CRISP-DM) |
| 1 | MD | 1. Setup |
| 2 | Código | Instalación de librerías |
| 3 | Código | Imports, rutas y configuración (CFG) |
| 4 | MD | 2. Datos: extracción, corrección EXIF y grupos de ángulo |
| 5 | Código | Extracción del zip + corrección EXIF |
| 6 | Código | Grupos de ángulo y split por grupo |
| 7 | Código | # Vista general: 24 fotos (título azul = val) |
| 8 | MD | 3. Preprocesamiento en el dominio de Fourier |
| 9 | Código | Fourier: filtro homomórfico + nitidez FFT |
| 10 | MD | 4. Modelos fundacionales: Grounding DINO + SAM 2 (Grounded-SAM) |
| 11 | Código | Carga GDINO + SAM 2 y geometría |
| 12 | MD | 5. Etapa 1 — Góndola: etiquetado con Grounded-SAM y vista previa de las 187 fotos |
| 13 | Código | Etiquetado de góndola + vista previa 187 fotos |
| 14 | Código | Dataset YOLO de góndola |
| 15 | Código | Entrenamiento YOLO góndola |
| 16 | MD | 6. Aplanado (rectificación de perspectiva) de cada góndola |
| 17 | Código | Aplanado por homografía |
| 18 | MD | 7. Etapa 2 — Producto: GDINO por mosaicos + filtros + SAM |
| 19 | Código | Funciones de producto + paso a paso |
| 20 | Código | Etiquetado de productos GDINO (con caché en Drive) |
| 21 | Código | Vista previa paginada de productos |
| 22 | MD | 7b. Maestro con etiquetas humanas (Humans in the Loop, CC0) + etiquetas híbridas |
| 23 | Código | Descarga y conversión del dataset Humans in the Loop (CC0, 45 fotos, ~11.7k cajas pro |
| 24 | Código | Maestro: YOLO entrenado con etiquetas HUMANAS (producto + precio) |
| 25 | Código | Etiquetas híbridas en NUESTRAS góndolas: maestro (mosaicos) + GDINO+SAM, fusión por a |
| 26 | Código | Comparación visual GDINO vs MAESTRO vs HÍBRIDO + activar etiquetas híbridas |
| 27 | Código | Dataset YOLO de producto + fotos HITL |
| 28 | Código | Data augmentation offline (solo TRAIN): contraste, sombras, ruido, compresión, desenf |
| 29 | Código | Entrenamiento YOLO producto + test humano HITL |
| 30 | MD | 8. Umbral de confianza para alcanzar P ≥ 80% |
| 31 | Código | Barrido de umbral (P >= 80%) |
| 32 | MD | 9. Repisas por geometría y espacios vacíos (con FFT) |
| 33 | Código | Repisas por geometría + vacíos |
| 34 | MD | 10. Pipeline completo de inferencia (lo que se entregaría a Walmart) + tiempo y VRAM |
| 35 | Código | Pipeline completo + tiempo y VRAM |
| 36 | Código | Visualización del pipeline |
| 37 | MD | 11. Los 3 mejores y los 3 peores resultados |
| 38 | Código | Mejores/peores: góndola |
| 39 | Código | Mejores/peores: producto |
| 40 | Código | Mejores/peores: vacíos |
| 41 | Código | Diagnóstico vs versión anterior |
| 42 | MD | 11b. Diagnóstico: sobreajuste, galería completa y auditoría de errores |
| 43 | Código | Diagnóstico de sobreajuste: train vs val + curvas de pérdida |
| 44 | Código | Galería completa de resultados de producto -> Drive (todas las de val + 20 de train) |
| 45 | Código | Auditoría de errores: ¿los FP son productos reales sin etiqueta? |
| 46 | MD | 12. Exportación a Drive |
| 47 | Código | Exportación a Drive |


### 1. Setup y acceso a Google Drive sin montar `MyDrive`

Esta versión no utiliza `drive.mount()`. La carpeta del proyecto se abre directamente por su ID mediante
Google Drive API. Esto permite ejecutar el notebook desde otra cuenta de Google siempre que esa cuenta tenga
acceso a la carpeta compartida.

Al ejecutar la celda de configuración, Google pedirá autenticación. Debes seleccionar la cuenta que tenga
acceso a la carpeta del proyecto.

Si esa cuenta posee permiso de edición, el notebook también podrá subir cachés, galerías y modelos. Si solo
tiene permiso de lectura, podrá utilizar las imágenes, pero los resultados permanecerán temporalmente en
`/content/stockeye_persist`.


In [ ]:
# [Celda 2]  Instalación de librerías
%pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow matplotlib pyyaml transformers accelerate pandas

In [ ]:
import os, re, gc, json, time, shutil, random, math, mimetypes, zipfile, socket, ssl

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import cv2
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

SHARED_DRIVE_FOLDER_ID = "1c5Aq5CLjikh9kwe3tSIkXspBSKEyHaTr"

auth.authenticate_user()

credenciales, _ = default()

drive_api = build(
    "drive",
    "v3",
    credentials=credenciales,
    cache_discovery=False
)

folder_info = drive_api.files().get(
    fileId=SHARED_DRIVE_FOLDER_ID,
    fields=(
        "id,"
        "name,"
        "mimeType,"
        "trashed,"
        "capabilities("
        "canAddChildren,"
        "canEdit,"
        "canDownload"
        ")"
    )
).execute()

assert (
    folder_info["mimeType"]
    == "application/vnd.google-apps.folder"
), "El ID indicado no corresponde a una carpeta."

assert not folder_info.get(
    "trashed",
    False
), "La carpeta está en la papelera."

DRIVE_CAN_WRITE = bool(
    folder_info.get(
        "capabilities",
        {}
    ).get(
        "canAddChildren",
        False
    )
)


SUBIR_PARA_ETIQUETAR_AUTOMATICO = False
SUBIR_GALERIA_AUTOMATICO = False
SUBIR_OUTPUTS_AUTOMATICO = True

# Optimización segura: no cambia modelos, resolución ni umbrales de decisión
MODO_PIPELINE = "completo"       # "completo" o "val" para pruebas rápidas
CERRAR_FIGURAS = True

MODO_BAJO_RECURSO = True
MODO_ULTRA_4GB_3GB = True

RAM_LIMITE_GB = 4.0
RAM_OBJETIVO_GB = 3.4

# PyTorch queda limitado por debajo de 3 GB para dejar margen
# al contexto CUDA y otras reservas que no cuenta memory_allocated().
VRAM_ALLOCATOR_LIMITE_GB = 2.30
VRAM_ALERTA_GB = 2.15

TRAIN_BATCH_ECO = 1
TRAIN_WORKERS_ECO = 0
VAL_BATCH_ECO = 1
VAL_WORKERS_ECO = 0
NOMINAL_BATCH = 64
SAM_LOTE_ECO = 1
CPU_THREADS_ECO = 1

PRODUCTO_TILE_PX = 640
PRODUCTO_TILE_OVERLAP = 0.20
PRODUCTO_TILE_MIN_VIS = 0.55

# Los modelos fundacionales pesados se ejecutan en CPU.
FOUNDATION_DEVICE = "cpu"

# Si no existe caché GDINO+SAM de producto, se usa el maestro HITL.
PERMITIR_GDINO_PRODUCTO_SIN_CACHE = False

# Evita conservar imágenes grandes para vistas previas.
MOSTRAR_PREVIEWS = False
CARPETAS_IGNORAR_DRIVE = {
    "outputs_v2",
    "resultados_producto",
    "para_etiquetar_producto",
    "runs",
    "cache"
}
METRICAS_RECURSOS = {}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path("/content/data")
RAW_DIR = DATA / "raw"
NORM_DIR = DATA / "norm"
EXPLORE_DIR = DATA / "explore"
GONDOLA_DS = DATA / "ds_gondola"
RECT_DIR = DATA / "gondolas_rect"
PRODUCTO_DS = DATA / "ds_producto"
PIPE_DIR = DATA / "pipeline_rect"
RUNS_DIR = Path("/content/runs")

PERSIST_DIR = Path("/content/stockeye_persist")
PERSIST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPORT_DIR = PERSIST_DIR / "outputs_v2"
GALERIA_LOCAL = PERSIST_DIR / "resultados_producto"


def listar_drive_recursivo(
    folder_id,
    ruta_relativa=""
):
    salida = []
    token = None

    while True:
        respuesta = drive_api.files().list(
            q=(
                f"'{folder_id}' in parents "
                "and trashed=false"
            ),
            fields=(
                "nextPageToken,"
                "files("
                "id,"
                "name,"
                "mimeType,"
                "size,"
                "parents"
                ")"
            ),
            pageSize=1000,
            pageToken=token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()

        for item in respuesta.get(
            "files",
            []
        ):
            rel = (
                f"{ruta_relativa}/{item['name']}"
                if ruta_relativa
                else item["name"]
            )

            registro = dict(item)
            registro["relative_path"] = rel
            salida.append(registro)

            if (
                item["mimeType"]
                == "application/vnd.google-apps.folder"
            ):
                if item["name"].strip().lower() in CARPETAS_IGNORAR_DRIVE:
                    continue

                salida.extend(
                    listar_drive_recursivo(
                        item["id"],
                        rel
                    )
                )

        token = respuesta.get(
            "nextPageToken"
        )

        if not token:
            break

    return salida


def buscar_hijo_drive(
    parent_id,
    nombre
):
    nombre_seguro = nombre.replace(
        "'",
        "\\'"
    )

    respuesta = drive_api.files().list(
        q=(
            f"'{parent_id}' in parents "
            "and trashed=false "
            f"and name='{nombre_seguro}'"
        ),
        fields=(
            "files("
            "id,"
            "name,"
            "mimeType,"
            "parents"
            ")"
        ),
        pageSize=100,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()

    archivos = respuesta.get(
        "files",
        []
    )

    return (
        archivos[0]
        if archivos
        else None
    )


def descargar_archivo_drive(
    file_id,
    destino
):
    destino = Path(
        destino
    )

    destino.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    request = drive_api.files().get_media(
        fileId=file_id,
        supportsAllDrives=True
    )

    with open(
        destino,
        "wb"
    ) as f:
        downloader = MediaIoBaseDownload(
            f,
            request,
            chunksize=8
            * 1024
            * 1024
        )

        terminado = False

        while not terminado:
            _, terminado = downloader.next_chunk()

    return destino



def safe_extract(
    zip_path,
    dest
):
    zip_path = Path(
        zip_path
    )

    dest = Path(
        dest
    )

    dest.mkdir(
        parents=True,
        exist_ok=True
    )

    dest_resolved = dest.resolve()

    with zipfile.ZipFile(
        zip_path
    ) as zf:
        for member in zf.infolist():
            target = (
                dest
                / member.filename
            ).resolve()

            if not str(
                target
            ).startswith(
                str(
                    dest_resolved
                )
            ):
                raise RuntimeError(
                    "Entrada ZIP sospechosa: "
                    + member.filename
                )

        zf.extractall(
            dest
        )

def obtener_o_crear_carpeta_drive(
    parent_id,
    nombre
):
    existente = buscar_hijo_drive(
        parent_id,
        nombre
    )

    if (
        existente is not None
        and existente[
            "mimeType"
        ]
        == "application/vnd.google-apps.folder"
    ):
        return existente["id"]

    if not DRIVE_CAN_WRITE:
        return None

    creada = drive_api.files().create(
        body={
            "name": nombre,
            "mimeType": (
                "application/vnd.google-apps.folder"
            ),
            "parents": [
                parent_id
            ]
        },
        fields="id,name",
        supportsAllDrives=True
    ).execute()

    return creada["id"]


def subir_archivo_drive(
    ruta_local,
    parent_id,
    nombre=None,
    reintentos=5
):
    if not DRIVE_CAN_WRITE:
        return None

    ruta_local = Path(
        ruta_local
    )

    nombre = (
        ruta_local.name
        if nombre is None
        else nombre
    )

    mime_type = (
        mimetypes.guess_type(
            str(
                ruta_local
            )
        )[0]
        or "application/octet-stream"
    )

    ultimo_error = None

    for intento in range(
        1,
        reintentos + 1
    ):
        try:
            existente = buscar_hijo_drive(
                parent_id,
                nombre
            )

            media = MediaFileUpload(
                str(
                    ruta_local
                ),
                mimetype=mime_type,
                chunksize=8
                * 1024
                * 1024,
                resumable=True
            )

            if (
                existente is not None
                and existente[
                    "mimeType"
                ]
                != "application/vnd.google-apps.folder"
            ):
                request = drive_api.files().update(
                    fileId=existente["id"],
                    media_body=media,
                    fields="id,name",
                    supportsAllDrives=True
                )
            else:
                request = drive_api.files().create(
                    body={
                        "name": nombre,
                        "parents": [
                            parent_id
                        ]
                    },
                    media_body=media,
                    fields="id,name",
                    supportsAllDrives=True
                )

            respuesta = None

            while respuesta is None:
                _, respuesta = request.next_chunk(
                    num_retries=3
                )

            return respuesta

        except (
            TimeoutError,
            socket.timeout,
            ssl.SSLError,
            OSError
        ) as ex:
            ultimo_error = ex

            print(
                f"Timeout al subir {nombre}. "
                f"Intento {intento}/{reintentos}."
            )

            if intento < reintentos:
                time.sleep(
                    min(
                        30,
                        2 ** intento
                    )
                )

    print(
        f"No se pudo subir {nombre}. "
        "El notebook continuará. "
        f"Último error: {ultimo_error}"
    )

    return None


def subir_carpeta_local_drive(
    carpeta_local,
    parent_id,
    detener_si_falla=False
):
    carpeta_local = Path(
        carpeta_local
    )

    destino_id = obtener_o_crear_carpeta_drive(
        parent_id,
        carpeta_local.name
    )

    if destino_id is None:
        return None

    fallos = []

    for item in sorted(
        carpeta_local.iterdir()
    ):
        if item.is_dir():
            resultado = subir_carpeta_local_drive(
                item,
                destino_id,
                detener_si_falla=detener_si_falla
            )

            if resultado is None:
                fallos.append(
                    item.name
                )
        else:
            resultado = subir_archivo_drive(
                item,
                destino_id
            )

            if resultado is None:
                fallos.append(
                    item.name
                )

                if detener_si_falla:
                    raise RuntimeError(
                        "No se pudo subir "
                        + item.name
                    )

    if fallos:
        print(
            "La carpeta se procesó con "
            f"{len(fallos)} archivo(s) no subido(s)."
        )

    return destino_id


ARCHIVOS_DRIVE = listar_drive_recursivo(
    SHARED_DRIVE_FOLDER_ID
)

IMAGENES_DRIVE = [
    item
    for item
    in ARCHIVOS_DRIVE
    if (
        item["mimeType"].startswith(
            "image/"
        )
        or item["name"].lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".webp"
            )
        )
    )
]

assert len(
    IMAGENES_DRIVE
) > 0, (
    "La cuenta autenticada puede abrir la carpeta, "
    "pero no se encontraron imágenes JPG/JPEG/PNG/WEBP."
)

CFG = dict(
    max_lado_foto=3072,
    val_ratio=0.20,
    wa_chunk=6,
    usar_homomorfico=True,
    homo_gamma_l=0.5,
    homo_gamma_h=1.3,
    homo_d0_frac=0.02,
    gdino_gondola_id="IDEA-Research/grounding-dino-tiny",
    gdino_producto_id="IDEA-Research/grounding-dino-base",
    prompt_gondola="store shelving unit. refrigerated display case.",
    umbral_gondola=0.25,
    gondola_min_area=0.15,
    gondola_contain=0.80,
    gondola_max=2,
    prompt_producto="bottle. can. box. bag. snack bag. jar. carton. package. price tag.",
    umbral_producto=0.20,
    text_umbral_producto=0.18,
    tile=480,
    tile_overlap=0.30,
    min_prod_fila=3,
    prod_area_max_vs_med=4.0,
    prod_area_min_vs_med=0.08,
    prod_aspect_min=0.15,
    prod_aspect_max=4.0,
    prod_grupo_min_hijos=3,
    sam_weights="mobile_sam.pt" if MODO_ULTRA_4GB_3GB else "sam2.1_b.pt",
    sam_lote=SAM_LOTE_ECO if MODO_BAJO_RECURSO else 32,
    sam_fill_min=0.35,
    sam_iou_min=0.50,
    rect_max_lado=1600,
    rect_min_lado=200,
    rect_aspect_max=4.0,
    rect_area_min_vs_bbox=0.60,
    yolo_gondola="yolo11n.pt",
    imgsz_gondola=640,
    ep_gondola=80,
    batch_gondola=TRAIN_BATCH_ECO if MODO_BAJO_RECURSO else 8,
    yolo_producto="yolo11s.pt",
    imgsz_producto=640 if MODO_ULTRA_4GB_3GB else 1024,
    ep_producto=150,
    batch_producto=TRAIN_BATCH_ECO if MODO_BAJO_RECURSO else 4,
    conf_gondola=0.40,
    precision_objetivo=0.80,
    gap_min_vs_ancho=0.9,
    score_vacio_min=0.50,
    alerta_ocupacion=0.30
)

CFG.update(
    dict(
        hitl_kaggle="humansintheloop/supermarket-shelves-dataset",
        hitl_val_imgs=8,
        hitl_max_lado=1600,
        hitl_tile=640,
        hitl_tile_overlap=0.20,
        maestro_epochs=120,
        maestro_imgsz=640,
        maestro_batch=TRAIN_BATCH_ECO if MODO_BAJO_RECURSO else 8,
        maestro_conf=0.25,
        maestro_conf_solo=0.50,
        gdino_conf_solo=0.20,
        maestro_textura_min=0.35,
        usar_cache_etiquetas=True,
        fusion_iou=0.50
    )
)

TORCH_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


if MODO_ULTRA_4GB_3GB and torch.cuda.is_available():
    props_gpu = torch.cuda.get_device_properties(0)
    total_gpu_bytes = float(props_gpu.total_memory)

    limite_allocator_bytes = (
        VRAM_ALLOCATOR_LIMITE_GB
        * (1024 ** 3)
    )

    fraccion_cuda = min(
        0.95,
        limite_allocator_bytes
        / total_gpu_bytes
    )

    torch.cuda.set_per_process_memory_fraction(
        fraccion_cuda,
        device=0
    )

    print(
        "Límite allocator PyTorch:",
        f"{VRAM_ALLOCATOR_LIMITE_GB:.2f} GB",
        "| fracción:",
        f"{fraccion_cuda:.3f}"
    )


if MODO_BAJO_RECURSO:
    try:
        torch.set_num_threads(CPU_THREADS_ECO)
    except Exception:
        pass
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass
    try:
        cv2.setNumThreads(1)
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False


def argumentos_entrenamiento_eco():
    if not MODO_BAJO_RECURSO:
        return {"workers": 2, "plots": True}
    return {
        "workers": TRAIN_WORKERS_ECO,
        "cache": False,
        "amp": True,
        "plots": False,
        "nbs": NOMINAL_BATCH
    }


def argumentos_validacion_eco():
    if not MODO_BAJO_RECURSO:
        return {}
    return {
        "batch": VAL_BATCH_ECO,
        "workers": VAL_WORKERS_ECO,
        "plots": False
    }


def liberar_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def limpiar_dir(
    p
):
    p = Path(
        p
    )

    if p.exists():
        shutil.rmtree(
            p
        )

    p.mkdir(
        parents=True,
        exist_ok=True
    )

    return p




def _ram_actual_gb():
    try:
        with open("/proc/self/status", "r") as f:
            for linea in f:
                if linea.startswith("VmRSS:"):
                    return float(linea.split()[1]) / 1024 / 1024
    except Exception:
        pass
    return None


def registrar_recursos(etapa, reset_gpu_peak=False):
    if torch.cuda.is_available() and reset_gpu_peak:
        torch.cuda.reset_peak_memory_stats()

    ram = _ram_actual_gb()
    gpu_alloc = (
        torch.cuda.memory_allocated() / 1e9
        if torch.cuda.is_available()
        else 0.0
    )
    gpu_res = (
        torch.cuda.memory_reserved() / 1e9
        if torch.cuda.is_available()
        else 0.0
    )
    gpu_peak = (
        torch.cuda.max_memory_allocated() / 1e9
        if torch.cuda.is_available()
        else 0.0
    )

    dato = {
        "ram_gb": None if ram is None else float(ram),
        "vram_alloc_gb": float(gpu_alloc),
        "vram_reserved_gb": float(gpu_res),
        "vram_peak_gb": float(gpu_peak)
    }
    METRICAS_RECURSOS[etapa] = dato

    print(
        f"[Recursos] {etapa}: "
        f"RAM={ram:.2f} GB | " if ram is not None else f"[Recursos] {etapa}: RAM=n/d | ",
        f"VRAM={gpu_alloc:.2f} GB | pico={gpu_peak:.2f} GB"
    )
    return dato


def ram_actual_gb():
    try:
        with open("/proc/self/status", "r") as f:
            for linea in f:
                if linea.startswith("VmRSS:"):
                    return float(linea.split()[1]) / 1024 / 1024
    except Exception:
        pass
    return None


def vram_actual_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9


def limpiar_memoria_agresivo():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def comprobar_recursos(etapa, estricto=False):
    limpiar_memoria_agresivo()

    ram = ram_actual_gb()
    vram = vram_actual_gb()

    print(
        f"[LÍMITES] {etapa}: "
        f"RAM={ram if ram is not None else -1:.2f} GB | "
        f"VRAM PyTorch={vram:.2f} GB"
    )

    if ram is not None and ram > RAM_OBJETIVO_GB:
        print(
            "Aviso: RAM sobre objetivo. "
            "Se liberaron cachés antes de continuar."
        )

    if estricto and ram is not None and ram >= RAM_LIMITE_GB:
        raise MemoryError(
            f"RAM >= {RAM_LIMITE_GB:.1f} GB "
            f"en {etapa}. Se detiene antes de "
            "iniciar otra etapa pesada."
        )

    if estricto and vram >= VRAM_ALERTA_GB:
        print(
            "VRAM cerca del límite del allocator. "
            "La siguiente operación se ejecutará "
            "con batch 1."
        )

    return {
        "ram_gb": ram,
        "vram_gb": vram
    }


def es_error_oom(ex):
    texto = str(ex).lower()

    return any(
        token in texto
        for token in (
            "out of memory",
            "cuda error",
            "memory allocation",
            "cublas_status_alloc_failed"
        )
    )


print(
    "Carpeta compartida:",
    folder_info["name"]
)

print(
    "ID:",
    SHARED_DRIVE_FOLDER_ID
)

print(
    "Imágenes encontradas:",
    len(
        IMAGENES_DRIVE
    )
)

print(
    "Permiso para guardar resultados:",
    DRIVE_CAN_WRITE
)

print(
    "Drive montado:",
    False
)

print(
    "Modo de acceso: DRIVE API"
)

print(
    "Dispositivo:",
    TORCH_DEVICE,
    "| torch",
    torch.__version__
)


print(
    "Modo bajo recurso:",
    MODO_BAJO_RECURSO
)

print(
    "Modo ULTRA 4GB RAM / 3GB VRAM:",
    MODO_ULTRA_4GB_3GB
)

if MODO_BAJO_RECURSO:
    print(
        f"ECO: batch={TRAIN_BATCH_ECO}, "
        f"workers={TRAIN_WORKERS_ECO}, "
        f"SAM lote={CFG['sam_lote']}, "
        f"nbs={NOMINAL_BATCH}"
    )


### Perfil ULTRA: hasta 4 GB RAM y hasta 3 GB de GPU

Esta versión prioriza el límite de memoria por encima de la velocidad. El flujo se vuelve mucho más
secuencial: `batch=1`, `workers=0`, SAM procesa una caja a la vez y los modelos fundacionales trabajan en
CPU. PyTorch limita su allocator CUDA a aproximadamente 2.30 GB para dejar margen al contexto CUDA.

Para conservar detalle de productos pequeños, el detector de producto trabaja con mosaicos de 640 px con
solapamiento. El umbral mínimo de producto continúa siendo `0.65`.

Si `YOLO11s` no cabe dentro del límite, el notebook prueba automáticamente una resolución menor y, como
último recurso, `YOLO11n`. El modelo y la resolución finalmente utilizados quedan registrados.


In [ ]:
print("CONFIGURACIÓN ULTRA 4GB / 3GB")
print("Batch góndola:", CFG["batch_gondola"])
print("Batch maestro:", CFG["maestro_batch"])
print("Batch producto:", CFG["batch_producto"])
print("SAM lote:", CFG["sam_lote"])
print("Workers:", TRAIN_WORKERS_ECO)
print("nbs:", NOMINAL_BATCH)
print("Producto:", CFG["yolo_producto"], "| imgsz:", CFG["imgsz_producto"])
print("Épocas producto:", CFG["ep_producto"])
registrar_recursos("inicio_modo_eco", reset_gpu_peak=True)

print("RAM objetivo:", RAM_OBJETIVO_GB, "GB")
print("RAM límite:", RAM_LIMITE_GB, "GB")
print("Allocator CUDA:", VRAM_ALLOCATOR_LIMITE_GB, "GB")
print("Foundation device:", FOUNDATION_DEVICE)
print("Tile producto:", PRODUCTO_TILE_PX, "px")
comprobar_recursos("inicio_ultra", estricto=True)


### 2. Datos: imágenes directas desde Google Drive

La carpeta compartida no contiene un archivo ZIP. El notebook busca recursivamente imágenes JPG, JPEG, PNG y WEBP en la carpeta y sus subcarpetas, las descarga directamente a Colab y luego aplica corrección EXIF y normalización antes de continuar.


In [ ]:
def descargar_imagenes_drive(
    items,
    destino
):
    destino = Path(destino)

    imagenes_existentes = [
        p
        for p in destino.rglob("*")
        if p.suffix.lower()
        in (
            ".jpg",
            ".jpeg",
            ".png",
            ".webp"
        )
    ] if destino.exists() else []

    if (
        len(imagenes_existentes)
        >= len(items)
        and len(items) > 0
    ):
        print(
            f"Las {len(imagenes_existentes)} imágenes "
            "ya están descargadas. "
            "No se descargarán nuevamente."
        )

        return

    limpiar_dir(destino)

    usados = set()
    total = len(items)

    for i, item in enumerate(
        items,
        start=1
    ):
        nombre = Path(
            item["name"]
        ).name

        stem = Path(nombre).stem
        ext = Path(nombre).suffix or ".jpg"

        nombre_final = nombre
        contador = 1

        while (
            nombre_final.lower()
            in usados
        ):
            nombre_final = (
                f"{stem}_{contador}{ext}"
            )
            contador += 1

        usados.add(
            nombre_final.lower()
        )

        destino_archivo = (
            destino
            / nombre_final
        )

        request = (
            drive_api.files()
            .get_media(
                fileId=item["id"]
            )
        )

        with open(
            destino_archivo,
            "wb"
        ) as f:
            downloader = MediaIoBaseDownload(
                f,
                request,
                chunksize=8
                * 1024
                * 1024
            )

            terminado = False

            while not terminado:
                _, terminado = (
                    downloader.next_chunk()
                )

        if (
            i % 20 == 0
            or i == total
        ):
            print(
                f"  {i}/{total} "
                "imágenes descargadas"
            )


NATIVE_RE = re.compile(
    r"IMG_(\d{8})_(\d{6})",
    re.IGNORECASE
)

WHATSAPP_RE = re.compile(
    r"IMG-(\d{8})-WA(\d{4})",
    re.IGNORECASE
)


def parse_metadata(
    nombre
):
    m = NATIVE_RE.search(nombre)

    if m:
        return {
            "kind": "native",
            "dt": datetime.strptime(
                m.group(1)
                + m.group(2),
                "%Y%m%d%H%M%S"
            ),
            "date": m.group(1),
            "seq": None
        }

    m = WHATSAPP_RE.search(nombre)

    if m:
        return {
            "kind": "whatsapp",
            "dt": None,
            "date": m.group(1),
            "seq": int(
                m.group(2)
            )
        }

    return {
        "kind": "unknown",
        "dt": None,
        "date": None,
        "seq": None
    }


print(
    "Preparando imágenes desde Google Drive..."
)

descargar_imagenes_drive(
    IMAGENES_DRIVE,
    RAW_DIR
)

limpiar_dir(
    NORM_DIR
)

entries = []
n_rotadas = 0

for p in sorted(
    RAW_DIR.rglob("*")
):
    if p.suffix.lower() not in (
        ".jpg",
        ".jpeg",
        ".png",
        ".webp"
    ):
        continue

    with Image.open(p) as im:
        orient = im.getexif().get(
            274,
            1
        )

        im = ImageOps.exif_transpose(
            im
        ).convert(
            "RGB"
        )

        escala = (
            CFG["max_lado_foto"]
            / max(im.size)
        )

        if escala < 1:
            im = im.resize(
                (
                    round(
                        im.width
                        * escala
                    ),
                    round(
                        im.height
                        * escala
                    )
                ),
                Image.LANCZOS
            )

        out = (
            NORM_DIR
            / (
                p.stem
                + ".jpg"
            )
        )

        im.save(
            out,
            quality=95
        )

        w, h = im.size

    n_rotadas += int(
        orient not in (
            1,
            None
        )
    )

    meta = parse_metadata(
        p.name
    )

    meta.update(
        {
            "path": out,
            "nombre": out.name,
            "width": w,
            "height": h
        }
    )

    entries.append(
        meta
    )


assert len(entries) > 0, (
    "No se encontraron imágenes válidas."
)

ENTRY = {
    e["nombre"]: e
    for e in entries
}

print(
    f"Fotos preparadas: "
    f"{len(entries)}"
)

print(
    f"Rotaciones EXIF corregidas: "
    f"{n_rotadas}"
)

print(
    "Tipos:",
    pd.Series(
        [
            e["kind"]
            for e in entries
        ]
    ).value_counts().to_dict()
)


In [ ]:
# [Celda 6]  Grupos de ángulo y split por grupo
BURST_GAP_SECONDS = 15

def construir_grupos(entries):
    grupos = {}
    # Ráfagas nativas (fotos separadas por <= 15 s)
    nativas = sorted([e for e in entries if e["kind"] == "native"], key=lambda e: e["dt"])
    actual, prev, k = [], None, 0
    for e in nativas:
        if prev is not None and (e["dt"] - prev).total_seconds() > BURST_GAP_SECONDS:
            grupos[f"burst_{k}"] = actual; actual, k = [], k + 1
        actual.append(e); prev = e["dt"]
    if actual:
        grupos[f"burst_{k}"] = actual
    # WhatsApp: bloques de fotos consecutivas (corta si hay salto de secuencia o tamaño > wa_chunk)
    por_fecha = defaultdict(list)
    for e in entries:
        if e["kind"] == "whatsapp":
            por_fecha[e["date"]].append(e)
    for fecha, lst in por_fecha.items():
        lst = sorted(lst, key=lambda e: e["seq"])
        bloque, k = [], 0
        for e in lst:
            if bloque and (e["seq"] - bloque[-1]["seq"] > 1 or len(bloque) >= CFG["wa_chunk"]):
                grupos[f"wa_{fecha}_{k}"] = bloque; bloque, k = [], k + 1
            bloque.append(e)
        if bloque:
            grupos[f"wa_{fecha}_{k}"] = bloque
    # Otros: cada foto es su propio grupo
    for e in entries:
        if e["kind"] == "unknown":
            grupos[f"otro_{e['nombre']}"] = [e]
    return grupos

def split_por_grupo(grupos, val_ratio, seed=SEED):
    ids = sorted(grupos)
    random.Random(seed).shuffle(ids)
    total = sum(len(g) for g in grupos.values())
    objetivo = val_ratio * total
    val_ids, n_val = set(), 0
    for gid in ids:
        if n_val >= objetivo:
            break
        if n_val + len(grupos[gid]) <= objetivo * 1.15:
            val_ids.add(gid); n_val += len(grupos[gid])
    return val_ids

ANGLE_GROUPS = construir_grupos(entries)
VAL_GROUPS = split_por_grupo(ANGLE_GROUPS, CFG["val_ratio"])
for gid, lst in ANGLE_GROUPS.items():
    for e in lst:
        e["grupo"] = gid
        e["split"] = "val" if gid in VAL_GROUPS else "train"

SPLIT_FOTO = {e["nombre"]: e["split"] for e in entries}
n_val = sum(1 for s in SPLIT_FOTO.values() if s == "val")
print(f"Grupos de ángulo: {len(ANGLE_GROUPS)} | val: {len(VAL_GROUPS)} grupos, {n_val} fotos | train: {len(entries)-n_val} fotos")
print("Val por tipo:", pd.Series([e['kind'] for e in entries if e['split']=='val']).value_counts().to_dict())

In [ ]:
# [Celda 7]
# Vista general: 24 fotos (título azul = val)
muestra = sorted(entries, key=lambda e: e["nombre"])[::max(1, len(entries) // 24)][:24]
fig, axes = plt.subplots(3, 8, figsize=(24, 9))
for ax in axes.ravel():
    ax.axis("off")
for ax, e in zip(axes.ravel(), muestra):
    im = Image.open(e["path"]); im.thumbnail((400, 400))
    ax.imshow(im)
    ax.set_title(f"{e['nombre'][:22]}\n{e['grupo']}", fontsize=7,
                 color="blue" if e["split"] == "val" else "black")
plt.tight_layout(); plt.show(); plt.close(fig)


### 3. Preprocesamiento en el dominio de Fourier

**Filtro homomórfico:** una imagen es aproximadamente *iluminación × reflectancia*. Con el logaritmo pasa a ser una suma: la iluminación (reflejos del vidrio, sombras, degradados) vive en las **bajas frecuencias** y el detalle de los productos en las **altas**. Con la FFT se atenúan las bajas (`gamma_l < 1`) y se realzan las altas (`gamma_h > 1`). Así quedan cajas más estables frente a reflejos (RF-08).

**Nitidez por FFT:** es la fracción de energía espectral en altas frecuencias. Sirve para detectar fotos borrosas o muy comprimidas (WhatsApp) y explicar después los peores resultados.

El filtro se aplica a las imágenes que ven GDINO y el YOLO de producto, siempre con la misma función `preprocesar()` en el entrenamiento y en la inferencia.


In [ ]:
# [Celda 9]  Fourier: filtro homomórfico + nitidez FFT
def filtro_homomorfico(img_bgr, gamma_l=None, gamma_h=None, d0_frac=None):
    gamma_l = CFG["homo_gamma_l"] if gamma_l is None else gamma_l
    gamma_h = CFG["homo_gamma_h"] if gamma_h is None else gamma_h
    d0_frac = CFG["homo_d0_frac"] if d0_frac is None else d0_frac

    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[..., 0].astype(np.float32) / 255.0
    logL = np.log1p(L)
    h, w = logL.shape

    # Componente de baja frecuencia vía FFT (en resolución reducida para velocidad)
    s = min(1.0, 512 / max(h, w))
    small = cv2.resize(logL, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
    hs, ws = small.shape
    F = np.fft.fftshift(np.fft.fft2(small))
    u = np.arange(hs) - hs / 2.0
    v = np.arange(ws) - ws / 2.0
    V, U = np.meshgrid(v, u)
    d0 = max(1.0, d0_frac * min(hs, ws))
    pasa_bajos = np.exp(-(U ** 2 + V ** 2) / (2 * d0 ** 2))
    low_small = np.real(np.fft.ifft2(np.fft.ifftshift(F * pasa_bajos))).astype(np.float32)
    low = cv2.resize(low_small, (w, h), interpolation=cv2.INTER_CUBIC)

    # H(u,v) = gamma_l * pasa_bajos + gamma_h * pasa_altos
    filtrada = gamma_l * low + gamma_h * (logL - low)
    out = np.expm1(filtrada)
    lo, hi = np.percentile(out, (0.5, 99.5))
    out = np.clip((out - lo) / (hi - lo + 1e-8), 0, 1)
    lab[..., 0] = (out * 255).astype(np.uint8)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def fft_nitidez(img_bgr, radio_frac=0.25):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    s = 512 / max(g.shape)
    if s < 1:
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    g = g - g.mean()
    mag = np.abs(np.fft.fftshift(np.fft.fft2(g))) ** 2
    h, w = mag.shape
    yy, xx = np.ogrid[:h, :w]
    r = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2)
    rmax = min(h, w) / 2
    total = mag[r > 1].sum() + 1e-8
    return float(mag[r > radio_frac * rmax].sum() / total)


def metricas_calidad(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    v, s = hsv[..., 2], hsv[..., 1]
    return {
        "nitidez_fft": fft_nitidez(img_bgr),
        "brillo": float(v.mean() / 255),
        "reflejos": float(((v > 245) & (s < 30)).mean()),   # píxeles quemados casi blancos
    }


def preprocesar(img_bgr):
    return filtro_homomorfico(img_bgr) if CFG["usar_homomorfico"] else img_bgr


for e in entries:
    e.update(metricas_calidad(cv2.imread(str(e["path"]))))

df_cal = pd.DataFrame([{k: e[k] for k in ("nombre", "kind", "nitidez_fft", "brillo", "reflejos")} for e in entries])
print(df_cal.groupby("kind")[["nitidez_fft", "brillo", "reflejos"]].mean().round(4))

# Antes / después en las 3 fotos con más reflejos
peores = df_cal.sort_values("reflejos", ascending=False).head(3)["nombre"]
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for fila, nombre in zip(axes, peores):
    img = cv2.imread(str(ENTRY[nombre]["path"]))
    pre = preprocesar(img)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    espectro = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(cv2.resize(g, (512, 512))))))
    fila[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); fila[0].set_title(f"Original — {nombre}", fontsize=9)
    fila[1].imshow(espectro, cmap="magma"); fila[1].set_title("Espectro de Fourier (log)", fontsize=9)
    fila[2].imshow(cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)); fila[2].set_title("Filtro homomórfico", fontsize=9)
    for a in fila: a.axis("off")
plt.tight_layout(); plt.show(); plt.close(fig)


### 4. Modelos fundacionales: Grounding DINO + SAM 2 (Grounded-SAM)

- **Grounding DINO** (LLM + visión, detección por texto) propone cajas a partir de un prompt.
- **SAM 2** (Segment Anything) recibe cada caja como *prompt* y devuelve la **máscara** del objeto. La caja final se recalcula desde la máscara, así queda ajustada al borde real.
- La **fracción de llenado** (área de la máscara / área de la caja) filtra cajas falsas: una caja sobre fondo o vidrio da una máscara pobre.

También se incluyen las utilidades geométricas que usa todo el notebook.


In [ ]:
# [Celda 11]  Carga GDINO + SAM 2 y geometría
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import YOLO, SAM

# ---------------- Geometría ----------------
def area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])

def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    u = area(a) + area(b) - inter
    return inter / u if u > 0 else 0.0

def contencion(chica, grande):
    ix1, iy1 = max(chica[0], grande[0]), max(chica[1], grande[1])
    ix2, iy2 = min(chica[2], grande[2]), min(chica[3], grande[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    a = area(chica)
    return inter / a if a > 0 else 0.0

def nms(cajas, thr=0.5):
    keep = []
    for c in sorted(cajas, key=lambda c: -c["score"]):
        if all(iou(c["box"], k["box"]) < thr for k in keep):
            keep.append(c)
    return keep

def filtrar_gondolas(cajas, w, h):
    # Área mínima, sin cajas contenidas en otra mayor, máximo N por foto
    cajas = [c for c in cajas if area(c["box"]) >= CFG["gondola_min_area"] * w * h]
    cajas = sorted(cajas, key=lambda c: -area(c["box"]))
    keep = []
    for c in cajas:
        if all(contencion(c["box"], k["box"]) < CFG["gondola_contain"] for k in keep):
            keep.append(c)
    return sorted(keep, key=lambda c: -c["score"])[:CFG["gondola_max"]]

# ---------------- Grounding DINO ----------------
_GDINO = {}
def cargar_gdino(model_id):
    if model_id not in _GDINO:
        proc = AutoProcessor.from_pretrained(model_id)
        mdl = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(FOUNDATION_DEVICE).eval()
        _GDINO[model_id] = (proc, mdl)
    return _GDINO[model_id]

def descargar_gdino():
    _GDINO.clear(); liberar_gpu()

def _pad_aspecto(image_pil, max_aspect=4.0):
    # Evita 'selected index k out of range' en imágenes muy alargadas (pad abajo/derecha)
    w, h = image_pil.size
    nw, nh = max(w, int(np.ceil(h / max_aspect))), max(h, int(np.ceil(w / max_aspect)))
    if (nw, nh) == (w, h):
        return image_pil
    lienzo = Image.new("RGB", (nw, nh), (114, 114, 114)); lienzo.paste(image_pil, (0, 0))
    return lienzo

@torch.no_grad()
def gdino_detectar(image_pil, prompt, model_id, box_thr, text_thr):
    proc, mdl = cargar_gdino(model_id)
    w, h = image_pil.size
    img_in = _pad_aspecto(image_pil)
    inputs = proc(images=img_in, text=prompt, return_tensors="pt").to(FOUNDATION_DEVICE)
    out = mdl(**inputs)
    kw = dict(text_threshold=text_thr, target_sizes=[img_in.size[::-1]])
    try:
        res = proc.post_process_grounded_object_detection(out, inputs.input_ids, threshold=box_thr, **kw)[0]
    except TypeError:   # versiones antiguas de transformers
        res = proc.post_process_grounded_object_detection(out, inputs.input_ids, box_threshold=box_thr, **kw)[0]
    labels = res.get("text_labels", res.get("labels", []))
    cajas = []
    for b, s, l in zip(res["boxes"].tolist(), res["scores"].tolist(), labels):
        b = [max(0.0, b[0]), max(0.0, b[1]), min(float(w), b[2]), min(float(h), b[3])]
        if b[2] - b[0] >= 2 and b[3] - b[1] >= 2:
            cajas.append({"box": b, "score": float(s), "label": str(l)})
    return cajas

# ---------------- SAM 2 ----------------
sam_model = None
SAM_PESOS_CARGADOS = None

def cargar_sam():
    global sam_model, SAM_PESOS_CARGADOS

    if sam_model is not None:
        return sam_model

    candidatos = (
        (CFG["sam_weights"],)
        if MODO_ULTRA_4GB_3GB
        else (CFG["sam_weights"], "mobile_sam.pt")
    )

    for pesos in candidatos:
        try:
            sam_model = SAM(pesos)
            SAM_PESOS_CARGADOS = pesos
            print("SAM cargado:", pesos)
            return sam_model
        except Exception as ex:
            print(f"No se pudo cargar {pesos}: {ex}")

    return None

def descargar_sam():
    global sam_model, SAM_PESOS_CARGADOS
    sam_model = None
    SAM_PESOS_CARGADOS = None
    liberar_gpu()

def sam_mascaras(img_bgr, boxes, lote_size=None):
    # El tamaño de lote solo cambia memoria/velocidad, no las cajas ni el umbral.
    if not boxes:
        return [None] * len(boxes)

    modelo_sam = cargar_sam()
    if modelo_sam is None:
        return [None] * len(boxes)

    lote_size = CFG["sam_lote"] if lote_size is None else int(lote_size)
    lote_size = max(1, lote_size)

    H, W = img_bgr.shape[:2]
    masks = []
    for i in range(0, len(boxes), lote_size):
        lote = [[float(v) for v in b] for b in boxes[i:i + lote_size]]
        try:
            r = modelo_sam(
                img_bgr,
                bboxes=lote,
                device=FOUNDATION_DEVICE,
                verbose=False
            )[0]
            m = r.masks.data.cpu().numpy() > 0.5 if r.masks is not None else None
        except Exception as ex:
            print("SAM error:", ex); m = None
        if m is None or len(m) != len(lote):
            masks += [None] * len(lote); continue
        for mk in m:
            if mk.shape != (H, W):
                mk = cv2.resize(mk.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST) > 0
            masks.append(mk)

        if MODO_ULTRA_4GB_3GB:
            gc.collect()

    return masks

def sam_refinar(img_bgr, cajas, guardar_mascara=False, max_lado_mascara=512):
    # Ajusta cada caja al borde de su máscara SAM y calcula la fracción de llenado
    masks = sam_mascaras(img_bgr, [c["box"] for c in cajas])
    out = []
    for c, m in zip(cajas, masks):
        c = dict(c); c["box_gdino"] = list(c["box"]); c["fill"] = None; c["refinada"] = False
        if m is not None and m.any():
            ys, xs = np.where(m)
            mb = [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]
            x1, y1, x2, y2 = [int(round(v)) for v in c["box"]]
            c["fill"] = float(m[max(0, y1):y2, max(0, x1):x2].sum() / max(1.0, area(c["box"])))
            if iou(mb, c["box"]) >= CFG["sam_iou_min"]:
                c["box"] = mb; c["refinada"] = True
            if guardar_mascara:
                s = min(1.0, max_lado_mascara / max(m.shape))
                c["mask_small"] = cv2.resize(m.astype(np.uint8), None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
                c["mask_escala"] = s
        out.append(c)
    return out

print("GDINO listo para carga bajo demanda; SAM se cargará solo cuando se necesite.")


### 5. Etapa 1 — Góndola: etiquetado con Grounded-SAM y vista previa de las 187 fotos

Por cada foto: GDINO (sobre la imagen con filtro homomórfico) → NMS → `filtrar_gondolas` (área ≥ 15%, sin cajas contenidas, máximo 2) → SAM ajusta la caja y guarda la máscara, que después sirve para el aplanado.

En la vista previa: **amarillo punteado** = caja original de GDINO, **verde** = caja ajustada con SAM, sombreado = máscara. Los títulos en **rojo** son fotos sin góndola. Revisa las grillas **antes** de seguir; si algo no te convence, cambia `prompt_gondola` o los filtros en la celda de configuración y vuelve a correr esta celda.


In [ ]:
# [Celda 13]  Etiquetado de góndola — ULTRA, dos pasadas
EXCLUIR = {
    "IMG_20261002_121957.jpg",
    "IMG_20261002_122004.jpg"
}

etiquetas_gondola = {}
propuestas_gondola = {}

print("Pasada 1/2: Grounding DINO tiny en CPU")

t0 = time.time()

for i, e in enumerate(entries):
    comprobar_recursos(
        f"gondola_gdino_{i}",
        estricto=True
    )

    img = cv2.imread(
        str(e["path"])
    )

    pre = preprocesar(img)

    pil = Image.fromarray(
        cv2.cvtColor(
            pre,
            cv2.COLOR_BGR2RGB
        )
    )

    cajas = gdino_detectar(
        pil,
        CFG["prompt_gondola"],
        CFG["gdino_gondola_id"],
        CFG["umbral_gondola"],
        CFG["umbral_gondola"]
    )

    cajas = filtrar_gondolas(
        nms(cajas, 0.5),
        e["width"],
        e["height"]
    )

    if e["nombre"] in EXCLUIR:
        cajas = []

    propuestas_gondola[e["nombre"]] = cajas

    del pil
    del pre
    del img
    limpiar_memoria_agresivo()

    if (i + 1) % 25 == 0:
        print(
            f"  GDINO {i+1}/{len(entries)} "
            f"({time.time()-t0:.0f}s)"
        )

descargar_gdino()

comprobar_recursos(
    "entre_gdino_y_sam",
    estricto=True
)

print("Pasada 2/2: MobileSAM en CPU, una caja por lote")

t0 = time.time()

for i, e in enumerate(entries):
    comprobar_recursos(
        f"gondola_sam_{i}",
        estricto=True
    )

    img = cv2.imread(
        str(e["path"])
    )

    cajas = sam_refinar(
        img,
        propuestas_gondola[e["nombre"]],
        guardar_mascara=True,
        max_lado_mascara=384
    )

    for c in cajas:
        c["box"] = c["box_gdino"]

    etiquetas_gondola[e["nombre"]] = cajas

    del img
    limpiar_memoria_agresivo()

    if (i + 1) % 25 == 0:
        print(
            f"  SAM {i+1}/{len(entries)} "
            f"({time.time()-t0:.0f}s)"
        )

descargar_sam()
propuestas_gondola.clear()
gc.collect()

n = [len(v) for v in etiquetas_gondola.values()]

print(
    f"\nFotos con 0 góndolas: {n.count(0)} | "
    f"1: {n.count(1)} | 2: {n.count(2)}"
)

print(
    f"Fotos excluidas a mano: {len(EXCLUIR)}"
)

print(
    "Máscaras SAM disponibles para aplanar:",
    sum(
        c.get("mask_small") is not None
        for v in etiquetas_gondola.values()
        for c in v
    ),
    "/",
    sum(n)
)

if MOSTRAR_PREVIEWS:
    def grilla_gondolas(nombres, titulo, cols=3, rows=2):
        por_fig = cols * rows

        for start in range(0, len(nombres), por_fig):
            lote = nombres[start:start + por_fig]

            fig, axes = plt.subplots(
                rows,
                cols,
                figsize=(cols * 4, rows * 3.5)
            )

            axes = np.atleast_1d(axes).ravel()

            for ax in axes:
                ax.axis("off")

            for ax, nombre in zip(axes, lote):
                e = ENTRY[nombre]

                img = cv2.cvtColor(
                    cv2.imread(str(e["path"])),
                    cv2.COLOR_BGR2RGB
                )

                s = min(
                    1.0,
                    480 / max(img.shape[:2])
                )

                img_s = cv2.resize(
                    img,
                    None,
                    fx=s,
                    fy=s
                )

                ax.imshow(img_s)

                for c in etiquetas_gondola[nombre]:
                    b = [v * s for v in c["box"]]

                    ax.add_patch(
                        plt.Rectangle(
                            (b[0], b[1]),
                            b[2] - b[0],
                            b[3] - b[1],
                            fill=False,
                            ec="lime",
                            lw=2
                        )
                    )

            plt.tight_layout()
            plt.show()
            plt.close(fig)
            limpiar_memoria_agresivo()

    grilla_gondolas(
        sorted(etiquetas_gondola),
        "Etiquetas de góndola"
    )
else:
    print(
        "Vista de 187 imágenes omitida "
        "en modo ULTRA."
    )

registrar_recursos(
    "post_etiquetado_gondola"
)


In [ ]:
# [Celda 14]  Dataset YOLO de góndola
limpiar_dir(GONDOLA_DS)
for split in ("train", "val"):
    (GONDOLA_DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (GONDOLA_DS / "labels" / split).mkdir(parents=True, exist_ok=True)

for e in entries:
    split = e["split"]
    shutil.copy2(e["path"], GONDOLA_DS / "images" / split / e["nombre"])
    w, h = e["width"], e["height"]
    with open(GONDOLA_DS / "labels" / split / (Path(e["nombre"]).stem + ".txt"), "w") as f:
        for c in etiquetas_gondola[e["nombre"]]:
            x1, y1, x2, y2 = c["box"]
            f.write(f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}\n")

YAML_GONDOLA = GONDOLA_DS / "data.yaml"
yaml.safe_dump({"path": str(GONDOLA_DS), "train": "images/train", "val": "images/val",
                "names": {0: "gondola"}}, open(YAML_GONDOLA, "w"), sort_keys=False)
print(open(YAML_GONDOLA).read())

In [ ]:
# [Celda 15]  Entrenamiento YOLO góndola
liberar_gpu()
model_g = YOLO(CFG["yolo_gondola"])
model_g.train(
    data=str(YAML_GONDOLA),
    epochs=CFG["ep_gondola"],
    patience=20,
    imgsz=CFG["imgsz_gondola"],
    batch=CFG["batch_gondola"],
    device=DEVICE,
    seed=SEED,
    project=str(RUNS_DIR),
    name="gondola",
    exist_ok=True,
    degrees=5,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    **argumentos_entrenamiento_eco()
)
BEST_GONDOLA = Path(model_g.trainer.save_dir) / "weights" / "best.pt"
del model_g
gc.collect()
liberar_gpu()
best_gondola = YOLO(str(BEST_GONDOLA))
m = best_gondola.val(
    data=str(YAML_GONDOLA),
    imgsz=CFG["imgsz_gondola"],
    device=DEVICE,
    verbose=False,
    **argumentos_validacion_eco()
)
METRICAS = {"gondola": {"P": float(m.box.mp), "R": float(m.box.mr), "mAP50": float(m.box.map50)}}
print(f"\nGóndola — P={m.box.mp:.1%}  R={m.box.mr:.1%}  mAP50={m.box.map50:.3f}")

# El entrenador ya fue liberado antes de cargar best_gondola.
gc.collect()
liberar_gpu()
registrar_recursos("post_train_gondola")


### 6. Aplanado (rectificación de perspectiva) de cada góndola

Desde la máscara SAM de la góndola: contorno → envolvente convexa → polígono de 4 vértices (`approxPolyDP`) → **homografía** a un rectángulo frontal. Así las repisas quedan horizontales y los productos tienen un tamaño parejo, lo que ayuda al detector y al cálculo de vacíos.

Si no se encuentra un cuadrilátero válido (convexo y que cubra ≥ 60% de la caja), se usa el recorte de la caja (**fallback**). Se **descartan** los recortes con lado < 200 px o proporción > 4:1 (eran los recortes "chicos o planos").

Todas las góndolas quedan normalizadas a un lado máximo de 1600 px. Se guarda la matriz `M` (foto → góndola aplanada) para poder volver a la foto original.


In [ ]:
# [Celda 17]  Aplanado por homografía
def ordenar_puntos(pts):
    pts = np.asarray(pts, dtype=np.float32)
    s, d = pts.sum(1), np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)], pts[np.argmax(s)], pts[np.argmax(d)]], dtype=np.float32)

def quad_desde_mascara(mask_small, escala, box):
    if mask_small is None:
        return None
    m = (mask_small > 0).astype(np.uint8)

    # Recortar la máscara a la caja y validar que sea "góndola" y no piso/techo
    bx = [int(v * escala) for v in box]
    recorte = np.zeros_like(m); recorte[bx[1]:bx[3], bx[0]:bx[2]] = 1
    m = m * recorte
    llenado = m.sum() / max(1, (bx[2] - bx[0]) * (bx[3] - bx[1]))
    franja_inf = m[max(bx[1], bx[3] - (bx[3] - bx[1]) // 8):bx[3], bx[0]:bx[2]]
    if not (0.40 <= llenado <= 0.92) or franja_inf.mean() > 0.85:
        return None   # máscara dudosa (muy vacía, todo el cuadro, o piso pegado abajo) -> fallback bbox

    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    hull = cv2.convexHull(max(cnts, key=cv2.contourArea))
    peri = cv2.arcLength(hull, True)
    quad = None
    for eps in np.linspace(0.01, 0.10, 10):
        ap = cv2.approxPolyDP(hull, eps * peri, True)
        if len(ap) == 4:
            quad = ap.reshape(4, 2).astype(np.float32); break
    if quad is None:
        return None
    quad = ordenar_puntos(quad / escala)
    if not cv2.isContourConvex(quad.reshape(-1, 1, 2).astype(np.int32)):
        return None
    if cv2.contourArea(quad) < CFG["rect_area_min_vs_bbox"] * area(box):
        return None
    return quad

def rectificar(img_bgr, box, quad):
    # Devuelve (imagen aplanada, M foto->aplanada, método)
    H_img, W_img = img_bgr.shape[:2]
    if quad is not None:
        tl, tr, br, bl = quad
        W = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
        H = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
        dst = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=np.float32)
        M = cv2.getPerspectiveTransform(quad, dst)
        out = cv2.warpPerspective(img_bgr, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        metodo = "homografia"
    else:
        x1, y1 = max(0, int(box[0])), max(0, int(box[1]))
        x2, y2 = min(W_img, int(box[2])), min(H_img, int(box[3]))
        out = img_bgr[y1:y2, x1:x2].copy()
        M = np.array([[1, 0, -x1], [0, 1, -y1], [0, 0, 1]], dtype=np.float64)
        metodo = "bbox"
    if out.size == 0:
        return None, None, metodo
    s = min(1.0, CFG["rect_max_lado"] / max(out.shape[:2]))
    if s < 1:
        out = cv2.resize(out, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        M = np.diag([s, s, 1.0]) @ M
    return out, M, metodo

def rect_valida(img):
    if img is None:
        return False
    h, w = img.shape[:2]
    return min(h, w) >= CFG["rect_min_lado"] and max(h, w) / max(1, min(h, w)) <= CFG["rect_aspect_max"]

def cajas_a_foto(cajas_rect, M):
    # Lleva cajas de la góndola aplanada a coordenadas de la foto original
    Minv = np.linalg.inv(M)
    out = []
    for b in cajas_rect:
        pts = np.array([[b[0], b[1]], [b[2], b[1]], [b[2], b[3]], [b[0], b[3]]], dtype=np.float32).reshape(-1, 1, 2)
        p = cv2.perspectiveTransform(pts, Minv).reshape(-1, 2)
        out.append([float(p[:, 0].min()), float(p[:, 1].min()), float(p[:, 0].max()), float(p[:, 1].max())])
    return out

limpiar_dir(RECT_DIR)
gondolas_rect, descartadas = [], []
for e in entries:
    img = cv2.imread(str(e["path"]))
    for k, c in enumerate(etiquetas_gondola[e["nombre"]]):
        quad = quad_desde_mascara(c.get("mask_small"), c.get("mask_escala", 1.0), c["box"])
        rect, M, metodo = rectificar(img, c["box"], quad)
        gid = f"{Path(e['nombre']).stem}_g{k}"
        if not rect_valida(rect):
            descartadas.append(gid); continue
        ruta = RECT_DIR / f"{gid}.jpg"
        cv2.imwrite(str(ruta), rect)
        gondolas_rect.append({"id": gid, "foto": e["nombre"], "split": e["split"], "grupo": e["grupo"],
                              "kind": e["kind"], "path": ruta, "M": M, "quad": quad, "metodo": metodo,
                              **metricas_calidad(rect)})
GR = {g["id"]: g for g in gondolas_rect}
print(f"Góndolas aplanadas: {len(gondolas_rect)} | descartadas (chicas/planas): {len(descartadas)}")
print("Método:", pd.Series([g['metodo'] for g in gondolas_rect]).value_counts().to_dict())

# Antes / después
if MOSTRAR_PREVIEWS:
    muestra = random.Random(SEED).sample(
        gondolas_rect,
        min(3, len(gondolas_rect))
    )

    fig, axes = plt.subplots(
        len(muestra),
        2,
        figsize=(11, 3.5 * len(muestra))
    )

    axes = np.atleast_2d(axes)

    for fila, g in zip(axes, muestra):
        img = cv2.cvtColor(
            cv2.imread(
                str(
                    ENTRY[g["foto"]]["path"]
                )
            ),
            cv2.COLOR_BGR2RGB
        )

        fila[0].imshow(img)

        fila[1].imshow(
            cv2.cvtColor(
                cv2.imread(str(g["path"])),
                cv2.COLOR_BGR2RGB
            )
        )

        for a in fila:
            a.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)
    limpiar_memoria_agresivo()
else:
    print(
        "Comparación antes/después omitida "
        "en modo ULTRA."
    )


### 7. Etapa 2 — Producto: GDINO por mosaicos + filtros + SAM

Sobre cada góndola aplanada (con filtro homomórfico):
1. **Mosaicos (tiling):** GDINO corre sobre la imagen completa y sobre mosaicos de 640 px con 25% de solapamiento. Los productos chicos se detectan mucho mejor así. Se descartan las cajas cortadas por el borde interior de un mosaico, porque aparecen completas en otro.
2. **Etiquetas de precio:** el prompt incluye `price tag.`. Esas cajas se **eliminan**, junto con los productos que caen casi enteros dentro de una etiqueta.
3. **Cajas-grupo:** se elimina una caja que contiene ≥ 3 cajas de producto, porque marca un grupo y no un producto.
4. **Tamaño y forma:** se eliminan las cajas con área muy distinta de la mediana de la góndola o con proporción extrema.
5. **SAM:** ajusta cada caja a su máscara. Las cajas con llenado < 35% se descartan (fondo o reflejo).

Primero se muestra el paso a paso en 3 góndolas, después se corre sobre todas.


In [ ]:
# [Celda 19]  Funciones de producto + paso a paso
def generar_tiles(w, h, tile, overlap):
    if max(w, h) <= tile:
        return [(0, 0, w, h)]
    step = max(1, int(tile * (1 - overlap)))
    def ejes(n):
        if n <= tile:
            return [0]
        xs = list(range(0, n - tile + 1, step))
        if xs[-1] + tile < n:
            xs.append(n - tile)
        return xs
    return [(x, y, min(x + tile, w), min(y + tile, h)) for y in ejes(h) for x in ejes(w)]

def es_precio(c):
    l = c["label"].lower()
    return "price" in l or "tag" in l

def gdino_tiles(img_pre_bgr):
    h, w = img_pre_bgr.shape[:2]
    pil = Image.fromarray(cv2.cvtColor(img_pre_bgr, cv2.COLOR_BGR2RGB))
    args = (CFG["prompt_producto"], CFG["gdino_producto_id"], CFG["umbral_producto"], CFG["text_umbral_producto"])
    cajas = gdino_detectar(pil, *args)
    tiles = generar_tiles(w, h, CFG["tile"], CFG["tile_overlap"])
    if len(tiles) > 1:
        for (x1, y1, x2, y2) in tiles:
            for c in gdino_detectar(pil.crop((x1, y1, x2, y2)), *args):
                b = c["box"]
                corta = ((b[0] <= 2 and x1 > 0) or (b[1] <= 2 and y1 > 0) or
                         (b[2] >= (x2 - x1) - 2 and x2 < w) or (b[3] >= (y2 - y1) - 2 and y2 < h))
                if not corta:
                    c["box"] = [b[0] + x1, b[1] + y1, b[2] + x1, b[3] + y1]
                    cajas.append(c)
    return cajas

def filtrar_productos(cajas):
    precios = nms([c for c in cajas if es_precio(c)], 0.5)
    prods = nms([c for c in cajas if not es_precio(c)], 0.5)
    prods = [c for c in prods if all(contencion(c["box"], p["box"]) < 0.6 for p in precios)]
    # Cajas-grupo: contienen el centro de >= N otras cajas
    def n_hijos(c):
        return sum(1 for o in prods if o is not c and area(o["box"]) < area(c["box"])
                   and c["box"][0] <= (o["box"][0]+o["box"][2])/2 <= c["box"][2]
                   and c["box"][1] <= (o["box"][1]+o["box"][3])/2 <= c["box"][3])
    prods = [c for c in prods if n_hijos(c) < CFG["prod_grupo_min_hijos"]]
    if prods:
        med = float(np.median([area(c["box"]) for c in prods]))
        def ok(c):
            a = area(c["box"]); bw = c["box"][2]-c["box"][0]; bh = c["box"][3]-c["box"][1]
            return (CFG["prod_area_min_vs_med"] * med <= a <= CFG["prod_area_max_vs_med"] * med and
                    CFG["prod_aspect_min"] <= bw / max(bh, 1e-6) <= CFG["prod_aspect_max"])
        prods = [c for c in prods if ok(c)]
    return prods, precios

def etiquetar_productos(rect_bgr):
    pre = preprocesar(rect_bgr)
    crudas = gdino_tiles(pre)
    filtradas, precios = filtrar_productos(crudas)
    refinadas = sam_refinar(rect_bgr, filtradas)
    finales = [c for c in refinadas if c["fill"] is None or c["fill"] >= CFG["sam_fill_min"]]
    finales = nms(finales, 0.6)
    return {"crudas": crudas, "filtradas": filtradas, "precios": precios, "finales": finales}

def dibujar(ax, img_rgb, grupos, titulo):
    ax.imshow(img_rgb); ax.axis("off"); ax.set_title(titulo, fontsize=9)
    for cajas, color, lw, ls in grupos:
        for b in cajas:
            b = b["box"] if isinstance(b, dict) else b
            ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, ec=color, lw=lw, ls=ls))

# Paso a paso en 3 góndolas
for g in random.Random(1).sample(gondolas_rect, min(3, len(gondolas_rect))):
    rect = cv2.imread(str(g["path"]))
    r = etiquetar_productos(rect)
    rgb = cv2.cvtColor(rect, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 4, figsize=(26, 6))
    axes[0].imshow(cv2.cvtColor(preprocesar(rect), cv2.COLOR_BGR2RGB)); axes[0].axis("off")
    axes[0].set_title(f"{g['id']} — homomórfico", fontsize=9)
    dibujar(axes[1], rgb, [(r["crudas"], "yellow", 1, "-")], f"GDINO crudo (mosaicos): {len(r['crudas'])}")
    dibujar(axes[2], rgb, [(r["filtradas"], "cyan", 1, "-"), (r["precios"], "magenta", 1.5, "-")],
            f"Filtrado: {len(r['filtradas'])} prod. | {len(r['precios'])} precios (magenta)")
    dibujar(axes[3], rgb, [(r["finales"], "lime", 1.5, "-")], f"Final con SAM: {len(r['finales'])}")
    plt.tight_layout(); plt.show()

In [ ]:
CACHE_ETIQ = (
    PERSIST_DIR
    / "cache_etiquetas_gdino.json"
)

CACHE_ETIQ_DRIVE = buscar_hijo_drive(
    SHARED_DRIVE_FOLDER_ID,
    "cache_etiquetas_gdino.json"
)

if (
    CFG.get(
        "usar_cache_etiquetas"
    )
    and not CACHE_ETIQ.exists()
    and CACHE_ETIQ_DRIVE is not None
):
    descargar_archivo_drive(
        CACHE_ETIQ_DRIVE[
            "id"
        ],
        CACHE_ETIQ
    )

    print(
        "Caché GDINO descargado "
        "desde la carpeta compartida."
    )


_claves = [
    "prompt_producto",
    "gdino_producto_id",
    "umbral_producto",
    "text_umbral_producto",
    "tile",
    "tile_overlap",
    "prod_area_max_vs_med",
    "prod_area_min_vs_med",
    "prod_aspect_min",
    "prod_aspect_max",
    "prod_grupo_min_hijos",
    "sam_fill_min",
    "usar_homomorfico"
]

FIRMA = json.dumps(
    {
        k: CFG[
            k
        ]
        for k
        in _claves
    }
    | {
        "ids": sorted(
            g[
                "id"
            ]
            for g
            in gondolas_rect
        )
    },
    sort_keys=True
)

cache = (
    json.load(
        open(
            CACHE_ETIQ
        )
    )
    if (
        CFG.get(
            "usar_cache_etiquetas"
        )
        and CACHE_ETIQ.exists()
    )
    else None
)

if (
    cache
    and cache.get(
        "firma"
    )
    == FIRMA
):
    etiquetas_producto = cache[
        "etiquetas"
    ]

    print(
        "Etiquetas GDINO cargadas "
        "desde caché."
    )

else:
    if (
        MODO_ULTRA_4GB_3GB
        and not PERMITIR_GDINO_PRODUCTO_SIN_CACHE
    ):
        etiquetas_producto = {
            g["id"]: []
            for g in gondolas_rect
        }

        for g in gondolas_rect:
            g["n_precios"] = 0

        print(
            "No existe un caché GDINO+SAM compatible. "
            "En modo ULTRA no se carga Grounding DINO base. "
            "Las etiquetas de producto se generarán con "
            "el maestro HITL."
        )

    else:
        etiquetas_producto = {}
        t0 = time.time()

        for i, g in enumerate(gondolas_rect):
            comprobar_recursos(
                f"gdino_producto_{i}",
                estricto=True
            )

            rect = cv2.imread(str(g["path"]))
            r = etiquetar_productos(rect)

            etiquetas_producto[g["id"]] = [
                {
                    "box": c["box"],
                    "score": c["score"],
                    "fill": c["fill"]
                }
                for c in r["finales"]
            ]

            g["n_precios"] = len(r["precios"])

            del rect
            del r
            limpiar_memoria_agresivo()

        with open(
            CACHE_ETIQ,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                {
                    "firma": FIRMA,
                    "etiquetas": etiquetas_producto
                },
                f
            )

        if DRIVE_CAN_WRITE:
            subir_archivo_drive(
                CACHE_ETIQ,
                SHARED_DRIVE_FOLDER_ID
            )


n = np.array(
    [
        len(
            v
        )
        for v
        in etiquetas_producto.values()
    ]
)

print(
    f"\nProductos por góndola: "
    f"media={n.mean():.1f}  "
    f"mediana={np.median(n):.0f}  "
    f"min={n.min()}  "
    f"max={n.max()}"
)

print(
    f"Góndolas con < 3 productos: "
    f"{(n < 3).sum()} / "
    f"{len(n)}"
)


# Las pseudoetiquetas ya están materializadas/caché; liberar modelos pesados.
descargar_gdino()
descargar_sam()
registrar_recursos("post_etiquetado_producto")


In [ ]:
# [Celda 21]  Vista previa paginada de productos
# Vista previa paginada de las etiquetas de producto (cambia PAGINA para ver más)
PAGINA, POR_PAGINA = 0, 12
ids = sorted(etiquetas_producto)
lote = ids[PAGINA * POR_PAGINA:(PAGINA + 1) * POR_PAGINA]
fig, axes = plt.subplots(3, 4, figsize=(24, 15))
for ax in axes.ravel():
    ax.axis("off")
for ax, gid in zip(axes.ravel(), lote):
    rgb = cv2.cvtColor(cv2.imread(str(GR[gid]["path"])), cv2.COLOR_BGR2RGB)
    dibujar(ax, rgb, [(etiquetas_producto[gid], "lime", 1.2, "-")],
            f"{gid} — {len(etiquetas_producto[gid])} prod. [{GR[gid]['split']}]")
fig.suptitle(f"Página {PAGINA} de {len(ids)//POR_PAGINA}", fontsize=12)
plt.tight_layout(); plt.show(); plt.close(fig)


### 7b. Maestro con etiquetas humanas y etiquetas híbridas

Se mantiene el mismo dataset de productos. Antes de entrenar se sanitizan todas las etiquetas YOLO para
evitar coordenadas fuera de `[0,1]` y duplicados exactos.

Esta corrección es importante porque en la ejecución anterior Ultralytics reportó al menos una imagen
ignorada por coordenadas fuera de rango y varias etiquetas duplicadas. Un dataset sin esas inconsistencias
permite mejorar el modelo sin recurrir a un umbral de inferencia más bajo.


In [ ]:
# [Celda 23]  Descarga y conversión del dataset Humans in the Loop (CC0, 45 fotos, ~11.7k cajas producto/precio)
%pip install -q kagglehub
import glob
HITL_RAW = DATA / "hitl_raw"
HITL_DS = DATA / "ds_hitl_tiles"          # dataset del maestro (mosaicos 640)
HITL_FULL = DATA / "hitl_full"            # imágenes completas (se suman al train del alumno)

def descargar_hitl():
    if any(HITL_RAW.rglob("*.json")):
        return HITL_RAW
    try:
        import kagglehub
        p = Path(kagglehub.dataset_download(CFG["hitl_kaggle"]))
        shutil.copytree(p, HITL_RAW, dirs_exist_ok=True)
    except Exception as ex:
        raise RuntimeError(
            "No fue posible descargar el dataset "
            "Humans in the Loop con kagglehub. "
            "La carpeta del proyecto no utiliza ZIP. "
            f"Error original: {ex}"
        )
    return HITL_RAW

def leer_anotaciones_hitl(raw):
    # Devuelve {ruta_imagen: [(clase, x1, y1, x2, y2), ...]} con clase 0=producto, 1=precio.
    # Soporta formato Supervisely (un .json por imagen) y COCO (un .json con 'images'/'annotations').
    imgs = {p.name: p for p in raw.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    def clase(nombre):
        n = str(nombre).lower()
        return 1 if ("price" in n or "precio" in n or "tag" in n) else 0
    out = {}
    for js in raw.rglob("*.json"):
        if js.name == "meta.json":
            continue
        d = json.load(open(js))
        if isinstance(d, dict) and "objects" in d:                     # Supervisely
            nombre = js.name[:-5]                                       # "001.jpg.json" -> "001.jpg"
            ruta = imgs.get(nombre) or next((p for n, p in imgs.items() if Path(n).stem == Path(nombre).stem), None)
            if ruta is None:
                continue
            cajas = []
            for o in d["objects"]:
                pts = o.get("points", {}).get("exterior", [])
                if len(pts) >= 2:
                    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
                    cajas.append((clase(o.get("classTitle", "")), min(xs), min(ys), max(xs), max(ys)))
            out[ruta] = cajas
        elif isinstance(d, dict) and "annotations" in d and "images" in d:   # COCO
            cats = {c["id"]: c["name"] for c in d.get("categories", [])}
            id2img = {im["id"]: imgs.get(Path(im["file_name"]).name) for im in d["images"]}
            for a in d["annotations"]:
                ruta = id2img.get(a["image_id"])
                if ruta is not None:
                    x, y, w, h = a["bbox"]
                    out.setdefault(ruta, []).append((clase(cats.get(a["category_id"], "")), x, y, x + w, y + h))
    return out

def escribir_mosaicos(img, cajas, destino_img, destino_lbl, stem, tile, overlap, min_vis=0.6):
    h, w = img.shape[:2]
    n = 0
    for (x1, y1, x2, y2) in generar_tiles(w, h, tile, overlap):
        tw, th = x2 - x1, y2 - y1
        lineas = []
        for (c, bx1, by1, bx2, by2) in cajas:
            cx1, cy1, cx2, cy2 = max(bx1, x1), max(by1, y1), min(bx2, x2), min(by2, y2)
            if cx2 <= cx1 or cy2 <= cy1:
                continue
            if (cx2 - cx1) * (cy2 - cy1) < min_vis * (bx2 - bx1) * (by2 - by1):
                continue                                               # producto muy cortado por el mosaico
            lineas.append(f"{c} {((cx1+cx2)/2-x1)/tw:.6f} {((cy1+cy2)/2-y1)/th:.6f} {(cx2-cx1)/tw:.6f} {(cy2-cy1)/th:.6f}")
        nombre = f"{stem}_t{n}"
        cv2.imwrite(str(destino_img / f"{nombre}.jpg"), img[y1:y2, x1:x2])
        open(destino_lbl / f"{nombre}.txt", "w").write("\n".join(lineas))
        n += 1
    return n

raw = descargar_hitl()
ANOT_HITL = leer_anotaciones_hitl(raw)
assert ANOT_HITL, f"No se encontraron anotaciones en {raw}. Revisa la estructura con: !find {raw} | head"
n_prod = sum(c[0] == 0 for v in ANOT_HITL.values() for c in v); n_prec = sum(c[0] == 1 for v in ANOT_HITL.values() for c in v)
print(f"HITL: {len(ANOT_HITL)} imágenes | {n_prod} productos | {n_prec} precios")

rutas = sorted(ANOT_HITL)
random.Random(SEED).shuffle(rutas)
HITL_SPLIT = {r: ("val" if i < CFG["hitl_val_imgs"] else "train") for i, r in enumerate(rutas)}

limpiar_dir(HITL_DS); limpiar_dir(HITL_FULL)
for s in ("train", "val"):
    (HITL_DS / "images" / s).mkdir(parents=True); (HITL_DS / "labels" / s).mkdir(parents=True)
    (HITL_FULL / "images" / s).mkdir(parents=True); (HITL_FULL / "labels" / s).mkdir(parents=True)

n_tiles = 0
for ruta, cajas in ANOT_HITL.items():
    img = cv2.imread(str(ruta)); s = min(1.0, CFG["hitl_max_lado"] / max(img.shape[:2]))
    if s < 1:
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    cajas = [(c, x1 * s, y1 * s, x2 * s, y2 * s) for (c, x1, y1, x2, y2) in cajas]
    sp, stem = HITL_SPLIT[ruta], f"hitl_{Path(ruta).stem}"
    n_tiles += escribir_mosaicos(img, cajas, HITL_DS / "images" / sp, HITL_DS / "labels" / sp, stem,
                                 CFG["hitl_tile"], CFG["hitl_tile_overlap"])
    # Imagen completa, solo clase producto -> se suma al train del alumno
    h, w = img.shape[:2]
    cv2.imwrite(str(HITL_FULL / "images" / sp / f"{stem}.jpg"), img)
    open(HITL_FULL / "labels" / sp / f"{stem}.txt", "w").write("\n".join(
        f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}" for (c, x1, y1, x2, y2) in cajas if c == 0))

YAML_HITL = HITL_DS / "data.yaml"
yaml.safe_dump({"path": str(HITL_DS), "train": "images/train", "val": "images/val",
                "names": {0: "producto", 1: "precio"}}, open(YAML_HITL, "w"), sort_keys=False)
print(f"Mosaicos para el maestro: {n_tiles} | test humano: {CFG['hitl_val_imgs']} imágenes")

# Vista de 2 imágenes con sus etiquetas humanas
fig, ax = plt.subplots(1, 2, figsize=(22, 8))
for a, ruta in zip(ax, rutas[:2]):
    dibujar(a, cv2.cvtColor(cv2.imread(str(ruta)), cv2.COLOR_BGR2RGB),
            [([c[1:] for c in ANOT_HITL[ruta] if c[0] == 0], "lime", 0.8, "-"),
             ([c[1:] for c in ANOT_HITL[ruta] if c[0] == 1], "magenta", 1.0, "-")],
            f"{ruta.name} — verde=producto, magenta=precio (etiqueta humana)")
plt.tight_layout(); plt.show()


In [ ]:
# [Celda 24]  Maestro HITL — entrenamiento ULTRA con fallback
liberar_gpu()

comprobar_recursos(
    "antes_train_maestro",
    estricto=True
)

CANDIDATOS_MAESTRO = [
    ("yolo11s.pt", 640),
    ("yolo11s.pt", 512),
    ("yolo11n.pt", 640),
    ("yolo11n.pt", 512)
]

BEST_MAESTRO = None
MAESTRO_MODELO_USADO = None
MAESTRO_IMGSZ_USADO = None
ultimo_error_maestro = None

for pesos, imgsz_try in CANDIDATOS_MAESTRO:
    print(
        "Probando maestro:",
        pesos,
        "| imgsz",
        imgsz_try
    )

    modelo_m = None

    try:
        limpiar_memoria_agresivo()

        modelo_m = YOLO(pesos)

        modelo_m.train(
            data=str(YAML_HITL),
            epochs=CFG["maestro_epochs"],
            patience=30,
            imgsz=imgsz_try,
            batch=1,
            device=DEVICE,
            seed=SEED,
            project=str(RUNS_DIR),
            name=(
                "maestro_hitl_ultra_"
                + Path(pesos).stem
                + f"_{imgsz_try}"
            ),
            exist_ok=True,
            degrees=3,
            perspective=0.0005,
            scale=0.5,
            hsv_h=0.015,
            hsv_s=0.6,
            hsv_v=0.4,
            fliplr=0.5,
            mosaic=1.0,
            **argumentos_entrenamiento_eco()
        )

        BEST_MAESTRO = (
            Path(modelo_m.trainer.save_dir)
            / "weights"
            / "best.pt"
        )

        MAESTRO_MODELO_USADO = pesos
        MAESTRO_IMGSZ_USADO = imgsz_try

        del modelo_m
        modelo_m = None
        limpiar_memoria_agresivo()
        break

    except RuntimeError as ex:
        ultimo_error_maestro = ex

        if modelo_m is not None:
            del modelo_m

        limpiar_memoria_agresivo()

        if not es_error_oom(ex):
            raise

        print(
            "OOM con",
            pesos,
            imgsz_try,
            "-> se prueba una opción menor."
        )

assert BEST_MAESTRO is not None, (
    "Ningún candidato del maestro cabía "
    "en el límite de VRAM. Último error: "
    f"{ultimo_error_maestro}"
)

print(
    "Maestro seleccionado:",
    MAESTRO_MODELO_USADO,
    "| imgsz",
    MAESTRO_IMGSZ_USADO
)

maestro = YOLO(str(BEST_MAESTRO))

m = maestro.val(
    data=str(YAML_HITL),
    imgsz=MAESTRO_IMGSZ_USADO,
    device=DEVICE,
    max_det=1000,
    verbose=False,
    **argumentos_validacion_eco()
)

for idx, cid in enumerate(m.box.ap_class_index):
    print(
        f"  {['producto','precio'][int(cid)]}: "
        f"P={m.box.p[idx]:.1%} "
        f"R={m.box.r[idx]:.1%} "
        f"AP50={m.box.ap50[idx]:.3f}"
    )

METRICAS["maestro_hitl_test_humano"] = {
    "P": float(m.box.mp),
    "R": float(m.box.mr),
    "mAP50": float(m.box.map50),
    "modelo": MAESTRO_MODELO_USADO,
    "imgsz": MAESTRO_IMGSZ_USADO
}

CFG["maestro_imgsz"] = MAESTRO_IMGSZ_USADO

gc.collect()
liberar_gpu()

registrar_recursos(
    "post_train_maestro"
)


In [ ]:
# [Celda 25]  Etiquetas híbridas en NUESTRAS góndolas: maestro (mosaicos) + GDINO+SAM, fusión por acuerdo
def maestro_tiles(img_bgr, conf=None):
    # Devuelve (productos, precios) en coordenadas de la góndola aplanada
    conf = CFG["maestro_conf"] if conf is None else conf
    h, w = img_bgr.shape[:2]
    prods, precios = [], []
    for (x1, y1, x2, y2) in generar_tiles(w, h, CFG["hitl_tile"], 0.25):
        tw, th = x2 - x1, y2 - y1
        r = maestro.predict(img_bgr[y1:y2, x1:x2], imgsz=CFG["maestro_imgsz"], conf=conf, max_det=1000,
                            device=DEVICE, verbose=False)[0]
        if r.boxes is None:
            continue
        for b, s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            b = [float(v) for v in b]
            if ((b[0] <= 2 and x1 > 0) or (b[1] <= 2 and y1 > 0) or
                    (b[2] >= tw - 2 and x2 < w) or (b[3] >= th - 2 and y2 < h)):
                continue                                               # cortada por borde interior del mosaico
            caja = {"box": [b[0] + x1, b[1] + y1, b[2] + x1, b[3] + y1], "score": float(s)}
            (precios if int(c) == 1 else prods).append(caja)
    precios = nms(precios, 0.5)
    prods = [p for p in nms(prods, 0.55) if all(contencion(p["box"], q["box"]) < 0.6 for q in precios)]
    return prods, precios

def fusionar(maes, gdino):
    # Si ambos coinciden -> caja segura (promedio ponderado). Si solo uno -> se exige confianza alta.
    usados_g, out = set(), []
    for a in sorted(maes, key=lambda c: -c["score"]):
        mejor, jm = 0.0, -1
        for j, g in enumerate(gdino):
            if j not in usados_g:
                v = iou(a["box"], g["box"])
                if v > mejor:
                    mejor, jm = v, j
        if mejor >= CFG["fusion_iou"]:
            g = gdino[jm]; usados_g.add(jm)
            wa, wg = a["score"], g["score"]
            out.append({"box": [(wa * p + wg * q) / (wa + wg) for p, q in zip(a["box"], g["box"])],
                        "score": max(wa, wg), "fuente": "ambos"})
        elif a["score"] >= CFG["maestro_conf_solo"]:
            out.append({"box": a["box"], "score": a["score"], "fuente": "maestro"})
    for j, g in enumerate(gdino):
        if j not in usados_g and g["score"] >= CFG["gdino_conf_solo"]:
            out.append({"box": g["box"], "score": g["score"], "fuente": "gdino"})
    return nms(out, 0.6)

def filtrar_sin_textura(cajas, img_bgr):
    # Cajas que SOLO vio el maestro y que no tienen textura (piso, pared lisa) se descartan.
    if not cajas:
        return cajas
    bordes = np.abs(cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_32F))
    def dens(b):
        x1, y1, x2, y2 = [int(v) for v in b]
        r = bordes[max(0, y1):y2, max(0, x1):x2]
        return float(r.mean()) if r.size else 0.0
    ref = np.median([dens(c["box"]) for c in cajas if c["fuente"] != "maestro"] or [dens(c["box"]) for c in cajas])
    return [c for c in cajas if c["fuente"] != "maestro" or dens(c["box"]) >= CFG["maestro_textura_min"] * ref]

if "etiquetas_gdino" not in globals():
    etiquetas_gdino = {k: list(v) for k, v in etiquetas_producto.items()}   # respaldo de las etiquetas solo-GDINO

etiquetas_maestro, precios_maestro, etiquetas_hibridas = {}, {}, {}
t0 = time.time()
for i, g in enumerate(gondolas_rect):
    comprobar_recursos(
        f"hibridacion_{i}",
        estricto=True
    )

    rect = cv2.imread(
        str(g["path"])
    )
    etiquetas_maestro[g["id"]], precios_maestro[g["id"]] = maestro_tiles(rect)
    gd = [c for c in etiquetas_gdino[g["id"]]                                  # GDINO sin cajas sobre precios
          if all(contencion(c["box"], q["box"]) < 0.6 for q in precios_maestro[g["id"]])]
    etiquetas_hibridas[g["id"]] = filtrar_sin_textura(fusionar(etiquetas_maestro[g["id"]], gd), rect)
    del rect
    limpiar_memoria_agresivo()

    if (i + 1) % 25 == 0:
        print(
            f"  {i+1}/{len(gondolas_rect)} "
            f"({time.time()-t0:.0f}s)"
        )

def _n(d): return np.array([len(v) for v in d.values()])
print(f"\nProductos por góndola (mediana): GDINO={np.median(_n(etiquetas_gdino)):.0f} | "
      f"MAESTRO={np.median(_n(etiquetas_maestro)):.0f} | HÍBRIDO={np.median(_n(etiquetas_hibridas)):.0f}")
print("Origen de las cajas híbridas:", pd.Series([c["fuente"] for v in etiquetas_hibridas.values() for c in v]).value_counts().to_dict())
print("Mediana por tipo de foto:", pd.DataFrame([{"kind": GR[k]["kind"], "n": len(v)} for k, v in etiquetas_hibridas.items()])
      .groupby("kind")["n"].median().to_dict())


In [ ]:
# [Celda 26]  Activar etiquetas híbridas / maestro
if MOSTRAR_PREVIEWS:
    print(
        "MOSTRAR_PREVIEWS=True: "
        "puedes visualizar las etiquetas manualmente."
    )
else:
    print(
        "Comparación visual de etiquetas omitida "
        "en modo ULTRA."
    )

etiquetas_producto = {
    k: [
        {
            "box": c["box"],
            "score": c["score"],
            "fill": None
        }
        for c in v
    ]
    for k, v
    in etiquetas_hibridas.items()
}

CFG["yolo_producto"] = str(BEST_MAESTRO)

print(
    "etiquetas_producto <- híbridas/maestro | "
    "YOLO producto parte del mejor maestro HITL"
)

try:
    del maestro
except NameError:
    pass

gc.collect()
liberar_gpu()

registrar_recursos(
    "post_hibridacion"
)


In [ ]:
# [Celda 27]  Dataset YOLO de producto + fotos HITL
limpiar_dir(PRODUCTO_DS)
for split in ("train", "val"):
    (PRODUCTO_DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (PRODUCTO_DS / "labels" / split).mkdir(parents=True, exist_ok=True)

def escribir_yolo(
    ruta_txt,
    cajas,
    w,
    h
):
    filas = set()

    for c in cajas:
        x1, y1, x2, y2 = (
            c["box"]
            if isinstance(
                c,
                dict
            )
            else c
        )

        x1 = float(
            np.clip(
                x1,
                0,
                w
            )
        )

        x2 = float(
            np.clip(
                x2,
                0,
                w
            )
        )

        y1 = float(
            np.clip(
                y1,
                0,
                h
            )
        )

        y2 = float(
            np.clip(
                y2,
                0,
                h
            )
        )

        if (
            x2
            - x1
            < 2
            or y2
            - y1
            < 2
        ):
            continue

        cx = (
            x1
            + x2
        ) / 2 / w

        cy = (
            y1
            + y2
        ) / 2 / h

        bw = (
            x2
            - x1
        ) / w

        bh = (
            y2
            - y1
        ) / h

        fila = (
            0,
            round(
                float(
                    np.clip(
                        cx,
                        0,
                        1
                    )
                ),
                6
            ),
            round(
                float(
                    np.clip(
                        cy,
                        0,
                        1
                    )
                ),
                6
            ),
            round(
                float(
                    np.clip(
                        bw,
                        1e-6,
                        1
                    )
                ),
                6
            ),
            round(
                float(
                    np.clip(
                        bh,
                        1e-6,
                        1
                    )
                ),
                6
            )
        )

        filas.add(
            fila
        )

    with open(
        ruta_txt,
        "w"
    ) as f:
        for fila in sorted(
            filas
        ):
            f.write(
                f"{fila[0]} "
                f"{fila[1]:.6f} "
                f"{fila[2]:.6f} "
                f"{fila[3]:.6f} "
                f"{fila[4]:.6f}\n"
            )



def sanitizar_txt_yolo(
    ruta_txt
):
    ruta_txt = Path(
        ruta_txt
    )

    if not ruta_txt.exists():
        return 0

    filas = set()

    for line in open(
        ruta_txt
    ):
        p = line.split()

        if len(
            p
        ) < 5:
            continue

        cls = int(
            float(
                p[
                    0
                ]
            )
        )

        cx, cy, bw, bh = map(
            float,
            p[
                1:5
            ]
        )

        x1 = np.clip(
            cx
            - bw
            / 2,
            0,
            1
        )

        y1 = np.clip(
            cy
            - bh
            / 2,
            0,
            1
        )

        x2 = np.clip(
            cx
            + bw
            / 2,
            0,
            1
        )

        y2 = np.clip(
            cy
            + bh
            / 2,
            0,
            1
        )

        if (
            x2
            - x1
            <= 1e-4
            or y2
            - y1
            <= 1e-4
        ):
            continue

        fila = (
            cls,
            round(
                float(
                    (
                        x1
                        + x2
                    ) / 2
                ),
                6
            ),
            round(
                float(
                    (
                        y1
                        + y2
                    ) / 2
                ),
                6
            ),
            round(
                float(
                    x2
                    - x1
                ),
                6
            ),
            round(
                float(
                    y2
                    - y1
                ),
                6
            )
        )

        filas.add(
            fila
        )

    with open(
        ruta_txt,
        "w"
    ) as f:
        for fila in sorted(
            filas
        ):
            f.write(
                f"{fila[0]} "
                f"{fila[1]:.6f} "
                f"{fila[2]:.6f} "
                f"{fila[3]:.6f} "
                f"{fila[4]:.6f}\n"
            )

    return len(
        filas
    )

def leer_yolo(ruta_txt, w, h):
    cajas = []
    if Path(ruta_txt).exists():
        for line in open(ruta_txt):
            p = line.split()
            if len(p) >= 5:
                cx, cy, bw, bh = map(float, p[1:5])
                cajas.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
    return cajas

for g in gondolas_rect:
    pre = preprocesar(cv2.imread(str(g["path"])))
    h, w = pre.shape[:2]
    cv2.imwrite(str(PRODUCTO_DS / "images" / g["split"] / f"{g['id']}.jpg"), pre)
    escribir_yolo(PRODUCTO_DS / "labels" / g["split"] / f"{g['id']}.txt", etiquetas_producto[g["id"]], w, h)

YAML_PRODUCTO = PRODUCTO_DS / "data.yaml"
yaml.safe_dump({"path": str(PRODUCTO_DS), "train": "images/train", "val": "images/val",
                "names": {0: "producto"}}, open(YAML_PRODUCTO, "w"), sort_keys=False)

# ---- Ground truth manual (opcional) ----
PARA_ETIQUETAR = (
    PERSIST_DIR
    / "para_etiquetar_producto"
)

limpiar_dir(
    PARA_ETIQUETAR
    / "images"
)

(
    PARA_ETIQUETAR
    / "labels"
).mkdir(
    exist_ok=True
)

for g in gondolas_rect:
    if g["split"] == "val":
        shutil.copy2(
            PRODUCTO_DS
            / "images"
            / "val"
            / f"{g['id']}.jpg",
            PARA_ETIQUETAR
            / "images"
        )

        shutil.copy2(
            PRODUCTO_DS
            / "labels"
            / "val"
            / f"{g['id']}.txt",
            PARA_ETIQUETAR
            / "labels"
        )

if (
    DRIVE_CAN_WRITE
    and SUBIR_PARA_ETIQUETAR_AUTOMATICO
):
    print(
        "Subiendo material opcional "
        "para etiquetado manual..."
    )

    subir_carpeta_local_drive(
        PARA_ETIQUETAR,
        SHARED_DRIVE_FOLDER_ID
    )
else:
    print(
        "Carpeta para etiquetado manual "
        "preparada localmente. "
        "No se sube durante el entrenamiento."
    )

GT_ZIP = (
    PERSIST_DIR
    / "producto_gt_manual.zip"
)

GT_ZIP_DRIVE = buscar_hijo_drive(
    SHARED_DRIVE_FOLDER_ID,
    "producto_gt_manual.zip"
)

if (
    GT_ZIP_DRIVE is not None
    and not GT_ZIP.exists()
):
    descargar_archivo_drive(
        GT_ZIP_DRIVE[
            "id"
        ],
        GT_ZIP
    )

GT_DIR = (
    DATA
    / "producto_gt_manual"
)

GT_MANUAL = GT_ZIP.exists()

if GT_MANUAL:
    limpiar_dir(
        GT_DIR
    )

    safe_extract(
        GT_ZIP,
        GT_DIR
    )

    print(
        "Usando GROUND TRUTH MANUAL "
        "descargado por Drive API."
    )
else:
    print(
        "Sin GT manual: val se evalúa "
        "contra las etiquetas GDINO+SAM "
        "(pseudo-etiquetas)."
    )


def gt_producto(
    gid
):
    if GT_MANUAL:
        txt = next(
            GT_DIR.rglob(
                f"{gid}.txt"
            ),
            None
        )

        if txt is not None:
            h, w = cv2.imread(
                str(
                    GR[
                        gid
                    ][
                        "path"
                    ]
                )
            ).shape[
                :2
            ]

            return leer_yolo(
                txt,
                w,
                h
            )

    return [
        c[
            "box"
        ]
        for c
        in etiquetas_producto[
            gid
        ]
    ]


n_tr = sum(
    g[
        "split"
    ]
    == "train"
    for g
    in gondolas_rect
)

print(
    f"Dataset producto: "
    f"{n_tr} train / "
    f"{len(gondolas_rect)-n_tr} "
    "val góndolas"
)

print(
    "Imágenes de val preparadas para "
    "etiquetado manual en:",
    PARA_ETIQUETAR
)

# --- Sumar fotos HITL (etiquetas humanas) al train del alumno ---
n_hitl = 0
for p in (HITL_FULL / "images" / "train").glob("*.jpg"):
    ok = cv2.imwrite(str(PRODUCTO_DS / "images" / "train" / p.name), preprocesar(cv2.imread(str(p))))
    assert ok, f"No se pudo escribir {p.name}"
    destino_label = (
        PRODUCTO_DS
        / "labels"
        / "train"
        / f"{p.stem}.txt"
    )

    shutil.copy2(
        HITL_FULL
        / "labels"
        / "train"
        / f"{p.stem}.txt",
        destino_label
    )

    sanitizar_txt_yolo(
        destino_label
    )

    n_hitl += 1
print(f"+ {n_hitl} fotos HITL con etiquetas humanas en train")

for split in (
    "train",
    "val"
):
    for txt in (
        PRODUCTO_DS
        / "labels"
        / split
    ).glob(
        "*.txt"
    ):
        sanitizar_txt_yolo(
            txt
        )

print(
    "Etiquetas YOLO sanitizadas: "
    "coordenadas recortadas y duplicados eliminados."
)


In [ ]:
# [Celda 28]  Data augmentation offline (solo TRAIN): contraste, sombras, ruido, compresión, desenfoque
%pip install -q -U albumentations
import albumentations as A

N_AUG = 2   # copias aumentadas por imagen de train

def _t(fn, **kw):
    # Crea la transformación; si la versión de albumentations no acepta los parámetros, usa los por defecto
    try:
        return fn(**kw)
    except TypeError:
        return fn(p=kw.get("p", 0.5))

AUG = A.Compose([
    _t(A.CLAHE, clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),                   # contraste local -> bordes más marcados
    _t(A.RandomBrightnessContrast, brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    _t(A.RandomGamma, gamma_limit=(70, 140), p=0.3),                                # sobre/sub-exposición
    _t(A.RandomShadow, p=0.3),                                                      # sombras de repisas/personas
    A.OneOf([_t(A.GaussNoise, p=1.0), _t(A.ISONoise, p=1.0)], p=0.3),               # ruido de sensor (poca luz)
    _t(A.ImageCompression, p=0.3),                                                  # compresión tipo WhatsApp
    A.OneOf([_t(A.MotionBlur, blur_limit=7, p=1.0), _t(A.GaussianBlur, blur_limit=(3, 5), p=1.0)], p=0.2),
], bbox_params=A.BboxParams(format="yolo", label_fields=["clases"], min_visibility=0.3))

img_dir, lbl_dir = PRODUCTO_DS / "images" / "train", PRODUCTO_DS / "labels" / "train"
originales = [p for p in sorted(img_dir.glob("*.jpg")) if "_aug" not in p.stem]
n_creadas = 0
for p in originales:
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    boxes, clases = [], []
    for line in open(lbl_dir / f"{p.stem}.txt"):
        c, cx, cy, w, h = line.split()[:5]
        cx, cy, w, h = map(float, (cx, cy, w, h))
        x1, y1 = max(0.0, cx - w / 2), max(0.0, cy - h / 2)          # recorta a [0,1] (requisito de albumentations)
        x2, y2 = min(1.0, cx + w / 2), min(1.0, cy + h / 2)
        if x2 - x1 > 1e-3 and y2 - y1 > 1e-3:
            boxes.append([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]); clases.append(int(c))
    for k in range(N_AUG):
        out = AUG(image=img, bboxes=boxes, clases=clases)
        nombre = f"{p.stem}_aug{k}"
        cv2.imwrite(str(img_dir / f"{nombre}.jpg"), cv2.cvtColor(out["image"], cv2.COLOR_RGB2BGR))
        ruta_aug_txt = (
            lbl_dir
            / f"{nombre}.txt"
        )

        with open(
            ruta_aug_txt,
            "w"
        ) as f:
            for (
                cx,
                cy,
                w,
                h
            ), c in zip(
                out["bboxes"],
                out["clases"]
            ):
                x1 = float(
                    np.clip(
                        cx
                        - w
                        / 2,
                        0,
                        1
                    )
                )

                y1 = float(
                    np.clip(
                        cy
                        - h
                        / 2,
                        0,
                        1
                    )
                )

                x2 = float(
                    np.clip(
                        cx
                        + w
                        / 2,
                        0,
                        1
                    )
                )

                y2 = float(
                    np.clip(
                        cy
                        + h
                        / 2,
                        0,
                        1
                    )
                )

                if (
                    x2
                    - x1
                    <= 1e-4
                    or y2
                    - y1
                    <= 1e-4
                ):
                    continue

                f.write(
                    f"{c} "
                    f"{(x1+x2)/2:.6f} "
                    f"{(y1+y2)/2:.6f} "
                    f"{x2-x1:.6f} "
                    f"{y2-y1:.6f}\n"
                )

        sanitizar_txt_yolo(
            ruta_aug_txt
        )

        n_creadas += 1
print(f"Train: {len(originales)} originales + {n_creadas} aumentadas (val no se toca)")

if MOSTRAR_PREVIEWS:
    p = random.Random(SEED).choice(originales)

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(12, 4)
    )

    rutas_muestra = [
        p,
        img_dir / f"{p.stem}_aug0.jpg"
    ]

    for a, ruta in zip(ax, rutas_muestra):
        a.imshow(
            cv2.cvtColor(
                cv2.imread(str(ruta)),
                cv2.COLOR_BGR2RGB
            )
        )
        a.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)
    limpiar_memoria_agresivo()
else:
    print(
        "Vista de augmentation omitida "
        "en modo ULTRA."
    )

errores_labels = []

for split in (
    "train",
    "val"
):
    for txt in (
        PRODUCTO_DS
        / "labels"
        / split
    ).glob(
        "*.txt"
    ):
        sanitizar_txt_yolo(
            txt
        )

        for line in open(
            txt
        ):
            p = line.split()

            if len(
                p
            ) < 5:
                errores_labels.append(
                    (
                        txt.name,
                        "formato"
                    )
                )

                continue

            valores = list(
                map(
                    float,
                    p[
                        1:5
                    ]
                )
            )

            cx, cy, bw, bh = valores

            if not (
                0
                <= cx
                <= 1
                and 0
                <= cy
                <= 1
                and 0
                < bw
                <= 1
                and 0
                < bh
                <= 1
            ):
                errores_labels.append(
                    (
                        txt.name,
                        valores
                    )
                )

print(
    "Errores de etiquetas después "
    "del augmentation:",
    len(
        errores_labels
    )
)

assert len(
    errores_labels
) == 0


### Dataset de producto por mosaicos para limitar VRAM

Las imágenes de producto se cortan en mosaicos de 640 px con 20% de solapamiento. Se escriben en disco uno por uno y se entrena con `batch=1`, conservando más detalle de los productos pequeños que al reducir toda la góndola.


In [ ]:
PRODUCTO_TILE_DS = DATA / "ds_producto_tiles_ultra"

limpiar_dir(PRODUCTO_TILE_DS)

for split in ("train", "val"):
    (PRODUCTO_TILE_DS / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )
    (PRODUCTO_TILE_DS / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )


def posiciones_tiles_ultra(largo, tile, overlap):
    if largo <= tile:
        return [0]

    paso = max(
        1,
        int(round(tile * (1.0 - overlap)))
    )

    posiciones = list(
        range(
            0,
            max(1, largo - tile + 1),
            paso
        )
    )

    ultimo = max(0, largo - tile)

    if not posiciones or posiciones[-1] != ultimo:
        posiciones.append(ultimo)

    return sorted(set(posiciones))


def cajas_yolo_px(ruta_txt, W, H):
    cajas = []

    if not Path(ruta_txt).exists():
        return cajas

    with open(ruta_txt) as f:
        for linea in f:
            p = linea.split()

            if len(p) < 5:
                continue

            cls = int(float(p[0]))
            cx, cy, bw, bh = map(float, p[1:5])

            cajas.append(
                (
                    cls,
                    (cx - bw / 2) * W,
                    (cy - bh / 2) * H,
                    (cx + bw / 2) * W,
                    (cy + bh / 2) * H
                )
            )

    return cajas


def crear_tiles_archivo(
    ruta_img,
    ruta_lbl,
    out_img,
    out_lbl,
    tile=640,
    overlap=0.20,
    min_vis=0.55,
    max_negativos=1
):
    img = cv2.imread(str(ruta_img))

    if img is None:
        return 0, 0

    H, W = img.shape[:2]
    cajas = cajas_yolo_px(ruta_lbl, W, H)

    xs = posiciones_tiles_ultra(W, tile, overlap)
    ys = posiciones_tiles_ultra(H, tile, overlap)

    n_pos = 0
    n_neg = 0
    indice = 0

    for y0 in ys:
        for x0 in xs:
            x1 = min(W, x0 + tile)
            y1 = min(H, y0 + tile)

            tw = x1 - x0
            th = y1 - y0
            lineas = []

            for cls, bx1, by1, bx2, by2 in cajas:
                ix1 = max(bx1, x0)
                iy1 = max(by1, y0)
                ix2 = min(bx2, x1)
                iy2 = min(by2, y1)

                if ix2 <= ix1 or iy2 <= iy1:
                    continue

                area_original = max(
                    1.0,
                    (bx2 - bx1) * (by2 - by1)
                )

                vis = (
                    (ix2 - ix1) * (iy2 - iy1)
                    / area_original
                )

                if vis < min_vis:
                    continue

                cx = (((ix1 + ix2) / 2) - x0) / tw
                cy = (((iy1 + iy2) / 2) - y0) / th
                bw = (ix2 - ix1) / tw
                bh = (iy2 - iy1) / th

                lineas.append(
                    (cls, cx, cy, bw, bh)
                )

            guardar = bool(lineas)

            if not guardar and n_neg < max_negativos:
                guardar = True
                n_neg += 1

            if not guardar:
                continue

            nombre = f"{Path(ruta_img).stem}_t{indice:03d}"
            crop = img[y0:y1, x0:x1]

            cv2.imwrite(
                str(out_img / f"{nombre}.jpg"),
                crop
            )

            with open(
                out_lbl / f"{nombre}.txt",
                "w"
            ) as f:
                for cls, cx, cy, bw, bh in lineas:
                    f.write(
                        f"{cls} "
                        f"{cx:.6f} "
                        f"{cy:.6f} "
                        f"{bw:.6f} "
                        f"{bh:.6f}\n"
                    )

            if lineas:
                n_pos += 1

            indice += 1

    del img
    limpiar_memoria_agresivo()

    return n_pos, n_neg


conteo_tiles = {
    "train": 0,
    "val": 0
}

for split in ("train", "val"):
    src_img = PRODUCTO_DS / "images" / split
    src_lbl = PRODUCTO_DS / "labels" / split
    out_img = PRODUCTO_TILE_DS / "images" / split
    out_lbl = PRODUCTO_TILE_DS / "labels" / split

    imagenes = sorted(src_img.glob("*.jpg"))

    for j, p in enumerate(imagenes, start=1):
        comprobar_recursos(
            f"tiles_{split}_{j}",
            estricto=True
        )

        pos, neg = crear_tiles_archivo(
            p,
            src_lbl / f"{p.stem}.txt",
            out_img,
            out_lbl,
            tile=PRODUCTO_TILE_PX,
            overlap=PRODUCTO_TILE_OVERLAP,
            min_vis=PRODUCTO_TILE_MIN_VIS,
            max_negativos=1
        )

        conteo_tiles[split] += pos + neg

        if j % 50 == 0 or j == len(imagenes):
            print(
                f"Tiles {split}: "
                f"{j}/{len(imagenes)}"
            )


YAML_PRODUCTO_ULTRA = PRODUCTO_TILE_DS / "data.yaml"

with open(YAML_PRODUCTO_ULTRA, "w") as f:
    yaml.safe_dump(
        {
            "path": str(PRODUCTO_TILE_DS),
            "train": "images/train",
            "val": "images/val",
            "names": {
                0: "producto"
            }
        },
        f,
        sort_keys=False
    )

print(
    "Dataset tiled ULTRA:",
    conteo_tiles
)

comprobar_recursos(
    "post_tiles_producto",
    estricto=True
)


In [ ]:
# [Celda 29]  Entrenamiento YOLO producto — ULTRA con fallback
descargar_gdino()
descargar_sam()
limpiar_memoria_agresivo()

comprobar_recursos(
    "antes_train_producto",
    estricto=True
)

CANDIDATOS_PRODUCTO = [
    (CFG["yolo_producto"], 640),
    (CFG["yolo_producto"], 512),
    ("yolo11n.pt", 640),
    ("yolo11n.pt", 512)
]

BEST_PRODUCTO = None
PRODUCTO_MODELO_USADO = None
PRODUCTO_IMGSZ_USADO = None
ultimo_error_producto = None
vram_train_peak = 0.0

for pesos, imgsz_try in CANDIDATOS_PRODUCTO:
    print(
        "Probando producto:",
        pesos,
        "| imgsz",
        imgsz_try
    )

    model_p = None

    try:
        limpiar_memoria_agresivo()

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        model_p = YOLO(str(pesos))

        model_p.train(
            data=str(YAML_PRODUCTO_ULTRA),
            epochs=CFG["ep_producto"],
            patience=30,
            imgsz=imgsz_try,
            batch=1,
            device=DEVICE,
            seed=SEED,
            project=str(RUNS_DIR),
            name=(
                "producto_ultra_"
                + Path(str(pesos)).stem
                + f"_{imgsz_try}"
            ),
            exist_ok=True,
            degrees=3,
            perspective=0.0003,
            scale=0.4,
            hsv_h=0.015,
            hsv_s=0.6,
            hsv_v=0.4,
            fliplr=0.5,
            mosaic=1.0,
            close_mosaic=15,
            **argumentos_entrenamiento_eco()
        )

        BEST_PRODUCTO = (
            Path(model_p.trainer.save_dir)
            / "weights"
            / "best.pt"
        )

        PRODUCTO_MODELO_USADO = str(pesos)
        PRODUCTO_IMGSZ_USADO = imgsz_try

        if torch.cuda.is_available():
            vram_train_peak = (
                torch.cuda.max_memory_allocated()
                / 1e9
            )

        del model_p
        model_p = None
        limpiar_memoria_agresivo()
        break

    except RuntimeError as ex:
        ultimo_error_producto = ex

        if model_p is not None:
            del model_p

        limpiar_memoria_agresivo()

        if not es_error_oom(ex):
            raise

        print(
            "OOM con",
            pesos,
            imgsz_try,
            "-> se prueba una opción menor."
        )


assert BEST_PRODUCTO is not None, (
    "Ningún candidato de producto cabía "
    "en el límite de VRAM. Último error: "
    f"{ultimo_error_producto}"
)

CFG["imgsz_producto"] = PRODUCTO_IMGSZ_USADO

print(
    "Producto seleccionado:",
    PRODUCTO_MODELO_USADO,
    "| imgsz",
    PRODUCTO_IMGSZ_USADO
)

best_producto = YOLO(str(BEST_PRODUCTO))

m = best_producto.val(
    data=str(YAML_PRODUCTO_ULTRA),
    imgsz=PRODUCTO_IMGSZ_USADO,
    batch=1,
    workers=0,
    device=DEVICE,
    max_det=1000,
    verbose=False,
    plots=False
)

METRICAS["producto_ultralytics_tiles"] = {
    "P": float(m.box.mp),
    "R": float(m.box.mr),
    "mAP50": float(m.box.map50),
    "modelo": PRODUCTO_MODELO_USADO,
    "imgsz": PRODUCTO_IMGSZ_USADO,
    "evaluacion": "tiles"
}

print(
    f"\nProducto tiles — "
    f"P={m.box.mp:.1%} "
    f"R={m.box.mr:.1%} "
    f"mAP50={m.box.map50:.3f}"
)

if torch.cuda.is_available():
    print(
        "VRAM pico PyTorch entrenamiento:",
        f"{vram_train_peak:.2f} GB"
    )

HITL_TEST_DIR = limpiar_dir(
    DATA / "hitl_test"
)

for sub in ("images/val", "labels/val"):
    (HITL_TEST_DIR / sub).mkdir(
        parents=True,
        exist_ok=True
    )

for p in (
    HITL_FULL
    / "images"
    / "val"
).glob("*.jpg"):
    img_hitl = cv2.imread(str(p))

    cv2.imwrite(
        str(
            HITL_TEST_DIR
            / "images"
            / "val"
            / p.name
        ),
        preprocesar(img_hitl)
    )

    del img_hitl

    shutil.copy2(
        HITL_FULL
        / "labels"
        / "val"
        / f"{p.stem}.txt",
        HITL_TEST_DIR
        / "labels"
        / "val"
        / f"{p.stem}.txt"
    )

    limpiar_memoria_agresivo()

YAML_HITL_TEST = HITL_TEST_DIR / "test.yaml"

with open(YAML_HITL_TEST, "w") as f:
    yaml.safe_dump(
        {
            "path": str(HITL_TEST_DIR),
            "train": "images/val",
            "val": "images/val",
            "names": {
                0: "producto"
            }
        },
        f,
        sort_keys=False
    )

mh = best_producto.val(
    data=str(YAML_HITL_TEST),
    imgsz=PRODUCTO_IMGSZ_USADO,
    batch=1,
    workers=0,
    device=DEVICE,
    max_det=1000,
    verbose=False,
    plots=False
)

METRICAS["producto_test_humano_hitl"] = {
    "P": float(mh.box.mp),
    "R": float(mh.box.mr),
    "mAP50": float(mh.box.map50),
    "modelo": PRODUCTO_MODELO_USADO,
    "imgsz": PRODUCTO_IMGSZ_USADO
}

print(
    "Producto en test HUMANO HITL — "
    f"P={mh.box.mp:.1%} "
    f"R={mh.box.mr:.1%} "
    f"mAP50={mh.box.map50:.3f}"
)

limpiar_memoria_agresivo()
comprobar_recursos(
    "post_train_producto",
    estricto=True
)

registrar_recursos(
    "post_train_producto"
)


### 8. Umbral de confianza fijo: nunca menor a 0.65

En la ejecución anterior el punto que alcanzó el objetivo de Precision fue:

```text
conf = 0.65
Precision = 80.6%
Recall = 48.1%
```

Esta versión establece `0.65` como piso absoluto. Ninguna etapa del pipeline puede ejecutar el detector de
productos con una confianza inferior.

Para recuperar productos sin bajar el umbral se aplica inferencia multivista:

- imagen completa preprocesada;
- imagen completa con contraste local reforzado;
- dos mosaicos horizontales solapados cuando la góndola es suficientemente ancha.

Todas las detecciones de todas las vistas deben superar exactamente el mismo umbral mínimo. Después se
fusionan mediante NMS para eliminar duplicados.

El barrido de Precision/Recall también comienza en 0.65; por lo tanto, el notebook nunca puede seleccionar
automáticamente un umbral inferior.


In [ ]:
CONF_PRODUCTO_MIN = 0.65

CFG.update(
    dict(
        conf_producto_min=CONF_PRODUCTO_MIN,
        tta_iou=0.55,
        tta_tile_frac=0.62,
        tta_tile_overlap=0.18
    )
)


def emparejar(
    pred,
    gt,
    thr=0.5
):
    orden = sorted(
        range(
            len(pred)
        ),
        key=lambda i: -pred[i][1]
    )

    usados = set()
    tp = []
    fp = []

    for i in orden:
        mejor = 0.0
        j_mejor = -1

        for j, g in enumerate(
            gt
        ):
            if j in usados:
                continue

            valor = iou(
                pred[i][0],
                g
            )

            if valor > mejor:
                mejor = valor
                j_mejor = j

        if mejor >= thr:
            usados.add(
                j_mejor
            )

            tp.append(
                (
                    i,
                    j_mejor
                )
            )
        else:
            fp.append(
                i
            )

    fn = [
        j
        for j
        in range(
            len(gt)
        )
        if j
        not in usados
    ]

    return (
        tp,
        fp,
        fn
    )


def predecir_vista(
    img,
    conf
):
    assert (
        conf
        >= CONF_PRODUCTO_MIN
    ), (
        "No se permite inferencia con "
        f"conf < {CONF_PRODUCTO_MIN:.2f}"
    )

    r = best_producto.predict(
        img,
        imgsz=CFG[
            "imgsz_producto"
        ],
        conf=conf,
        max_det=1000,
        device=DEVICE,
        verbose=False
    )[0]

    if (
        r.boxes is None
        or len(
            r.boxes
        ) == 0
    ):
        return []

    return [
        (
            [
                float(v)
                for v
                in b
            ],
            float(s)
        )
        for b, s
        in zip(
            r.boxes.xyxy
            .cpu()
            .numpy(),
            r.boxes.conf
            .cpu()
            .numpy()
        )
    ]


def mejorar_contraste_inferencia(
    img
):
    lab = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2LAB
    )

    l, a, b = cv2.split(
        lab
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.2,
        tileGridSize=(
            8,
            8
        )
    )

    l = clahe.apply(
        l
    )

    out = cv2.cvtColor(
        cv2.merge(
            (
                l,
                a,
                b
            )
        ),
        cv2.COLOR_LAB2BGR
    )

    suave = cv2.GaussianBlur(
        out,
        (
            0,
            0
        ),
        0.8
    )

    return cv2.addWeighted(
        out,
        1.18,
        suave,
        -0.18,
        0
    )


def nms_predicciones(
    predicciones,
    iou_thr=None
):
    if not predicciones:
        return []

    iou_thr = (
        CFG[
            "tta_iou"
        ]
        if iou_thr is None
        else float(
            iou_thr
        )
    )

    orden = sorted(
        range(
            len(
                predicciones
            )
        ),
        key=lambda i: -predicciones[
            i
        ][
            1
        ]
    )

    conservar = []

    for idx in orden:
        caja, score = predicciones[
            idx
        ]

        if all(
            iou(
                caja,
                predicciones[
                    j
                ][
                    0
                ]
            )
            < iou_thr
            for j
            in conservar
        ):
            conservar.append(
                idx
            )

    return [
        predicciones[
            i
        ]
        for i
        in conservar
    ]


def predicciones_mosaico(
    img,
    conf
):
    H, W = img.shape[:2]
    tile_px = int(PRODUCTO_TILE_PX)

    xs = posiciones_tiles_ultra(
        W,
        tile_px,
        PRODUCTO_TILE_OVERLAP
    )

    ys = posiciones_tiles_ultra(
        H,
        tile_px,
        PRODUCTO_TILE_OVERLAP
    )

    salida = []

    for y0 in ys:
        for x0 in xs:
            x1 = min(W, x0 + tile_px)
            y1 = min(H, y0 + tile_px)

            tile = img[y0:y1, x0:x1]

            pred = predecir_vista(
                tile,
                conf
            )

            if not pred:
                tile_c = mejorar_contraste_inferencia(
                    tile
                )

                pred = predecir_vista(
                    tile_c,
                    conf
                )

                del tile_c

            for caja, score in pred:
                salida.append(
                    (
                        [
                            caja[0] + x0,
                            caja[1] + y0,
                            caja[2] + x0,
                            caja[3] + y0
                        ],
                        score
                    )
                )

            del tile
            limpiar_memoria_agresivo()

    return salida


def predecir_productos(
    img_pre,
    conf
):
    conf = max(
        float(conf),
        CONF_PRODUCTO_MIN
    )

    predicciones = []

    predicciones.extend(
        predecir_vista(
            img_pre,
            conf
        )
    )

    predicciones.extend(
        predicciones_mosaico(
            img_pre,
            conf
        )
    )

    salida = nms_predicciones(
        predicciones
    )

    limpiar_memoria_agresivo()

    return [
        p
        for p in salida
        if p[1] >= conf
    ]


val_ids = [
    g[
        "id"
    ]
    for g
    in gondolas_rect
    if g[
        "split"
    ]
    == "val"
]

PRED_VAL = {}

for i, gid in enumerate(
    val_ids,
    start=1
):
    img = cv2.imread(
        str(
            GR[
                gid
            ][
                "path"
            ]
        )
    )

    PRED_VAL[
        gid
    ] = predecir_productos(
        preprocesar(
            img
        ),
        CONF_PRODUCTO_MIN
    )

    if (
        i % 10 == 0
        or i
        == len(
            val_ids
        )
    ):
        print(
            f"Validación multivista: "
            f"{i}/{len(val_ids)}"
        )


GT_VAL = {
    gid: gt_producto(
        gid
    )
    for gid
    in val_ids
}


filas = []

for c in np.round(
    np.arange(
        CONF_PRODUCTO_MIN,
        0.96,
        0.05
    ),
    2
):
    TP = 0
    FP = 0
    FN = 0

    for gid in val_ids:
        pred = [
            p
            for p
            in PRED_VAL[
                gid
            ]
            if p[
                1
            ]
            >= c
        ]

        tp, fp, fn = emparejar(
            pred,
            GT_VAL[
                gid
            ]
        )

        TP += len(
            tp
        )

        FP += len(
            fp
        )

        FN += len(
            fn
        )

    P = TP / max(
        1,
        TP
        + FP
    )

    R = TP / max(
        1,
        TP
        + FN
    )

    F1 = (
        2
        * P
        * R
        / max(
            1e-9,
            P
            + R
        )
    )

    filas.append(
        {
            "conf": c,
            "P": P,
            "R": R,
            "F1": F1,
            "TP": TP,
            "FP": FP,
            "FN": FN
        }
    )


df_conf = pd.DataFrame(
    filas
)

ok = df_conf[
    df_conf[
        "P"
    ]
    >= CFG[
        "precision_objetivo"
    ]
]

if len(
    ok
):
    fila = ok.iloc[
        0
    ]

    print(
        f"Objetivo alcanzado sin bajar "
        f"de {CONF_PRODUCTO_MIN:.2f}: "
        f"conf={fila.conf:.2f} -> "
        f"P={fila.P:.1%} "
        f"R={fila.R:.1%} "
        f"F1={fila.F1:.3f}"
    )
else:
    fila = df_conf.loc[
        df_conf[
            "F1"
        ].idxmax()
    ]

    print(
        f"No se llegó a "
        f"P>={CFG['precision_objetivo']:.0%}. "
        f"Se conserva conf >= "
        f"{CONF_PRODUCTO_MIN:.2f}: "
        f"conf={fila.conf:.2f} -> "
        f"P={fila.P:.1%} "
        f"R={fila.R:.1%}"
    )


CONF_PRODUCTO = max(
    CONF_PRODUCTO_MIN,
    float(
        fila.conf
    )
)

assert (
    CONF_PRODUCTO
    >= CONF_PRODUCTO_MIN
)


METRICAS[
    "producto_umbral"
] = {
    "conf": CONF_PRODUCTO,
    "conf_minimo": CONF_PRODUCTO_MIN,
    "P": float(
        fila.P
    ),
    "R": float(
        fila.R
    ),
    "F1": float(
        fila.F1
    ),
    "metodo_inferencia": (
        "multivista_mismo_umbral"
    ),
    "gt": (
        "manual"
        if GT_MANUAL
        else "pseudo (GDINO+SAM)"
    )
}


fig, ax = plt.subplots(
    1,
    2,
    figsize=(
        14,
        4.5
    )
)

ax[
    0
].plot(
    df_conf.conf,
    df_conf.P,
    "-o",
    label="Precisión"
)

ax[
    0
].plot(
    df_conf.conf,
    df_conf.R,
    "-o",
    label="Recall"
)

ax[
    0
].plot(
    df_conf.conf,
    df_conf.F1,
    "-o",
    label="F1"
)

ax[
    0
].axhline(
    CFG[
        "precision_objetivo"
    ],
    ls="--"
)

ax[
    0
].axvline(
    CONF_PRODUCTO_MIN,
    ls=":",
    label="Piso 0.65"
)

ax[
    0
].axvline(
    CONF_PRODUCTO,
    ls="-.",
    label="Elegido"
)

ax[
    0
].set_xlabel(
    "conf"
)

ax[
    0
].legend()

ax[
    0
].set_title(
    "Métricas con conf >= 0.65"
)

ax[
    1
].plot(
    df_conf.R,
    df_conf.P,
    "-o"
)

ax[
    1
].set_xlabel(
    "Recall"
)

ax[
    1
].set_ylabel(
    "Precisión"
)

ax[
    1
].set_title(
    "Curva P-R"
)

plt.tight_layout()
plt.show()

print(
    df_conf
    .round(
        3
    )
    .to_string(
        index=False
    )
)


### 9. Vacíos mejorados: geometría + estructura visual + revalidación YOLO

La clase `1 = vacío` sigue sin entrenarse directamente porque no existe ground truth humano de huecos. En
esta versión se fortalece el detector híbrido manteniendo `CONF_PRODUCTO >= 0.65`.

La decisión de vacío combina ahora:

- agrupación robusta de productos por repisa;
- líneas horizontales mediante Hough y morfología;
- límites horizontales calculados por repisa, no solo un límite global;
- densidad de bordes verticales, que ayuda a diferenciar productos de rieles horizontales;
- textura, saturación, Canny y entropía;
- soporte físico de línea de repisa bajo o sobre el hueco;
- clasificación del hueco como interno, lateral o repisa completa;
- estimación del número de posiciones de producto faltantes;
- revalidación con YOLO sobre el recorte del hueco usando exactamente el mismo umbral `>= 0.65`.

La revalidación por recorte es especialmente útil: un producto pequeño que no se detectó en la góndola
completa puede hacerse mucho más grande al recortar el supuesto hueco. Si YOLO encuentra producto en ese
recorte con `conf >= 0.65`, la región se rechaza como vacío.

Además, el algoritmo diferencia `vacío confirmado`, `ocupación por productos` e `incertidumbre`, evitando
asumir que todo hueco rechazado automáticamente está ocupado.


In [ ]:
CFG.update(
    dict(
        gap_min_vs_ancho=0.72,
        gap_lateral_min_vs_ancho=0.95,
        margen_ocupacion=0.08,
        shelf_line_min_frac=0.26,
        shelf_line_max_slope=0.10,
        shelf_pair_min_vs_h=0.62,
        shelf_pair_max_vs_h=2.45,
        vacio_inner_y=0.16,
        vacio_score_final_min=0.56,
        vacio_score_total_min=0.61,
        vacio_max_actividad_objeto=0.58,
        vacio_recheck_margen_score=0.12,
        vacio_recheck_laterales=True,
        vacio_recheck_total=True,
        vacio_recheck_internos_dudosos=True,
        vacio_soporte_lateral_min=0.30,
        vacio_soporte_total_min=0.22
    )
)


def agrupar_filas(cajas):
    cajas = [
        [float(v) for v in b]
        for b in cajas
        if len(b) == 4
        and b[2] > b[0]
        and b[3] > b[1]
    ]

    if not cajas:
        return []

    alturas = [
        b[3] - b[1]
        for b in cajas
    ]

    med_h = max(
        8.0,
        float(
            np.median(
                alturas
            )
        )
    )

    orden = sorted(
        cajas,
        key=lambda b: (
            b[1] + b[3]
        ) / 2
    )

    filas = []

    for caja in orden:
        cy = (
            caja[1] + caja[3]
        ) / 2

        mejor = None
        mejor_dist = None

        for idx, fila in enumerate(
            filas
        ):
            centros = [
                (
                    b[1] + b[3]
                ) / 2
                for b in fila
            ]

            hs = [
                b[3] - b[1]
                for b in fila
            ]

            cy_ref = float(
                np.median(
                    centros
                )
            )

            h_ref = max(
                med_h,
                float(
                    np.median(
                        hs
                    )
                )
            )

            distancia = abs(
                cy - cy_ref
            )

            if (
                distancia
                <= 0.62 * h_ref
                and (
                    mejor_dist is None
                    or distancia < mejor_dist
                )
            ):
                mejor = idx
                mejor_dist = distancia

        if mejor is None:
            filas.append(
                [caja]
            )
        else:
            filas[mejor].append(
                caja
            )

    salida = []

    for fila in filas:
        salida.append(
            {
                "y1": float(
                    np.percentile(
                        [
                            b[1]
                            for b in fila
                        ],
                        8
                    )
                ),
                "y2": float(
                    np.percentile(
                        [
                            b[3]
                            for b in fila
                        ],
                        92
                    )
                ),
                "cajas": sorted(
                    fila,
                    key=lambda b: b[0]
                ),
                "origen": "productos"
            }
        )

    return sorted(
        salida,
        key=lambda f: (
            f["y1"] + f["y2"]
        ) / 2
    )


def detectar_lineas_repisas(
    img_bgr,
    med_h
):
    H, W = img_bgr.shape[:2]

    gris = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2GRAY
    )

    gris = cv2.GaussianBlur(
        gris,
        (5, 5),
        0
    )

    bordes = cv2.Canny(
        gris,
        45,
        135
    )

    min_len = max(
        50,
        int(
            W
            * CFG[
                "shelf_line_min_frac"
            ]
        )
    )

    lineas_hough = cv2.HoughLinesP(
        bordes,
        1,
        np.pi / 180,
        threshold=max(
            28,
            int(
                W
                * 0.032
            )
        ),
        minLineLength=min_len,
        maxLineGap=max(
            18,
            int(
                W
                * 0.07
            )
        )
    )

    segmentos = []

    if lineas_hough is not None:
        for linea in lineas_hough[
            :,
            0
        ]:
            x1, y1, x2, y2 = [
                int(v)
                for v in linea
            ]

            dx = x2 - x1
            dy = y2 - y1

            if abs(
                dx
            ) < 10:
                continue

            pendiente = abs(
                dy / dx
            )

            if (
                pendiente
                > CFG[
                    "shelf_line_max_slope"
                ]
            ):
                continue

            xa = min(
                x1,
                x2
            )

            xb = max(
                x1,
                x2
            )

            segmentos.append(
                {
                    "x1": float(xa),
                    "x2": float(xb),
                    "y": float(
                        (
                            y1 + y2
                        ) / 2
                    ),
                    "longitud": float(
                        xb - xa
                    ),
                    "fuente": "hough"
                }
            )

    kernel_w = max(
        25,
        int(
            W * 0.09
        )
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (
            kernel_w,
            1
        )
    )

    horizontales = cv2.morphologyEx(
        bordes,
        cv2.MORPH_OPEN,
        kernel
    )

    contornos, _ = cv2.findContours(
        horizontales,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contornos:
        x, y, w, h = cv2.boundingRect(
            cnt
        )

        if (
            w < min_len
            or h > max(
                8,
                int(
                    0.18 * med_h
                )
            )
        ):
            continue

        segmentos.append(
            {
                "x1": float(x),
                "x2": float(
                    x + w
                ),
                "y": float(
                    y + h / 2
                ),
                "longitud": float(w),
                "fuente": "morfologia"
            }
        )

    if not segmentos:
        return []

    tolerancia = max(
        5,
        int(
            0.20 * med_h
        )
    )

    segmentos = sorted(
        segmentos,
        key=lambda s: s["y"]
    )

    grupos = []

    for seg in segmentos:
        mejor = None
        mejor_d = None

        for i, grupo in enumerate(
            grupos
        ):
            pesos = [
                max(
                    1.0,
                    s[
                        "longitud"
                    ]
                )
                for s in grupo
            ]

            y_ref = float(
                np.average(
                    [
                        s["y"]
                        for s in grupo
                    ],
                    weights=pesos
                )
            )

            d = abs(
                seg["y"]
                - y_ref
            )

            if (
                d <= tolerancia
                and (
                    mejor_d is None
                    or d < mejor_d
                )
            ):
                mejor = i
                mejor_d = d

        if mejor is None:
            grupos.append(
                [seg]
            )
        else:
            grupos[
                mejor
            ].append(
                seg
            )

    fusionadas = []

    for grupo in grupos:
        pesos = np.array(
            [
                max(
                    1.0,
                    s[
                        "longitud"
                    ]
                )
                for s in grupo
            ],
            dtype=np.float32
        )

        ys = np.array(
            [
                s["y"]
                for s in grupo
            ],
            dtype=np.float32
        )

        fusionadas.append(
            {
                "x1": float(
                    min(
                        s["x1"]
                        for s in grupo
                    )
                ),
                "x2": float(
                    max(
                        s["x2"]
                        for s in grupo
                    )
                ),
                "y": float(
                    np.average(
                        ys,
                        weights=pesos
                    )
                ),
                "soporte": float(
                    min(
                        1.0,
                        sum(
                            s[
                                "longitud"
                            ]
                            for s in grupo
                        ) / max(
                            1.0,
                            W
                        )
                    )
                ),
                "fuentes": sorted(
                    set(
                        s[
                            "fuente"
                        ]
                        for s in grupo
                    )
                )
            }
        )

    return sorted(
        fusionadas,
        key=lambda l: l["y"]
    )


def estimar_x_global(
    W,
    cajas,
    lineas
):
    if cajas:
        anchos = [
            b[2] - b[0]
            for b in cajas
        ]

        med_w = max(
            8.0,
            float(
                np.median(
                    anchos
                )
            )
        )

        x_prod_0 = max(
            0.0,
            min(
                b[0]
                for b in cajas
            )
            - 0.75
            * med_w
        )

        x_prod_1 = min(
            float(W),
            max(
                b[2]
                for b in cajas
            )
            + 0.75
            * med_w
        )
    else:
        med_w = max(
            12.0,
            0.075
            * W
        )

        x_prod_0 = 0.04 * W
        x_prod_1 = 0.96 * W

    utiles = [
        l
        for l in lineas
        if (
            l["x2"]
            - l["x1"]
        )
        >= 0.38 * W
    ]

    if utiles:
        x_line_0 = float(
            np.percentile(
                [
                    l["x1"]
                    for l in utiles
                ],
                18
            )
        )

        x_line_1 = float(
            np.percentile(
                [
                    l["x2"]
                    for l in utiles
                ],
                82
            )
        )

        if (
            x_line_1
            - x_line_0
        ) >= 0.46 * W:
            x0 = min(
                x_prod_0,
                x_line_0
            )

            x1 = max(
                x_prod_1,
                x_line_1
            )
        else:
            x0 = x_prod_0
            x1 = x_prod_1
    else:
        x0 = x_prod_0
        x1 = x_prod_1

    x0 = float(
        np.clip(
            x0,
            0.01 * W,
            0.93 * W
        )
    )

    x1 = float(
        np.clip(
            x1,
            0.07 * W,
            0.99 * W
        )
    )

    if x1 <= x0:
        x0 = 0.04 * W
        x1 = 0.96 * W

    return (
        x0,
        x1,
        med_w
    )


def construir_bandas_repisas(
    H,
    cajas,
    filas_producto,
    lineas,
    med_h
):
    bandas = []

    for fila in filas_producto:
        y1 = max(
            0.0,
            fila["y1"]
            - 0.10
            * med_h
        )

        y2 = min(
            float(H),
            fila["y2"]
            + 0.12
            * med_h
        )

        base = float(
            np.median(
                [
                    b[3]
                    for b in fila[
                        "cajas"
                    ]
                ]
            )
        )

        candidatas = [
            l
            for l in lineas
            if (
                base
                - 0.18
                * med_h
                <= l["y"]
                <= base
                + 0.70
                * med_h
            )
        ]

        if candidatas:
            mejor = min(
                candidatas,
                key=lambda l: abs(
                    l["y"]
                    - base
                )
            )

            if (
                mejor["y"]
                > y1 + 5
            ):
                y2 = min(
                    float(H),
                    max(
                        y2,
                        mejor["y"]
                    )
                )

        bandas.append(
            {
                "y1": float(y1),
                "y2": float(y2),
                "cajas": fila[
                    "cajas"
                ],
                "origen": "productos",
                "vacia_total_candidata": False
            }
        )

    if len(
        lineas
    ) >= 2:
        centros_producto = [
            (
                b[1] + b[3]
            ) / 2
            for b in cajas
        ]

        ys = sorted(
            [
                l["y"]
                for l in lineas
            ]
        )

        for ya, yb in zip(
            ys[:-1],
            ys[1:]
        ):
            alto = yb - ya

            if not (
                CFG[
                    "shelf_pair_min_vs_h"
                ]
                * med_h
                <= alto
                <= CFG[
                    "shelf_pair_max_vs_h"
                ]
                * med_h
            ):
                continue

            y1 = ya + 0.10 * alto
            y2 = yb - 0.10 * alto

            if (
                y2 - y1
                < 10
            ):
                continue

            tiene_producto = any(
                y1
                <= cy
                <= y2
                for cy
                in centros_producto
            )

            if tiene_producto:
                continue

            centro = (
                y1 + y2
            ) / 2

            solapa = any(
                abs(
                    centro
                    - (
                        b[
                            "y1"
                        ]
                        + b[
                            "y2"
                        ]
                    ) / 2
                )
                <= 0.42
                * max(
                    med_h,
                    b[
                        "y2"
                    ]
                    - b[
                        "y1"
                    ]
                )
                for b in bandas
            )

            if solapa:
                continue

            bandas.append(
                {
                    "y1": float(y1),
                    "y2": float(y2),
                    "cajas": [],
                    "origen": "lineas",
                    "vacia_total_candidata": True
                }
            )

    return sorted(
        bandas,
        key=lambda b: (
            b["y1"] + b["y2"]
        ) / 2
    )


def limites_x_repisa(
    W,
    banda,
    cajas_banda,
    lineas,
    x_global,
    ancho_ref,
    med_h
):
    gx0, gx1 = x_global

    cerca = [
        l
        for l in lineas
        if (
            abs(
                l["y"]
                - banda[
                    "y2"
                ]
            )
            <= 0.38
            * med_h
            or abs(
                l["y"]
                - banda[
                    "y1"
                ]
            )
            <= 0.30
            * med_h
        )
    ]

    cerca = [
        l
        for l in cerca
        if (
            l["x2"]
            - l["x1"]
        )
        >= 0.32
        * (
            gx1 - gx0
        )
    ]

    if cerca:
        mejor = max(
            cerca,
            key=lambda l: (
                l["x2"]
                - l["x1"]
            )
            * (
                0.5
                + l[
                    "soporte"
                ]
            )
        )

        x0 = max(
            gx0,
            mejor[
                "x1"
            ]
        )

        x1 = min(
            gx1,
            mejor[
                "x2"
            ]
        )

        if (
            x1 - x0
            >= 0.42
            * (
                gx1 - gx0
            )
        ):
            return (
                float(x0),
                float(x1),
                "linea"
            )

    if cajas_banda:
        x0 = max(
            gx0,
            min(
                b[0]
                for b in cajas_banda
            )
            - 1.15
            * ancho_ref
        )

        x1 = min(
            gx1,
            max(
                b[2]
                for b in cajas_banda
            )
            + 1.15
            * ancho_ref
        )

        return (
            float(x0),
            float(x1),
            "productos"
        )

    return (
        float(gx0),
        float(gx1),
        "global"
    )


def fusionar_intervalos(
    intervalos
):
    intervalos = sorted(
        [
            (
                float(a),
                float(b)
            )
            for a, b
            in intervalos
            if b > a
        ]
    )

    if not intervalos:
        return []

    union = [
        intervalos[
            0
        ]
    ]

    for a, b in intervalos[
        1:
    ]:
        ua, ub = union[
            -1
        ]

        if a <= ub:
            union[
                -1
            ] = (
                ua,
                max(
                    ub,
                    b
                )
            )
        else:
            union.append(
                (
                    a,
                    b
                )
            )

    return union


def huecos_desde_union(
    union,
    x0,
    x1
):
    libres = []
    x = x0

    for a, b in union:
        a = max(
            x0,
            a
        )

        b = min(
            x1,
            b
        )

        if a > x:
            libres.append(
                (
                    x,
                    a
                )
            )

        x = max(
            x,
            b
        )

    if x < x1:
        libres.append(
            (
                x,
                x1
            )
        )

    return libres


def promedio_region(
    mapa,
    y1,
    y2,
    x1,
    x2
):
    H, W = mapa.shape[
        :2
    ]

    xa = int(
        np.clip(
            round(
                x1
            ),
            0,
            W - 1
        )
    )

    xb = int(
        np.clip(
            round(
                x2
            ),
            xa + 1,
            W
        )
    )

    ya = int(
        np.clip(
            round(
                y1
            ),
            0,
            H - 1
        )
    )

    yb = int(
        np.clip(
            round(
                y2
            ),
            ya + 1,
            H
        )
    )

    region = mapa[
        ya:yb,
        xa:xb
    ]

    return (
        float(
            np.mean(
                region
            )
        )
        if region.size
        else 0.0
    )


def entropia_region(
    gris,
    y1,
    y2,
    x1,
    x2
):
    H, W = gris.shape[
        :2
    ]

    xa = int(
        np.clip(
            round(
                x1
            ),
            0,
            W - 1
        )
    )

    xb = int(
        np.clip(
            round(
                x2
            ),
            xa + 1,
            W
        )
    )

    ya = int(
        np.clip(
            round(
                y1
            ),
            0,
            H - 1
        )
    )

    yb = int(
        np.clip(
            round(
                y2
            ),
            ya + 1,
            H
        )
    )

    region = gris[
        ya:yb,
        xa:xb
    ]

    if region.size < 16:
        return 0.0

    hist = cv2.calcHist(
        [
            region.astype(
                np.uint8
            )
        ],
        [0],
        None,
        [16],
        [0, 256]
    ).ravel()

    p = hist / max(
        1.0,
        hist.sum()
    )

    p = p[
        p > 0
    ]

    if len(
        p
    ) == 0:
        return 0.0

    return float(
        -np.sum(
            p
            * np.log2(
                p
            )
        )
        / 4.0
    )


def soporte_linea_hueco(
    lineas,
    a,
    b,
    y1,
    y2,
    med_h
):
    ancho = max(
        1.0,
        b - a
    )

    soportes = []

    for linea in lineas:
        if not (
            abs(
                linea["y"]
                - y1
            )
            <= 0.30
            * med_h
            or abs(
                linea["y"]
                - y2
            )
            <= 0.38
            * med_h
        ):
            continue

        inter = max(
            0.0,
            min(
                b,
                linea["x2"]
            )
            - max(
                a,
                linea["x1"]
            )
        )

        if inter <= 0:
            continue

        soportes.append(
            min(
                1.0,
                inter / ancho
            )
        )

    return (
        max(
            soportes
        )
        if soportes
        else 0.0
    )


def features_region(
    mapas,
    bbox
):
    x1, y1, x2, y2 = bbox

    return {
        "vertical": promedio_region(
            mapas[
                "vertical"
            ],
            y1,
            y2,
            x1,
            x2
        ),
        "canny": promedio_region(
            mapas[
                "canny"
            ],
            y1,
            y2,
            x1,
            x2
        ),
        "textura": promedio_region(
            mapas[
                "textura"
            ],
            y1,
            y2,
            x1,
            x2
        ),
        "sat": promedio_region(
            mapas[
                "sat"
            ],
            y1,
            y2,
            x1,
            x2
        ),
        "entropia": entropia_region(
            mapas[
                "gris"
            ],
            y1,
            y2,
            x1,
            x2
        )
    }


def mediana_features(
    lista,
    fallback
):
    if not lista:
        return dict(
            fallback
        )

    salida = {}

    for clave in fallback:
        valores = [
            x[
                clave
            ]
            for x in lista
        ]

        salida[
            clave
        ] = max(
            1e-6,
            float(
                np.median(
                    valores
                )
            )
        )

    return salida


def score_vacio_desde_features(
    feat,
    ref,
    ancho,
    ancho_ref,
    soporte,
    tipo
):
    def score_bajo(
        valor,
        referencia
    ):
        return float(
            np.clip(
                1.0
                - valor
                / (
                    referencia
                    + 1e-6
                ),
                0.0,
                1.0
            )
        )

    s_vertical = score_bajo(
        feat[
            "vertical"
        ],
        ref[
            "vertical"
        ]
    )

    s_canny = score_bajo(
        feat[
            "canny"
        ],
        ref[
            "canny"
        ]
    )

    s_textura = score_bajo(
        feat[
            "textura"
        ],
        ref[
            "textura"
        ]
    )

    s_sat = score_bajo(
        feat[
            "sat"
        ],
        ref[
            "sat"
        ]
    )

    s_entropia = score_bajo(
        feat[
            "entropia"
        ],
        ref[
            "entropia"
        ]
    )

    score_visual = float(
        np.clip(
            0.30
            * s_vertical
            + 0.20
            * s_textura
            + 0.18
            * s_canny
            + 0.17
            * s_entropia
            + 0.15
            * s_sat,
            0.0,
            1.0
        )
    )

    actividad_objeto = float(
        np.clip(
            0.34
            * min(
                feat[
                    "vertical"
                ]
                / (
                    ref[
                        "vertical"
                    ]
                    + 1e-6
                ),
                1.5
            )
            + 0.26
            * min(
                feat[
                    "textura"
                ]
                / (
                    ref[
                        "textura"
                    ]
                    + 1e-6
                ),
                1.5
            )
            + 0.20
            * min(
                feat[
                    "canny"
                ]
                / (
                    ref[
                        "canny"
                    ]
                    + 1e-6
                ),
                1.5
            )
            + 0.20
            * min(
                feat[
                    "entropia"
                ]
                / (
                    ref[
                        "entropia"
                    ]
                    + 1e-6
                ),
                1.5
            ),
            0.0,
            1.0
        )
    )

    score_geometria = float(
        np.clip(
            ancho
            / max(
                ancho_ref,
                1.0
            )
            / 1.65,
            0.0,
            1.0
        )
    )

    if tipo.startswith(
        "lateral"
    ):
        pesos = (
            0.60,
            0.16,
            0.24
        )
    elif tipo == "total":
        pesos = (
            0.57,
            0.13,
            0.30
        )
    else:
        pesos = (
            0.70,
            0.20,
            0.10
        )

    score_final = float(
        np.clip(
            pesos[
                0
            ]
            * score_visual
            + pesos[
                1
            ]
            * score_geometria
            + pesos[
                2
            ]
            * soporte,
            0.0,
            1.0
        )
    )

    return {
        "score_visual": score_visual,
        "score_geometria": score_geometria,
        "score_soporte": float(
            soporte
        ),
        "score_final": score_final,
        "actividad_objeto": actividad_objeto
    }


def revisar_hueco_yolo(
    img_bgr,
    bbox,
    conf
):
    if (
        "best_producto"
        not in globals()
        or best_producto is None
    ):
        return {
            "producto_en_hueco": False,
            "n": 0,
            "max_conf": 0.0
        }

    conf = max(
        float(
            conf
        ),
        CONF_PRODUCTO_MIN
    )

    H, W = img_bgr.shape[
        :2
    ]

    x1, y1, x2, y2 = bbox

    ancho = x2 - x1
    alto = y2 - y1

    mx = 0.08 * ancho
    my = 0.06 * alto

    xa = int(
        np.clip(
            np.floor(
                x1 - mx
            ),
            0,
            W - 1
        )
    )

    xb = int(
        np.clip(
            np.ceil(
                x2 + mx
            ),
            xa + 1,
            W
        )
    )

    ya = int(
        np.clip(
            np.floor(
                y1 - my
            ),
            0,
            H - 1
        )
    )

    yb = int(
        np.clip(
            np.ceil(
                y2 + my
            ),
            ya + 1,
            H
        )
    )

    crop = img_bgr[
        ya:yb,
        xa:xb
    ]

    if (
        crop.size == 0
        or min(
            crop.shape[
                :2
            ]
        ) < 24
    ):
        return {
            "producto_en_hueco": False,
            "n": 0,
            "max_conf": 0.0
        }

    crop_pre = preprocesar(
        crop
    )

    pred = predecir_vista(
        crop_pre,
        conf
    )

    if not pred:
        return {
            "producto_en_hueco": False,
            "n": 0,
            "max_conf": 0.0
        }

    area_crop = max(
        1.0,
        crop.shape[
            0
        ]
        * crop.shape[
            1
        ]
    )

    validas = []

    for caja, score in pred:
        area_box = max(
            0.0,
            caja[
                2
            ]
            - caja[
                0
            ]
        ) * max(
            0.0,
            caja[
                3
            ]
            - caja[
                1
            ]
        )

        if (
            area_box
            / area_crop
            >= 0.018
        ):
            validas.append(
                (
                    caja,
                    score
                )
            )

    return {
        "producto_en_hueco": bool(
            validas
        ),
        "n": int(
            len(
                validas
            )
        ),
        "max_conf": float(
            max(
                [
                    p[
                        1
                    ]
                    for p in validas
                ],
                default=0.0
            )
        )
    }


def preparar_mapas_vacio(img_bgr):
    gris = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2GRAY
    ).astype(
        np.float32
    )

    hsv = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2HSV
    )

    sat = hsv[
        ...,
        1
    ].astype(
        np.float32
    ) / 255.0

    canny = cv2.Canny(
        gris.astype(
            np.uint8
        ),
        55,
        145
    ).astype(
        np.float32
    ) / 255.0

    gx = np.abs(
        cv2.Sobel(
            gris,
            cv2.CV_32F,
            1,
            0,
            ksize=3
        )
    )

    gy = np.abs(
        cv2.Sobel(
            gris,
            cv2.CV_32F,
            0,
            1,
            ksize=3
        )
    )

    thr_gx = max(
        6.0,
        float(
            np.percentile(
                gx,
                72
            )
        )
    )

    vertical = (
        (
            gx > thr_gx
        )
        & (
            gx
            > 1.12
            * gy
        )
    ).astype(
        np.float32
    )

    media_local = cv2.GaussianBlur(
        gris,
        (
            0,
            0
        ),
        2.0
    )

    media2_local = cv2.GaussianBlur(
        gris
        * gris,
        (
            0,
            0
        ),
        2.0
    )

    textura = np.sqrt(
        np.maximum(
            media2_local
            - media_local
            * media_local,
            0
        )
    )

    textura = textura / max(
        1.0,
        float(
            np.percentile(
                textura,
                95
            )
        )
    )

    textura = np.clip(
        textura,
        0.0,
        1.0
    )

    mapas = {
        "gris": gris,
        "sat": sat,
        "canny": canny,
        "vertical": vertical,
        "textura": textura
    }

    return mapas


def analizar_gondola(
    img_bgr,
    cajas,
    cajas_guard=None,
    recheck_yolo=False,
    conf_recheck=None,
    modo_referencia=False,
    mapas_precalculados=None
):
    H, W = img_bgr.shape[
        :2
    ]

    cajas_visibles = [
        [
            float(v)
            for v in b
        ]
        for b in cajas
    ]

    cajas_guard = (
        cajas_visibles
        if cajas_guard is None
        else [
            [
                float(v)
                for v in b
            ]
            for b in cajas_guard
        ]
    )

    alturas = [
        b[3] - b[1]
        for b in cajas_guard
        if b[3] > b[1]
    ]

    med_h = (
        max(
            12.0,
            float(
                np.median(
                    alturas
                )
            )
        )
        if alturas
        else max(
            30.0,
            0.12
            * H
        )
    )

    filas_producto = agrupar_filas(
        cajas_guard
    )

    lineas = detectar_lineas_repisas(
        img_bgr,
        med_h
    )

    GX0, GX1, med_w_global = estimar_x_global(
        W,
        cajas_guard,
        lineas
    )

    bandas = construir_bandas_repisas(
        H,
        cajas_guard,
        filas_producto,
        lineas,
        med_h
    )

    mapas = (
        preparar_mapas_vacio(img_bgr)
        if mapas_precalculados is None
        else mapas_precalculados
    )

    gris = mapas["gris"]
    sat = mapas["sat"]
    canny = mapas["canny"]
    vertical = mapas["vertical"]
    textura = mapas["textura"]

    features_productos = [
        features_region(
            mapas,
            b
        )
        for b in cajas_guard
        if (
            b[2] - b[0]
            >= 4
            and b[3] - b[1]
            >= 4
        )
    ]

    fallback_ref = {
        "vertical": max(
            0.01,
            float(
                np.percentile(
                    vertical,
                    72
                )
            )
        ),
        "canny": max(
            0.01,
            float(
                np.percentile(
                    canny,
                    72
                )
            )
        ),
        "textura": max(
            0.02,
            float(
                np.percentile(
                    textura,
                    72
                )
            )
        ),
        "sat": max(
            0.03,
            float(
                np.percentile(
                    sat,
                    65
                )
            )
        ),
        "entropia": 0.55
    }

    ref_global = mediana_features(
        features_productos,
        fallback_ref
    )

    resultados_filas = []

    for idx, banda in enumerate(
        bandas,
        start=1
    ):
        y1 = float(
            np.clip(
                banda[
                    "y1"
                ],
                0,
                H - 1
            )
        )

        y2 = float(
            np.clip(
                banda[
                    "y2"
                ],
                y1 + 1,
                H
            )
        )

        cajas_banda = [
            b
            for b in cajas_guard
            if (
                y1
                <= (
                    b[1] + b[3]
                ) / 2
                <= y2
            )
        ]

        cajas_vis_banda = [
            b
            for b in cajas_visibles
            if (
                y1
                <= (
                    b[1] + b[3]
                ) / 2
                <= y2
            )
        ]

        anchos = [
            b[2] - b[0]
            for b in cajas_banda
            if b[2] > b[0]
        ]

        ancho_ref = (
            max(
                8.0,
                float(
                    np.median(
                        anchos
                    )
                )
            )
            if anchos
            else med_w_global
        )

        X0, X1, fuente_x = limites_x_repisa(
            W,
            {
                "y1": y1,
                "y2": y2
            },
            cajas_banda,
            lineas,
            (
                GX0,
                GX1
            ),
            ancho_ref,
            med_h
        )

        margen = (
            CFG[
                "margen_ocupacion"
            ]
            * ancho_ref
        )

        intervalos = [
            (
                max(
                    X0,
                    b[0] - margen
                ),
                min(
                    X1,
                    b[2] + margen
                )
            )
            for b in cajas_banda
        ]

        union = fusionar_intervalos(
            intervalos
        )

        libres = huecos_desde_union(
            union,
            X0,
            X1
        )

        refs_fila = [
            features_region(
                mapas,
                b
            )
            for b in cajas_banda
            if (
                b[2] - b[0]
                >= 4
                and b[3] - b[1]
                >= 4
            )
        ]

        ref_fila = mediana_features(
            refs_fila,
            ref_global
        )

        huecos = []

        for a, b in libres:
            ancho = b - a

            tolerancia_lateral = (
                0.12
                * ancho_ref
            )

            if not union:
                tipo = "total"
            elif (
                a
                <= X0
                + tolerancia_lateral
            ):
                tipo = "lateral_izq"
            elif (
                b
                >= X1
                - tolerancia_lateral
            ):
                tipo = "lateral_der"
            else:
                tipo = "interno"

            minimo = (
                CFG[
                    "gap_lateral_min_vs_ancho"
                ]
                * ancho_ref
                if tipo.startswith(
                    "lateral"
                )
                else CFG[
                    "gap_min_vs_ancho"
                ]
                * ancho_ref
            )

            if (
                tipo != "total"
                and ancho < minimo
            ):
                continue

            alto_banda = max(
                1.0,
                y2 - y1
            )

            yi1 = (
                y1
                + CFG[
                    "vacio_inner_y"
                ]
                * alto_banda
            )

            yi2 = (
                y2
                - CFG[
                    "vacio_inner_y"
                ]
                * alto_banda
            )

            if yi2 <= yi1:
                yi1 = y1
                yi2 = y2

            feat = features_region(
                mapas,
                (
                    a,
                    yi1,
                    b,
                    yi2
                )
            )

            soporte = soporte_linea_hueco(
                lineas,
                a,
                b,
                y1,
                y2,
                med_h
            )

            scores = score_vacio_desde_features(
                feat,
                ref_fila,
                ancho,
                ancho_ref,
                soporte,
                tipo
            )

            umbral_score = (
                CFG[
                    "vacio_score_total_min"
                ]
                if tipo == "total"
                else CFG[
                    "vacio_score_final_min"
                ]
            )

            soporte_ok = True

            if tipo.startswith(
                "lateral"
            ):
                soporte_ok = (
                    soporte
                    >= CFG[
                        "vacio_soporte_lateral_min"
                    ]
                    or fuente_x
                    == "productos"
                )

            if tipo == "total":
                soporte_ok = (
                    soporte
                    >= CFG[
                        "vacio_soporte_total_min"
                    ]
                )

            if modo_referencia:
                confirmado_pre = (
                    tipo == "total"
                    or ancho
                    >= CFG[
                        "gap_min_vs_ancho"
                    ]
                    * ancho_ref
                )

                confirmado_pre = (
                    confirmado_pre
                    and (
                        tipo == "interno"
                        or soporte_ok
                        or fuente_x
                        == "linea"
                    )
                )
            else:
                confirmado_pre = (
                    scores[
                        "score_final"
                    ]
                    >= umbral_score
                    and scores[
                        "actividad_objeto"
                    ]
                    <= CFG[
                        "vacio_max_actividad_objeto"
                    ]
                    and soporte_ok
                )

            necesita_recheck = False

            if (
                confirmado_pre
                and recheck_yolo
                and not modo_referencia
            ):
                if (
                    tipo == "total"
                    and CFG[
                        "vacio_recheck_total"
                    ]
                ):
                    necesita_recheck = True

                elif (
                    tipo.startswith(
                        "lateral"
                    )
                    and CFG[
                        "vacio_recheck_laterales"
                    ]
                ):
                    necesita_recheck = True

                elif (
                    tipo == "interno"
                    and CFG[
                        "vacio_recheck_internos_dudosos"
                    ]
                    and (
                        scores[
                            "score_final"
                        ]
                        < umbral_score
                        + CFG[
                            "vacio_recheck_margen_score"
                        ]
                        or scores[
                            "actividad_objeto"
                        ]
                        > 0.34
                    )
                ):
                    necesita_recheck = True

            recheck = {
                "producto_en_hueco": False,
                "n": 0,
                "max_conf": 0.0
            }

            if necesita_recheck:
                recheck = revisar_hueco_yolo(
                    img_bgr,
                    (
                        a,
                        y1,
                        b,
                        y2
                    ),
                    (
                        CONF_PRODUCTO
                        if conf_recheck is None
                        else max(
                            float(
                                conf_recheck
                            ),
                            CONF_PRODUCTO_MIN
                        )
                    )
                )

            confirmado = (
                bool(
                    confirmado_pre
                )
                and not recheck[
                    "producto_en_hueco"
                ]
            )

            slots = max(
                1,
                int(
                    round(
                        ancho
                        / max(
                            ancho_ref,
                            1.0
                        )
                    )
                )
            )

            huecos.append(
                {
                    "class_id": 1,
                    "label": "vacio",
                    "x1": float(a),
                    "y1": float(y1),
                    "x2": float(b),
                    "y2": float(y2),
                    "tipo": tipo,
                    "repisa": idx,
                    "slots_estimados": slots,
                    "score_visual": float(
                        scores[
                            "score_visual"
                        ]
                    ),
                    "score_geometria": float(
                        scores[
                            "score_geometria"
                        ]
                    ),
                    "score_soporte": float(
                        scores[
                            "score_soporte"
                        ]
                    ),
                    "score_final": float(
                        scores[
                            "score_final"
                        ]
                    ),
                    "actividad_objeto": float(
                        scores[
                            "actividad_objeto"
                        ]
                    ),
                    "recheck_yolo": bool(
                        necesita_recheck
                    ),
                    "yolo_producto_en_hueco": bool(
                        recheck[
                            "producto_en_hueco"
                        ]
                    ),
                    "yolo_max_conf": float(
                        recheck[
                            "max_conf"
                        ]
                    ),
                    "confirmado": confirmado,
                    "vacio_total": bool(
                        tipo == "total"
                    )
                }
            )

        ancho_util = max(
            1.0,
            X1 - X0
        )

        ocupado_px = sum(
            max(
                0.0,
                bb - aa
            )
            for aa, bb
            in union
        )

        vacio_px = sum(
            h[
                "x2"
            ]
            - h[
                "x1"
            ]
            for h
            in huecos
            if h[
                "confirmado"
            ]
        )

        ocupacion_producto = float(
            np.clip(
                ocupado_px
                / ancho_util,
                0.0,
                1.0
            )
        )

        vacio_confirmado = float(
            np.clip(
                vacio_px
                / ancho_util,
                0.0,
                1.0
            )
        )

        incertidumbre = float(
            np.clip(
                1.0
                - ocupacion_producto
                - vacio_confirmado,
                0.0,
                1.0
            )
        )

        alerta = bool(
            vacio_confirmado
            >= 1.0
            - CFG[
                "alerta_ocupacion"
            ]
        )

        resultados_filas.append(
            {
                "y1": y1,
                "y2": y2,
                "x1": float(X0),
                "x2": float(X1),
                "fuente_x": fuente_x,
                "cajas": cajas_vis_banda,
                "cajas_guard": cajas_banda,
                "huecos": huecos,
                "ocupacion": ocupacion_producto,
                "vacio_confirmado": vacio_confirmado,
                "incertidumbre": incertidumbre,
                "ancho_ref": float(
                    ancho_ref
                ),
                "alerta": alerta,
                "origen": banda[
                    "origen"
                ],
                "vacia_total_candidata": bool(
                    banda[
                        "vacia_total_candidata"
                    ]
                )
            }
        )

    vacios = [
        h
        for fila
        in resultados_filas
        for h
        in fila[
            "huecos"
        ]
        if h[
            "confirmado"
        ]
    ]

    if resultados_filas:
        pesos = np.array(
            [
                max(
                    1.0,
                    f[
                        "x2"
                    ]
                    - f[
                        "x1"
                    ]
                )
                for f
                in resultados_filas
            ],
            dtype=np.float32
        )

        ocupacion = float(
            np.average(
                [
                    f[
                        "ocupacion"
                    ]
                    for f
                    in resultados_filas
                ],
                weights=pesos
            )
        )

        vacio_total = float(
            np.average(
                [
                    f[
                        "vacio_confirmado"
                    ]
                    for f
                    in resultados_filas
                ],
                weights=pesos
            )
        )

        incertidumbre_total = float(
            np.average(
                [
                    f[
                        "incertidumbre"
                    ]
                    for f
                    in resultados_filas
                ],
                weights=pesos
            )
        )
    else:
        ocupacion = None
        vacio_total = None
        incertidumbre_total = None

    return {
        "filas": resultados_filas,
        "vacios": vacios,
        "ocupacion": ocupacion,
        "vacio_confirmado": vacio_total,
        "incertidumbre": incertidumbre_total,
        "alertas": int(
            sum(
                f[
                    "alerta"
                ]
                for f
                in resultados_filas
            )
        ),
        "x_util": (
            float(GX0),
            float(GX1)
        ),
        "lineas_repisas": lineas,
        "n_productos_guard": len(
            cajas_guard
        )
    }


def dibujar_analisis(
    ax,
    img_bgr,
    cajas,
    analisis,
    titulo
):
    ax.imshow(
        cv2.cvtColor(
            img_bgr,
            cv2.COLOR_BGR2RGB
        )
    )

    ax.axis(
        "off"
    )

    for caja in cajas:
        ax.add_patch(
            plt.Rectangle(
                (
                    caja[0],
                    caja[1]
                ),
                caja[2]
                - caja[0],
                caja[3]
                - caja[1],
                fill=False,
                ec="lime",
                lw=1.1
            )
        )

    for idx, fila in enumerate(
        analisis[
            "filas"
        ],
        start=1
    ):
        ax.add_patch(
            plt.Rectangle(
                (
                    fila[
                        "x1"
                    ],
                    fila[
                        "y1"
                    ]
                ),
                fila[
                    "x2"
                ]
                - fila[
                    "x1"
                ],
                fila[
                    "y2"
                ]
                - fila[
                    "y1"
                ],
                fill=False,
                ec="deepskyblue",
                lw=1.0,
                ls="--"
            )
        )

        for hueco in fila[
            "huecos"
        ]:
            if not hueco[
                "confirmado"
            ]:
                continue

            ax.add_patch(
                plt.Rectangle(
                    (
                        hueco[
                            "x1"
                        ],
                        hueco[
                            "y1"
                        ]
                    ),
                    hueco[
                        "x2"
                    ]
                    - hueco[
                        "x1"
                    ],
                    hueco[
                        "y2"
                    ]
                    - hueco[
                        "y1"
                    ],
                    fill=True,
                    fc="red",
                    ec="red",
                    alpha=0.28,
                    lw=1.5
                )
            )

            texto = (
                "VACÍO "
                f"x{hueco['slots_estimados']} "
                f"{hueco['score_final']:.0%}"
            )

            ax.text(
                hueco[
                    "x1"
                ],
                hueco[
                    "y1"
                ]
                + 13,
                texto,
                color="white",
                fontsize=7,
                backgroundcolor="red"
            )

        ax.text(
            fila[
                "x1"
            ]
            + 4,
            fila[
                "y1"
            ]
            + 13,
            (
                f"R{idx} "
                f"prod {fila['ocupacion']:.0%} "
                f"vac {fila['vacio_confirmado']:.0%} "
                f"? {fila['incertidumbre']:.0%}"
            ),
            color="white",
            fontsize=7,
            backgroundcolor=(
                "red"
                if fila[
                    "alerta"
                ]
                else "black"
            )
        )

    ax.set_title(
        titulo,
        fontsize=9
    )


for gid in val_ids[
    :3
]:
    img = cv2.imread(
        str(
            GR[
                gid
            ][
                "path"
            ]
        )
    )

    pred_visible = [
        p[
            0
        ]
        for p
        in PRED_VAL[
            gid
        ]
        if p[
            1
        ]
        >= CONF_PRODUCTO
    ]

    analisis = analizar_gondola(
        img,
        pred_visible,
        pred_visible,
        recheck_yolo=True,
        conf_recheck=CONF_PRODUCTO
    )

    fig, ax = plt.subplots(
        figsize=(
            11,
            7
        )
    )

    dibujar_analisis(
        ax,
        img,
        pred_visible,
        analisis,
        (
            f"{gid} — "
            f"{len(pred_visible)} productos — "
            f"{len(analisis['vacios'])} vacíos — "
            f"vacío confirmado "
            f"{(analisis['vacio_confirmado'] or 0):.0%} — "
            f"incertidumbre "
            f"{(analisis['incertidumbre'] or 0):.0%}"
        )
    )

    plt.show()


### 10. Pipeline completo con doble validación de vacío

El pipeline mantiene el umbral mínimo `0.65` para producto. Después de detectar productos, construye las
repisas y genera candidatos de vacío.

Un candidato no se acepta solamente porque falte una detección. Primero debe superar la validación
estructural/visual y, para los casos de mayor riesgo, se recorta esa región y se vuelve a ejecutar YOLO con
el mismo umbral. Si aparece producto, el supuesto vacío se descarta.

La salida de cada repisa incluye ahora tres porcentajes distintos: producto confirmado, vacío confirmado e
incertidumbre.


In [ ]:
limpiar_dir(
    PIPE_DIR
)


def vacios_rect_desde_analisis(
    analisis
):
    return [
        [
            h[
                "x1"
            ],
            h[
                "y1"
            ],
            h[
                "x2"
            ],
            h[
                "y2"
            ]
        ]
        for h
        in analisis[
            "vacios"
        ]
    ]


def pipeline_foto(
    ruta,
    conf_prod=None
):
    conf_prod = (
        CONF_PRODUCTO
        if conf_prod is None
        else float(
            conf_prod
        )
    )

    conf_prod = max(
        conf_prod,
        CONF_PRODUCTO_MIN
    )

    t0 = time.time()

    img = cv2.imread(
        str(
            ruta
        )
    )

    H, W = img.shape[
        :2
    ]

    r = best_gondola.predict(
        img,
        imgsz=CFG[
            "imgsz_gondola"
        ],
        conf=CFG[
            "conf_gondola"
        ],
        device=DEVICE,
        verbose=False
    )[0]

    cajas = (
        []
        if r.boxes is None
        else [
            {
                "box": [
                    float(v)
                    for v in b
                ],
                "score": float(s)
            }
            for b, s
            in zip(
                r.boxes.xyxy
                .cpu()
                .numpy(),
                r.boxes.conf
                .cpu()
                .numpy()
            )
        ]
    )

    cajas = sam_refinar(
        img,
        filtrar_gondolas(
            cajas,
            W,
            H
        ),
        guardar_mascara=True
    )

    if MODO_ULTRA_4GB_3GB:
        descargar_sam()
        comprobar_recursos(
            "pipeline_post_sam",
            estricto=True
        )

    for c in cajas:
        c[
            "box"
        ] = c[
            "box_gdino"
        ]

    gondolas = []

    for k, c in enumerate(
        cajas
    ):
        quad = quad_desde_mascara(
            c.get(
                "mask_small"
            ),
            c.get(
                "mask_escala",
                1.0
            ),
            c[
                "box"
            ]
        )

        rect, M, metodo = rectificar(
            img,
            c[
                "box"
            ],
            quad
        )

        if not rect_valida(
            rect
        ):
            continue

        rect_pre = preprocesar(
            rect
        )

        prods_all = predecir_productos(
            rect_pre,
            conf_prod
        )

        productos_det = [
            {
                "class_id": 0,
                "label": "producto",
                "box": p[
                    0
                ],
                "score": float(
                    p[
                        1
                    ]
                )
            }
            for p
            in prods_all
            if p[
                1
            ]
            >= conf_prod
        ]

        productos = [
            p[
                "box"
            ]
            for p
            in productos_det
        ]

        productos_guard = list(
            productos
        )

        analisis = analizar_gondola(
            rect,
            productos,
            productos_guard,
            recheck_yolo=True,
            conf_recheck=conf_prod
        )

        vacios_rect = (
            vacios_rect_desde_analisis(
                analisis
            )
        )

        vacios_foto = (
            cajas_a_foto(
                vacios_rect,
                M
            )
            if vacios_rect
            else []
        )

        vacios_det = []

        for i, hueco in enumerate(
            analisis[
                "vacios"
            ]
        ):
            vacios_det.append(
                {
                    "class_id": 1,
                    "label": "vacio",
                    "box_rect": vacios_rect[
                        i
                    ],
                    "box_foto": vacios_foto[
                        i
                    ],
                    "score": float(
                        hueco[
                            "score_final"
                        ]
                    ),
                    "score_visual": float(
                        hueco[
                            "score_visual"
                        ]
                    ),
                    "actividad_objeto": float(
                        hueco[
                            "actividad_objeto"
                        ]
                    ),
                    "tipo": hueco[
                        "tipo"
                    ],
                    "slots_estimados": int(
                        hueco[
                            "slots_estimados"
                        ]
                    ),
                    "repisa": int(
                        hueco[
                            "repisa"
                        ]
                    ),
                    "recheck_yolo": bool(
                        hueco[
                            "recheck_yolo"
                        ]
                    ),
                    "yolo_max_conf": float(
                        hueco[
                            "yolo_max_conf"
                        ]
                    )
                }
            )

        ruta_rect = (
            PIPE_DIR
            / (
                f"{Path(ruta).stem}"
                f"_g{k}.jpg"
            )
        )

        cv2.imwrite(
            str(
                ruta_rect
            ),
            rect
        )

        gondolas.append(
            {
                "box": c[
                    "box"
                ],
                "score": c[
                    "score"
                ],
                "metodo": metodo,
                "M": M,
                "rect_path": ruta_rect,
                "productos": productos,
                "productos_det": productos_det,
                "productos_guard": productos_guard,
                "vacios": vacios_det,
                "analisis": analisis
            }
        )

    return {
        "foto": Path(
            ruta
        ).name,
        "gondolas": gondolas,
        "segundos": time.time()
        - t0
    }


if not MODO_ULTRA_4GB_3GB:
    _ = pipeline_foto(
        entries[
            0
        ][
            "path"
        ]
    )

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

RESULTADOS = {}

fotos_pipeline = (
    [e for e in entries if e["split"] == "val"]
    if MODO_PIPELINE == "val"
    else list(entries)
)

print(
    f"Modo pipeline: {MODO_PIPELINE} | "
    f"fotos a procesar: {len(fotos_pipeline)}"
)
registrar_recursos("antes_pipeline", reset_gpu_peak=True)

for i, e in enumerate(
    fotos_pipeline
):
    RESULTADOS[
        e[
            "nombre"
        ]
    ] = pipeline_foto(
        e[
            "path"
        ]
    )

    if MODO_ULTRA_4GB_3GB:
        descargar_sam()
        limpiar_memoria_agresivo()
        comprobar_recursos(
            f"pipeline_{i+1}",
            estricto=True
        )

    if (
        i + 1
    ) % 25 == 0:
        print(
            f"  {i+1}/"
            f"{len(fotos_pipeline)}"
        )

tiempos = np.array(
    [
        r[
            "segundos"
        ]
        for r
        in RESULTADOS.values()
    ]
)

vram = (
    torch.cuda.max_memory_allocated()
    / 1e9
    if torch.cuda.is_available()
    else 0.0
)

n_gondolas = sum(
    len(
        r[
            "gondolas"
        ]
    )
    for r
    in RESULTADOS.values()
)

n_productos = sum(
    len(
        g[
            "productos"
        ]
    )
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
)

n_vacios = sum(
    len(
        g[
            "vacios"
        ]
    )
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
)

n_slots_vacios = sum(
    v[
        "slots_estimados"
    ]
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
    for v
    in g[
        "vacios"
    ]
)

n_alertas = sum(
    g[
        "analisis"
    ][
        "alertas"
    ]
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
)

METRICAS[
    "inferencia"
] = {
    "seg_media": float(
        tiempos.mean()
    ),
    "seg_p95": float(
        np.percentile(
            tiempos,
            95
        )
    ),
    "vram_pico_gb": float(
        vram
    ),
    "gondolas": int(
        n_gondolas
    ),
    "productos": int(
        n_productos
    ),
    "vacios": int(
        n_vacios
    ),
    "slots_vacios_estimados": int(
        n_slots_vacios
    )
}

print(
    f"\nTiempo por foto: "
    f"media {tiempos.mean():.2f}s | "
    f"p95 {np.percentile(tiempos,95):.2f}s | "
    f"máx {tiempos.max():.2f}s"
)

print(
    f"VRAM pico en inferencia: "
    f"{vram:.2f} GB"
)

print(
    f"Góndolas analizadas: "
    f"{n_gondolas}"
)

print(
    f"Productos detectados: "
    f"{n_productos}"
)

print(
    f"Regiones vacías detectadas: "
    f"{n_vacios}"
)

print(
    f"Posiciones vacías estimadas: "
    f"{n_slots_vacios}"
)

print(
    f"Repisas en alerta: "
    f"{n_alertas}"
)


registrar_recursos("post_pipeline")
METRICAS["recursos"] = dict(METRICAS_RECURSOS)


In [ ]:
muestra = [
    n
    for n in sorted(RESULTADOS)
    if SPLIT_FOTO[n] == "val"
    and RESULTADOS[n]["gondolas"]
][:8]

for nombre in muestra:
    res = RESULTADOS[nombre]
    g = res["gondolas"][0]

    img = cv2.imread(str(ENTRY[nombre]["path"]))
    rect = cv2.imread(str(g["rect_path"]))

    fig, ax = plt.subplots(1, 2, figsize=(20, 7))

    ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax[0].axis("off")

    for gg in res["gondolas"]:
        b = gg["box"]

        ax[0].add_patch(
            plt.Rectangle(
                (b[0], b[1]),
                b[2] - b[0],
                b[3] - b[1],
                fill=False,
                ec="cyan",
                lw=2.5
            )
        )

        productos_foto = cajas_a_foto(
            gg["productos"],
            gg["M"]
        )

        for p in productos_foto:
            ax[0].add_patch(
                plt.Rectangle(
                    (p[0], p[1]),
                    p[2] - p[0],
                    p[3] - p[1],
                    fill=False,
                    ec="lime",
                    lw=0.9
                )
            )

        for vacio in gg["vacios"]:
            p = vacio["box_foto"]

            ax[0].add_patch(
                plt.Rectangle(
                    (p[0], p[1]),
                    p[2] - p[0],
                    p[3] - p[1],
                    fill=True,
                    fc="red",
                    ec="red",
                    alpha=0.25,
                    lw=1.2
                )
            )

            ax[0].text(
                p[0],
                p[1],
                "1 vacío",
                color="white",
                fontsize=7,
                backgroundcolor="red"
            )

    ax[0].set_title(
        f"{nombre} — {res['segundos']:.2f}s",
        fontsize=10
    )

    a = g["analisis"]

    dibujar_analisis(
        ax[1],
        rect,
        g["productos"],
        a,
        (
            f"Aplanada ({g['metodo']}) — "
            f"{len(g['productos'])} prod. — "
            f"{len(g['vacios'])} vacíos — "
            f"ocupación {(a['ocupacion'] or 0):.0%}"
        )
    )

    plt.tight_layout()
    plt.show()


### 11. Evaluación de vacíos mejorada

La referencia proxy de vacío se construye desde las cajas de producto de referencia usando principalmente
geometría de repisa. La predicción, en cambio, debe superar la validación visual y la revalidación YOLO.
Esto reduce la circularidad de evaluar exactamente el mismo criterio contra sí mismo.

Se calculan ahora:

- Precision por área de vacío;
- Recall por área de vacío;
- F1 por área;
- IoU de regiones vacías;
- error absoluto de ocupación;
- error absoluto de porcentaje vacío;
- coincidencia de alertas.

Estas métricas siguen siendo `proxy` mientras no exista anotación humana explícita de cada espacio vacío.


In [ ]:
# [Celda 38]  Mejores/peores: góndola
# ---- 11.1 Góndola ----
filas_g = []
for nombre in [n for n in RESULTADOS if SPLIT_FOTO[n] == "val"]:
    gt = [c["box"] for c in etiquetas_gondola[nombre]]
    pr = [g["box"] for g in RESULTADOS[nombre]["gondolas"]]
    if not gt and not pr:
        continue
    ious = [max([iou(g, p) for p in pr], default=0.0) for g in gt] or [0.0]
    filas_g.append({"foto": nombre, "iou_medio": float(np.mean(ious)), "n_gt": len(gt), "n_pred": len(pr),
                    "nitidez_fft": ENTRY[nombre]["nitidez_fft"], "reflejos": ENTRY[nombre]["reflejos"],
                    "kind": ENTRY[nombre]["kind"]})
df_g = pd.DataFrame(filas_g).sort_values("iou_medio", ascending=False)
print(f"Góndola (val): IoU medio = {df_g.iou_medio.mean():.3f}")

def mostrar_gondolas(df, titulo):
    if len(df) == 0:
        print('Sin casos para', titulo); return
    fig, axes = plt.subplots(1, len(df), figsize=(7 * len(df), 5.5))
    for ax, (_, f) in zip(np.atleast_1d(axes), df.iterrows()):
        ax.imshow(cv2.cvtColor(cv2.imread(str(ENTRY[f.foto]["path"])), cv2.COLOR_BGR2RGB)); ax.axis("off")
        for c in etiquetas_gondola[f.foto]:
            b = c["box"]; ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, ec="lime", lw=2.5))
        for g in RESULTADOS[f.foto]["gondolas"]:
            b = g["box"]; ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, ec="magenta", lw=2, ls="--"))
        ax.set_title(f"{f.foto}\nIoU={f.iou_medio:.2f} | nitidez={f.nitidez_fft:.3f} | reflejos={f.reflejos:.1%}", fontsize=9)
    fig.suptitle(f"{titulo} (verde = etiqueta, magenta = predicción)", fontsize=12)
    plt.tight_layout(); plt.show(); plt.close(fig)

mostrar_gondolas(df_g.head(3), "Góndola — 3 MEJORES")
mostrar_gondolas(df_g.tail(3).iloc[::-1], "Góndola — 3 PEORES")


In [ ]:
# [Celda 39]  Mejores/peores: producto
# ---- 11.2 Producto ----
filas_p = []
for gid in val_ids:
    pred = [p for p in PRED_VAL[gid] if p[1] >= CONF_PRODUCTO]
    gt = GT_VAL[gid]
    if not gt and not pred:
        continue
    tp, fp, fn = emparejar(pred, gt)
    P = len(tp) / max(1, len(tp) + len(fp)); R = len(tp) / max(1, len(tp) + len(fn))
    g = GR[gid]
    filas_p.append({"id": gid, "F1": 2*P*R/max(1e-9, P+R), "P": P, "R": R, "n_gt": len(gt), "n_pred": len(pred),
                    "nitidez_fft": g["nitidez_fft"], "reflejos": g["reflejos"], "brillo": g["brillo"],
                    "kind": g["kind"], "metodo": g["metodo"]})
df_p = pd.DataFrame(filas_p).sort_values("F1", ascending=False)

def mostrar_productos(df, titulo):
    if len(df) == 0:
        print('Sin casos para', titulo); return
    fig, axes = plt.subplots(1, len(df), figsize=(8 * len(df), 6))
    for ax, (_, f) in zip(np.atleast_1d(axes), df.iterrows()):
        img = cv2.imread(str(GR[f.id]["path"]))
        pred = [p for p in PRED_VAL[f.id] if p[1] >= CONF_PRODUCTO]; gt = GT_VAL[f.id]
        tp, fp, fn = emparejar(pred, gt)
        dibujar(ax, cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
                [([pred[i][0] for i, _ in tp], "lime", 1.5, "-"), ([pred[i][0] for i in fp], "red", 1.5, "-"),
                 ([gt[j] for j in fn], "orange", 1.5, "--")],
                f"{f.id} [{f.kind}, {f.metodo}]\nF1={f.F1:.2f} P={f.P:.2f} R={f.R:.2f} | GT={f.n_gt} | "
                f"nitidez={f.nitidez_fft:.3f} reflejos={f.reflejos:.1%}")
    fig.suptitle(f"{titulo} (verde=TP, rojo=FP, naranja=FN)", fontsize=12)
    plt.tight_layout(); plt.show(); plt.close(fig)

mostrar_productos(df_p.head(3), "Producto — 3 MEJORES")
mostrar_productos(df_p.tail(3).iloc[::-1], "Producto — 3 PEORES")
print(df_p.head(3).round(3).to_string(index=False)); print("..."); print(df_p.tail(3).round(3).to_string(index=False))


In [ ]:
def mascara_vacios(
    shape,
    analisis
):
    H, W = shape[
        :2
    ]

    mask = np.zeros(
        (
            H,
            W
        ),
        dtype=np.uint8
    )

    for hueco in analisis[
        "vacios"
    ]:
        x1 = int(
            np.clip(
                round(
                    hueco[
                        "x1"
                    ]
                ),
                0,
                W - 1
            )
        )

        x2 = int(
            np.clip(
                round(
                    hueco[
                        "x2"
                    ]
                ),
                x1 + 1,
                W
            )
        )

        y1 = int(
            np.clip(
                round(
                    hueco[
                        "y1"
                    ]
                ),
                0,
                H - 1
            )
        )

        y2 = int(
            np.clip(
                round(
                    hueco[
                        "y2"
                    ]
                ),
                y1 + 1,
                H
            )
        )

        mask[
            y1:y2,
            x1:x2
        ] = 1

    return mask


def metricas_masks(
    ref,
    pred
):
    ref = ref > 0
    pred = pred > 0

    tp = int(
        np.logical_and(
            ref,
            pred
        ).sum()
    )

    fp = int(
        np.logical_and(
            ~ref,
            pred
        ).sum()
    )

    fn = int(
        np.logical_and(
            ref,
            ~pred
        ).sum()
    )

    union = tp + fp + fn

    if (
        ref.sum() == 0
        and pred.sum() == 0
    ):
        precision = 1.0
        recall = 1.0
        f1 = 1.0
        iou_vacio = 1.0
    else:
        precision = (
            tp
            / max(
                1,
                tp + fp
            )
        )

        recall = (
            tp
            / max(
                1,
                tp + fn
            )
        )

        f1 = (
            2
            * precision
            * recall
            / max(
                1e-9,
                precision
                + recall
            )
        )

        iou_vacio = (
            tp
            / max(
                1,
                union
            )
        )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "P": float(
            precision
        ),
        "R": float(
            recall
        ),
        "F1": float(
            f1
        ),
        "IoU": float(
            iou_vacio
        )
    }


filas_v = []

TP_AREA = 0
FP_AREA = 0
FN_AREA = 0

for gid in val_ids:
    img = cv2.imread(
        str(
            GR[
                gid
            ][
                "path"
            ]
        )
    )

    mapas_vacio = preparar_mapas_vacio(
        img
    )

    pred_visible = [
        p[
            0
        ]
        for p
        in PRED_VAL[
            gid
        ]
        if p[
            1
        ]
        >= CONF_PRODUCTO
    ]

    referencia = GT_VAL[
        gid
    ]

    a_ref = analizar_gondola(
        img,
        referencia,
        referencia,
        recheck_yolo=False,
        modo_referencia=True,
        mapas_precalculados=mapas_vacio
    )

    a_pred = analizar_gondola(
        img,
        pred_visible,
        pred_visible,
        recheck_yolo=True,
        conf_recheck=CONF_PRODUCTO,
        modo_referencia=False,
        mapas_precalculados=mapas_vacio
    )

    if (
        a_ref[
            "ocupacion"
        ]
        is None
        or a_pred[
            "ocupacion"
        ]
        is None
    ):
        continue

    mask_ref = mascara_vacios(
        img.shape,
        a_ref
    )

    mask_pred = mascara_vacios(
        img.shape,
        a_pred
    )

    mm = metricas_masks(
        mask_ref,
        mask_pred
    )

    TP_AREA += mm[
        "tp"
    ]

    FP_AREA += mm[
        "fp"
    ]

    FN_AREA += mm[
        "fn"
    ]

    vac_ref = (
        a_ref[
            "vacio_confirmado"
        ]
        or 0.0
    )

    vac_pred = (
        a_pred[
            "vacio_confirmado"
        ]
        or 0.0
    )

    filas_v.append(
        {
            "id": gid,
            "ocup_ref": a_ref[
                "ocupacion"
            ],
            "ocup_pred": a_pred[
                "ocupacion"
            ],
            "vacio_ref": vac_ref,
            "vacio_pred": vac_pred,
            "incertidumbre_pred": (
                a_pred[
                    "incertidumbre"
                ]
                or 0.0
            ),
            "error_abs": abs(
                a_ref[
                    "ocupacion"
                ]
                - a_pred[
                    "ocupacion"
                ]
            ),
            "error_vacio": abs(
                vac_ref
                - vac_pred
            ),
            "precision_vacio": mm[
                "P"
            ],
            "recall_vacio": mm[
                "R"
            ],
            "f1_vacio": mm[
                "F1"
            ],
            "iou_vacio": mm[
                "IoU"
            ],
            "vacios_ref": len(
                a_ref[
                    "vacios"
                ]
            ),
            "vacios_pred": len(
                a_pred[
                    "vacios"
                ]
            ),
            "repisas_ref": len(
                a_ref[
                    "filas"
                ]
            ),
            "repisas_pred": len(
                a_pred[
                    "filas"
                ]
            ),
            "alerta_coincide": (
                a_ref[
                    "alertas"
                ]
                > 0
            ) == (
                a_pred[
                    "alertas"
                ]
                > 0
            )
        }
    )


df_v = pd.DataFrame(
    filas_v
)

if len(
    df_v
):
    precision_area = (
        TP_AREA
        / max(
            1,
            TP_AREA
            + FP_AREA
        )
    )

    recall_area = (
        TP_AREA
        / max(
            1,
            TP_AREA
            + FN_AREA
        )
    )

    f1_area = (
        2
        * precision_area
        * recall_area
        / max(
            1e-9,
            precision_area
            + recall_area
        )
    )

    iou_area = (
        TP_AREA
        / max(
            1,
            TP_AREA
            + FP_AREA
            + FN_AREA
        )
    )

    mae_ocupacion = float(
        df_v[
            "error_abs"
        ].mean()
    )

    mae_vacio = float(
        df_v[
            "error_vacio"
        ].mean()
    )

    alertas_coinciden = float(
        df_v[
            "alerta_coincide"
        ].mean()
    )

    METRICAS[
        "vacios_proxy"
    ] = {
        "precision_area": float(
            precision_area
        ),
        "recall_area": float(
            recall_area
        ),
        "f1_area": float(
            f1_area
        ),
        "iou_area": float(
            iou_area
        ),
        "mae_ocupacion": mae_ocupacion,
        "mae_vacio": mae_vacio,
        "alertas_coinciden": alertas_coinciden,
        "n_gondolas": int(
            len(
                df_v
            )
        )
    }

    df_v = df_v.sort_values(
        [
            "f1_vacio",
            "iou_vacio",
            "error_vacio"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    print(
        f"Precision de vacío por área: "
        f"{precision_area:.1%}"
    )

    print(
        f"Recall de vacío por área: "
        f"{recall_area:.1%}"
    )

    print(
        f"F1 de vacío por área: "
        f"{f1_area:.1%}"
    )

    print(
        f"IoU global de vacío: "
        f"{iou_area:.1%}"
    )

    print(
        f"Error medio de ocupación: "
        f"{mae_ocupacion:.1%}"
    )

    print(
        f"Error medio de porcentaje vacío: "
        f"{mae_vacio:.1%}"
    )

    print(
        f"Alertas coincidentes: "
        f"{alertas_coinciden:.1%}"
    )


def mostrar_vacios(
    df,
    titulo
):
    if len(
        df
    ) == 0:
        print(
            "Sin casos para",
            titulo
        )

        return

    for _, fila in df.iterrows():
        img = cv2.imread(
            str(
                GR[
                    fila.id
                ][
                    "path"
                ]
            )
        )

        mapas_vacio = preparar_mapas_vacio(
            img
        )

        pred_visible = [
            p[
                0
            ]
            for p
            in PRED_VAL[
                fila.id
            ]
            if p[
                1
            ]
            >= CONF_PRODUCTO
        ]

        referencia = GT_VAL[
            fila.id
        ]

        analisis_ref = analizar_gondola(
            img,
            referencia,
            referencia,
            recheck_yolo=False,
            modo_referencia=True,
            mapas_precalculados=mapas_vacio
        )

        analisis_pred = analizar_gondola(
            img,
            pred_visible,
            pred_visible,
            recheck_yolo=True,
            conf_recheck=CONF_PRODUCTO,
            mapas_precalculados=mapas_vacio
        )

        fig, ax = plt.subplots(
            1,
            2,
            figsize=(
                18,
                6
            )
        )

        dibujar_analisis(
            ax[
                0
            ],
            img,
            referencia,
            analisis_ref,
            (
                "Referencia geométrica — "
                f"{len(analisis_ref['vacios'])} vacíos — "
                f"vac "
                f"{(analisis_ref['vacio_confirmado'] or 0):.0%}"
            )
        )

        dibujar_analisis(
            ax[
                1
            ],
            img,
            pred_visible,
            analisis_pred,
            (
                "Predicción — "
                f"{len(analisis_pred['vacios'])} vacíos — "
                f"vac "
                f"{(analisis_pred['vacio_confirmado'] or 0):.0%} — "
                f"? "
                f"{(analisis_pred['incertidumbre'] or 0):.0%}"
            )
        )

        fig.suptitle(
            (
                f"{titulo}: "
                f"{fila.id} — "
                f"P {fila.precision_vacio:.0%} — "
                f"R {fila.recall_vacio:.0%} — "
                f"F1 {fila.f1_vacio:.0%} — "
                f"IoU {fila.iou_vacio:.0%}"
            ),
            fontsize=12
        )

        plt.tight_layout()
        plt.show()


if len(
    df_v
):
    mostrar_vacios(
        df_v.head(
            3
        ),
        "Vacíos — MEJOR"
    )

    mostrar_vacios(
        df_v.tail(
            3
        ).iloc[
            ::-1
        ],
        "Vacíos — PEOR"
    )


In [ ]:
# [Celda 41]  Diagnóstico vs versión anterior
# ---- 11.4 Diagnóstico: ¿qué explica los peores casos? ----
print("F1 de producto por tipo de foto:");      print(df_p.groupby("kind")["F1"].agg(["mean", "count"]).round(3), "\n")
print("F1 de producto por método de aplanado:"); print(df_p.groupby("metodo")["F1"].agg(["mean", "count"]).round(3), "\n")
corr = df_p[["F1", "nitidez_fft", "reflejos", "brillo", "n_gt"]].corr()["F1"].drop("F1").round(3)
print("Correlación de F1 con calidad de imagen:"); print(corr.to_string(), "\n")

peor, mejor = df_p.tail(3), df_p.head(3)
print("Comparación 3 mejores vs 3 peores (promedios):")
print(pd.DataFrame({"mejores": mejor[["F1", "nitidez_fft", "reflejos", "brillo", "n_gt"]].mean(),
                    "peores": peor[["F1", "nitidez_fft", "reflejos", "brillo", "n_gt"]].mean()}).round(3))

print("\nResumen vs versión anterior:")
anterior = {"gondola": "P 80.4% R 67.2% mAP50 0.769 (val con fuga de datos)",
            "producto": "P 63.8% R 57.8% mAP50 0.636 (recortes de repisa)",
            "modelo_unico_producto": "P 31.9% R 27.1% AP50 0.190"}
for k, v in anterior.items():
    print(f"  antes  {k:24s}: {v}")
for k, v in METRICAS.items():
    print(f"  ahora  {k:24s}: " + ", ".join(f"{a}={b:.3f}" if isinstance(b, float) else f"{a}={b}" for a, b in v.items()))

### 11b. Diagnóstico: sobreajuste, galería completa y auditoría de errores

- **Sobreajuste:** compara métricas en train y val, y muestra las curvas de pérdida.
- **Galería:** guarda en `MyDrive/stockeye/resultados_producto/` TODAS las góndolas de val (y 20 de train) con etiqueta vs predicción.
- **Auditoría:** recortes de falsos positivos y negativos para ver si son errores del modelo o de las etiquetas.


In [ ]:
# [Celda 43]  Diagnóstico de sobreajuste: train vs val + curvas de pérdida
mt = best_producto.val(data=str(YAML_PRODUCTO), split="train", imgsz=CFG["imgsz_producto"], device=DEVICE,
                       max_det=1000, verbose=False, plots=False)
mv = best_producto.val(data=str(YAML_PRODUCTO), split="val", imgsz=CFG["imgsz_producto"], device=DEVICE,
                       max_det=1000, verbose=False, plots=False)
df_ov = pd.DataFrame({"train": [mt.box.mp, mt.box.mr, mt.box.map50], "val": [mv.box.mp, mv.box.mr, mv.box.map50]},
                     index=["P", "R", "mAP50"]).round(3)
df_ov["brecha"] = (df_ov.train - df_ov.val).round(3)
print(df_ov, "\n")

brecha, map_tr = df_ov.loc["mAP50", "brecha"], df_ov.loc["mAP50", "train"]
if brecha > 0.20:
    print("=> SOBREAJUSTE: el modelo memoriza train y no generaliza a val.")
elif map_tr < 0.70:
    print("=> NO es sobreajuste: ni en train llega alto. El problema son ETIQUETAS INCONSISTENTES "
          "(GDINO marca un producto en una foto y lo omite en otra) o falta de capacidad.")
else:
    print("=> Brecha moderada: el modelo generaliza parecido en train y val.")

res = pd.read_csv(BEST_PRODUCTO.parent.parent / "results.csv"); res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
for k, nombre in enumerate(["box_loss", "cls_loss", "dfl_loss"]):
    ax[k].plot(res["epoch"], res[f"train/{nombre}"], label="train")
    ax[k].plot(res["epoch"], res[f"val/{nombre}"], label="val")
    ax[k].set_title(nombre); ax[k].set_xlabel("época"); ax[k].legend()
plt.suptitle("Si la pérdida de val sube mientras la de train baja = sobreajuste")
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# [Celda 44]  Galería completa de resultados de producto -> Drive (todas las de val + 20 de train)
GALERIA = GALERIA_LOCAL
for s in ("val", "train"):
    limpiar_dir(GALERIA / s)

CONF_VIS = float(df_conf.loc[df_conf["F1"].idxmax(), "conf"])   # umbral de mejor equilibrio P/R para mirar
print(f"Umbral para visualizar: {CONF_VIS} (máx. F1) | umbral P>=80%: {CONF_PRODUCTO}")

train_ids = [g["id"] for g in gondolas_rect if g["split"] == "train"]
muestra_train = random.Random(SEED).sample(train_ids, min(20, len(train_ids)))
PRED_TRAIN = {
    gid: predecir_productos(
        preprocesar(
            cv2.imread(
                str(
                    GR[
                        gid
                    ][
                        "path"
                    ]
                )
            )
        ),
        CONF_PRODUCTO
    )
    for gid
    in muestra_train
}

def panel_resultado(gid, preds, conf, ruta_salida=None, mostrar=False):
    img = cv2.cvtColor(cv2.imread(str(GR[gid]["path"])), cv2.COLOR_BGR2RGB)
    gt = gt_producto(gid)
    pred = [p for p in preds if p[1] >= conf]
    tp, fp, fn = emparejar(pred, gt)
    P = len(tp) / max(1, len(tp) + len(fp)); R = len(tp) / max(1, len(tp) + len(fn))
    fig, ax = plt.subplots(1, 2, figsize=(20, 8))
    dibujar(ax[0], img, [(gt, "cyan", 1.2, "-")], f"Etiqueta GDINO+SAM: {len(gt)} productos")
    dibujar(ax[1], img, [([pred[i][0] for i, _ in tp], "lime", 1.5, "-"), ([pred[i][0] for i in fp], "red", 1.5, "-"),
                         ([gt[j] for j in fn], "orange", 1.5, "--")],
            f"YOLO conf>={conf}: TP={len(tp)} FP={len(fp)} FN={len(fn)} | P={P:.2f} R={R:.2f}")
    fig.suptitle(f"{gid} [{GR[gid]['split']}, {GR[gid]['kind']}, {GR[gid]['metodo']}]  "
                 f"(verde=acierto, rojo=FP, naranja=no detectado)", fontsize=11)
    plt.tight_layout()
    if ruta_salida:
        fig.savefig(ruta_salida, dpi=90)
    if mostrar:
        plt.show()
    plt.close(fig)

for gid in val_ids:
    panel_resultado(gid, PRED_VAL[gid], CONF_VIS, GALERIA / "val" / f"{gid}.jpg")
for gid in muestra_train:
    panel_resultado(gid, PRED_TRAIN[gid], CONF_VIS, GALERIA / "train" / f"{gid}.jpg")
print(f"Guardadas {len(val_ids)} (val) + {len(muestra_train)} (train) imágenes en {GALERIA}")

for gid in val_ids[:4]:              # muestra en pantalla; el resto está en Drive
    panel_resultado(gid, PRED_VAL[gid], CONF_VIS, mostrar=True)


if (
    DRIVE_CAN_WRITE
    and SUBIR_GALERIA_AUTOMATICO
):
    print(
        "Subiendo galería a "
        "la carpeta compartida..."
    )

    galeria_drive_id = (
        subir_carpeta_local_drive(
            GALERIA,
            SHARED_DRIVE_FOLDER_ID
        )
    )

    print(
        "Galería subida a Drive:",
        galeria_drive_id
    )
else:
    print(
        "La galería quedó local en:",
        GALERIA
    )

    print(
        "La subida automática de la galería "
        "está desactivada para evitar timeouts."
    )


In [ ]:
# [Celda 45]  Auditoría de errores: ¿los FP son productos reales sin etiqueta?
def recortes(lista, n=48, margen=0.15):
    lista = random.Random(SEED).sample(lista, min(n, len(lista)))
    out = []
    for gid, b in lista:
        img = cv2.imread(str(GR[gid]["path"])); H, W = img.shape[:2]
        mw, mh = (b[2] - b[0]) * margen, (b[3] - b[1]) * margen
        x1, y1 = int(max(0, b[0] - mw)), int(max(0, b[1] - mh))
        x2, y2 = int(min(W, b[2] + mw)), int(min(H, b[3] + mh))
        if x2 > x1 and y2 > y1:
            out.append((gid, cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)))
    return out

FP_LIST, FN_LIST = [], []
for gid in val_ids:
    pred = [p for p in PRED_VAL[gid] if p[1] >= CONF_VIS]; gt = GT_VAL[gid]
    tp, fp, fn = emparejar(pred, gt)
    FP_LIST += [(gid, pred[i][0]) for i in fp]
    FN_LIST += [(gid, gt[j]) for j in fn]

def grilla_recortes(items, titulo, cols=8):
    rows = max(1, int(np.ceil(len(items) / cols)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.6))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, (k, (gid, rc)) in zip(np.atleast_1d(axes).ravel(), enumerate(items)):
        ax.imshow(rc); ax.set_title(f"#{k}", fontsize=8)
    fig.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print(f"Total val: FP={len(FP_LIST)} | FN={len(FN_LIST)} (conf={CONF_VIS})")
grilla_recortes(recortes(FP_LIST), "FALSOS POSITIVOS (YOLO detectó, GDINO no etiquetó): ¿cuántos SON productos reales?")
grilla_recortes(recortes(FN_LIST), "FALSOS NEGATIVOS (GDINO etiquetó, YOLO no detectó): ¿cuántos son productos reales?")
print("Cuenta a ojo cuántos de los 48 FP son productos reales.\n"
      "Si son más de la mitad, la precisión REAL es mucho mayor que la medida: el problema son las etiquetas, no el modelo.")


### 11c. Resumen final de productos y vacíos

Este resumen muestra el rendimiento del algoritmo de vacío antes de exportar.

`Exactitud de ocupación` se representa como `1 - error absoluto medio de ocupación`. `IoU de vacío` mide directamente cuánto se solapan las regiones vacías de referencia y predicción. `Coincidencia de alertas` mide si ambos análisis toman la misma decisión sobre la existencia de una repisa en alerta.

También se informa cuántos productos y cuántos vacíos produjo el pipeline completo. Las métricas de vacío siguen siendo proxy mientras no exista una anotación humana específica de los huecos.


In [ ]:
n_productos_pipeline = sum(
    len(
        g[
            "productos"
        ]
    )
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
)

n_vacios_pipeline = sum(
    len(
        g[
            "vacios"
        ]
    )
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
)

n_slots_vacios_pipeline = sum(
    v[
        "slots_estimados"
    ]
    for r
    in RESULTADOS.values()
    for g
    in r[
        "gondolas"
    ]
    for v
    in g[
        "vacios"
    ]
)

if (
    "METRICAS"
    in globals()
    and "vacios_proxy"
    in METRICAS
):
    vm = METRICAS[
        "vacios_proxy"
    ]

    nombres = [
        "Precisión vacío",
        "Recall vacío",
        "F1 vacío",
        "IoU vacío",
        "Acuerdo alertas"
    ]

    valores = [
        vm[
            "precision_area"
        ]
        * 100,
        vm[
            "recall_area"
        ]
        * 100,
        vm[
            "f1_area"
        ]
        * 100,
        vm[
            "iou_area"
        ]
        * 100,
        vm[
            "alertas_coinciden"
        ]
        * 100
    ]

    fig, ax = plt.subplots(
        figsize=(
            10,
            5
        )
    )

    barras = ax.bar(
        nombres,
        valores
    )

    ax.set_ylim(
        0,
        100
    )

    ax.set_ylabel(
        "Porcentaje"
    )

    ax.set_title(
        "Evaluación proxy de espacios vacíos"
    )

    ax.tick_params(
        axis="x",
        rotation=18
    )

    for barra, valor in zip(
        barras,
        valores
    ):
        ax.text(
            barra.get_x()
            + barra.get_width()
            / 2,
            valor
            + 1.2,
            f"{valor:.1f}%",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()
    plt.close(
        fig
    )

    print(
        "Error medio de ocupación:",
        f"{vm['mae_ocupacion']:.1%}"
    )

    print(
        "Error medio de porcentaje vacío:",
        f"{vm['mae_vacio']:.1%}"
    )

print(
    "Productos detectados por el pipeline:",
    n_productos_pipeline
)

print(
    "Regiones vacías detectadas:",
    n_vacios_pipeline
)

print(
    "Posiciones vacías estimadas:",
    n_slots_vacios_pipeline
)


print("\nPRUEBA DE OPTIMIZACIÓN")
print("-" * 60)
pruebas_opt = {
    "YOLO producto sigue siendo 11s": "yolo11s" in str(CFG["yolo_producto"]).lower() or str(CFG["yolo_producto"]).endswith("best.pt"),
    "Resolución producto sigue en 1024": int(CFG["imgsz_producto"]) == 1024,
    "Confianza mínima sigue en 0.65": float(CONF_PRODUCTO_MIN) == 0.65,
    "SAM usa lote 32": int(CFG["sam_lote"]) == 32,
    "Rectificación máxima sigue en 1600": int(CFG["rect_max_lado"]) == 1600,
    "Detector vacío mantiene recheck YOLO": bool(CFG["vacio_recheck_total"]),
}

for nombre_prueba, ok in pruebas_opt.items():
    print(f"{'OK' if ok else 'REVISAR'} - {nombre_prueba}")

if all(pruebas_opt.values()):
    print("Optimización segura aplicada sin reducir resolución, umbral ni capacidad del detector final.")
else:
    print("Hay parámetros críticos distintos de los esperados; revisar antes de comparar eficacia.")

if METRICAS_RECURSOS:
    print("\nRecursos medidos en esta ejecución:")
    for etapa, dato in METRICAS_RECURSOS.items():
        print(
            f"  {etapa}: "
            f"RAM={dato['ram_gb'] if dato['ram_gb'] is not None else 'n/d'} GB | "
            f"VRAM pico={dato['vram_peak_gb']:.2f} GB"
        )


print("\nPRUEBA REAL DE REUTILIZACIÓN DE MAPAS")
print("-" * 60)

if len(val_ids):
    gid_prueba_opt = val_ids[0]
    img_prueba_opt = cv2.imread(
        str(
            GR[
                gid_prueba_opt
            ][
                "path"
            ]
        )
    )

    cajas_prueba_opt = [
        p[0]
        for p
        in PRED_VAL[
            gid_prueba_opt
        ]
        if p[1]
        >= CONF_PRODUCTO
    ]

    t0_opt = time.perf_counter()
    analisis_sin_cache = analizar_gondola(
        img_prueba_opt,
        cajas_prueba_opt,
        cajas_prueba_opt,
        recheck_yolo=False
    )
    tiempo_sin_cache = time.perf_counter() - t0_opt

    t0_opt = time.perf_counter()
    mapas_prueba_opt = preparar_mapas_vacio(
        img_prueba_opt
    )
    analisis_con_cache = analizar_gondola(
        img_prueba_opt,
        cajas_prueba_opt,
        cajas_prueba_opt,
        recheck_yolo=False,
        mapas_precalculados=mapas_prueba_opt
    )
    tiempo_con_cache = time.perf_counter() - t0_opt

    campos_prueba = [
        "ocupacion",
        "vacio_confirmado",
        "incertidumbre",
        "alertas",
        "n_productos_guard"
    ]

    igualdad_campos = all(
        analisis_sin_cache[k]
        == analisis_con_cache[k]
        for k
        in campos_prueba
    )

    igualdad_vacios = (
        json.dumps(
            analisis_sin_cache[
                "vacios"
            ],
            sort_keys=True
        )
        == json.dumps(
            analisis_con_cache[
                "vacios"
            ],
            sort_keys=True
        )
    )

    print(
        "Resultado idéntico:",
        bool(
            igualdad_campos
            and igualdad_vacios
        )
    )

    print(
        f"Sin reutilizar mapas: "
        f"{tiempo_sin_cache:.4f}s"
    )

    print(
        f"Con mapas reutilizables: "
        f"{tiempo_con_cache:.4f}s"
    )

    METRICAS[
        "prueba_optimizacion"
    ] = {
        "gid": gid_prueba_opt,
        "salida_identica": bool(
            igualdad_campos
            and igualdad_vacios
        ),
        "tiempo_sin_cache_s": float(
            tiempo_sin_cache
        ),
        "tiempo_con_cache_s": float(
            tiempo_con_cache
        )
    }
else:
    print(
        "No hay imágenes de validación disponibles "
        "para la prueba automática."
    )


### Subidas opcionales a Drive

Las carpetas con muchas imágenes no se suben automáticamente porque no son necesarias para entrenar y pueden provocar `TimeoutError`. Si después quieres guardar el material de etiquetado o la galería, ejecuta la siguiente celda cambiando el interruptor correspondiente a `True`.


In [ ]:
SUBIR_AHORA_PARA_ETIQUETAR = False
SUBIR_AHORA_GALERIA = False

if (
    DRIVE_CAN_WRITE
    and SUBIR_AHORA_PARA_ETIQUETAR
):
    subir_carpeta_local_drive(
        PARA_ETIQUETAR,
        SHARED_DRIVE_FOLDER_ID
    )

if (
    DRIVE_CAN_WRITE
    and SUBIR_AHORA_GALERIA
):
    subir_carpeta_local_drive(
        GALERIA,
        SHARED_DRIVE_FOLDER_ID
    )

print(
    "Subidas opcionales terminadas "
    "o desactivadas."
)


### 12. Exportación a la carpeta compartida mediante Drive API

Se exportan los dos modelos, las métricas y los resultados por fotografía. Primero se generan localmente en Colab y, si la cuenta autenticada tiene permiso de edición, se suben a la carpeta compartida mediante Google Drive API.

Cada góndola incluye ahora dos conjuntos de detecciones:

```text
class_id 0 = producto
class_id 1 = vacío
```

Para cada vacío se guardan sus coordenadas tanto en la góndola aplanada como en la fotografía original, además del score visual y la repisa a la que pertenece.


### Comprobación final de límites

Se informa el RSS del proceso y la memoria asignada por PyTorch. El allocator queda limitado a 2.30 GB para dejar margen al contexto CUDA, por lo que `nvidia-smi` puede mostrar algo más que `torch.cuda.memory_allocated()`.


In [ ]:
estado_final = comprobar_recursos(
    "final",
    estricto=True
)

if torch.cuda.is_available():
    pico_gpu = (
        torch.cuda.max_memory_allocated()
        / 1e9
    )
else:
    pico_gpu = 0.0

print(
    "Pico PyTorch registrado:",
    f"{pico_gpu:.2f} GB"
)

print(
    "Objetivo RAM:",
    f"< {RAM_LIMITE_GB:.1f} GB"
)

print(
    "Objetivo VRAM total:",
    "< 3.0 GB"
)

METRICAS["limites_ultra"] = {
    "ram_limite_gb": RAM_LIMITE_GB,
    "ram_objetivo_gb": RAM_OBJETIVO_GB,
    "vram_allocator_limite_gb": VRAM_ALLOCATOR_LIMITE_GB,
    "vram_pico_pytorch_gb": float(pico_gpu),
    "ram_final_gb": estado_final["ram_gb"]
}


In [ ]:
import ultralytics
import transformers

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    BEST_GONDOLA,
    EXPORT_DIR / "best_gondola.pt"
)

shutil.copy2(
    BEST_PRODUCTO,
    EXPORT_DIR / "best_producto.pt"
)

resumen = []

for nombre, r in RESULTADOS.items():
    for k, g in enumerate(r["gondolas"]):
        resumen.append(
            {
                "foto": nombre,
                "split": SPLIT_FOTO[nombre],
                "gondola": k,
                "box": g["box"],
                "metodo_aplanado": g["metodo"],
                "n_productos": len(g["productos"]),
                "n_productos_guard": len(g["productos_guard"]),
                "n_vacios": len(g["vacios"]),
                "productos": g["productos_det"],
                "vacios": g["vacios"],
                "ocupacion": g["analisis"]["ocupacion"],
                "repisas": [
                    {
                        "y1": f["y1"],
                        "y2": f["y2"],
                        "ocupacion": f["ocupacion"],
                        "vacio_confirmado": f["vacio_confirmado"],
                        "incertidumbre": f["incertidumbre"],
                        "alerta": f["alerta"],
                        "origen": f["origen"],
                        "huecos": [
                            h
                            for h in f["huecos"]
                            if h["confirmado"]
                        ]
                    }
                    for f in g["analisis"]["filas"]
                ],
                "segundos": r["segundos"],
                "timestamp": datetime.now().isoformat(timespec="seconds")
            }
        )

with open(
    EXPORT_DIR / "resultados_productos_vacios.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        resumen,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    EXPORT_DIR / "config_metricas.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "CFG": CFG,
            "CONF_PRODUCTO": CONF_PRODUCTO,
            "CONF_PRODUCTO_MIN": CONF_PRODUCTO_MIN,
            "umbral_reducido_en_pipeline": False,
            "inferencia_producto": "multivista_mismo_umbral",
            "detector_vacio": "geometria_visual_recheck_yolo",
            "optimizacion": {
                "modo_ultra_4gb_3gb": MODO_ULTRA_4GB_3GB,
                "ram_limite_gb": RAM_LIMITE_GB,
                "ram_objetivo_gb": RAM_OBJETIVO_GB,
                "vram_allocator_limite_gb": VRAM_ALLOCATOR_LIMITE_GB,
                "foundation_device": FOUNDATION_DEVICE,
                "producto_tile_px": PRODUCTO_TILE_PX,
                "producto_tile_overlap": PRODUCTO_TILE_OVERLAP,
                "sam_lote": CFG["sam_lote"],
                "lazy_sam": True,
                "liberacion_modelos": True,
                "mapas_vacio_reutilizados": True,
                "modo_pipeline": MODO_PIPELINE,
                "carpetas_drive_ignoradas": sorted(CARPETAS_IGNORAR_DRIVE)
            },
            "recursos": METRICAS_RECURSOS,
            "METRICAS": METRICAS,
            "clases_salida": {
                "0": "producto",
                "1": "vacio"
            },
            "versiones": {
                "ultralytics": ultralytics.__version__,
                "transformers": transformers.__version__,
                "torch": torch.__version__,
                "opencv": cv2.__version__
            },
            "val_grupos": sorted(VAL_GROUPS)
        },
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

df_p.to_csv(
    EXPORT_DIR / "producto_val_por_gondola.csv",
    index=False
)

df_conf.to_csv(
    EXPORT_DIR / "producto_barrido_conf.csv",
    index=False
)

if "df_v" in globals() and len(df_v):
    df_v.to_csv(
        EXPORT_DIR / "vacios_val_por_gondola.csv",
        index=False
    )

print(
    "Exportado a:",
    EXPORT_DIR
)

for p in sorted(EXPORT_DIR.iterdir()):
    print("  ", p.name)


if (
    DRIVE_CAN_WRITE
    and SUBIR_OUTPUTS_AUTOMATICO
):
    print(
        "\nSubiendo outputs_v2 a "
        "la carpeta compartida..."
    )

    try:
        outputs_drive_id = (
            subir_carpeta_local_drive(
                EXPORT_DIR,
                SHARED_DRIVE_FOLDER_ID
            )
        )

        print(
            "Outputs procesados para Drive:",
            outputs_drive_id
        )

    except Exception as ex:
        print(
            "Drive no respondió correctamente. "
            "Los resultados locales NO se perdieron."
        )

        print(
            "Error de subida:",
            type(ex).__name__,
            str(ex)
        )

        print(
            "Resultados locales:",
            EXPORT_DIR
        )
else:
    print(
        "\nLa cuenta tiene acceso de lectura, "
        "pero no permiso para guardar en la carpeta."
    )

    print(
        "Los resultados quedaron temporalmente en:",
        EXPORT_DIR
    )
